# Replay verification, no model

Applies 8 patches (that resolved LOCALLY) to the base snapshots, applies the official
test_patch, and runs exactly the test nodes swegemma would run. CPU only.

Purpose: is the local sandbox the reason these tasks resolve locally but not on Kaggle?


In [ ]:

import base64, json, os, subprocess, sys
from pathlib import Path

print('input:', sorted(os.listdir('/kaggle/input')))
TASKS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'tasks.jsonl' in files:
        TASKS = Path(root) / 'tasks.jsonl'
        break
SNAPS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.tgz') for f in files):
        SNAPS = Path(root)
        break
print('tasks:', TASKS)
print('snaps:', SNAPS)
assert TASKS and SNAPS

PATCHES = json.loads("""{"fastapi_15661": "LS0tIGEvc2NyaXB0cy9wcmVwYXJlX3JlbGVhc2UucHkKKysrIGIvc2NyaXB0cy9wcmVwYXJlX3JlbGVhc2UucHkKQEAgLTAsMCArMSwyMTYgQEAKKyIiIlByZXBhcmUgYSByZWxlYXNlIGJ5IHVwZGF0aW5nIHRoZSBwYWNrYWdlIHZlcnNpb24gYW5kIHJlbGVhc2Ugbm90ZXMuIiIiCisKK2ltcG9ydCByZQorZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZQorZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCitmcm9tIHR5cGluZyBpbXBvcnQgQW5ub3RhdGVkLCBMaXRlcmFsCisKK2ltcG9ydCB0eXBlcgorCitWRVJTSU9OX1BBVFRFUk4gPSByZS5jb21waWxlKHInKD9tKV5fX3ZlcnNpb25fXyA9ICIoXGQrXC5cZCtcLlxkKykiJCcpCitWRVJTSU9OX0hFQURJTkdfUEFUVEVSTiA9IHJlLmNvbXBpbGUociIoP20pXiMjIChcZCtcLlxkK1wuXGQrKSg/OiBcKFteKV0rXCkpPyQiKQorUkVMRUFTRV9OT1RFU19IRUFERVIgPSAiIiItLS0KK2hpZGU6CisgIC0gbmF2aWdhdGlvbgorLS0tCisKKyMgUmVsZWFzZSBOb3RlcworCisiIiIKK0xBVEVTVF9DSEFOR0VTX0hFQURFUiA9ICIjIyBMYXRlc3QgQ2hhbmdlcyIKK0J1bXBUeXBlID0gTGl0ZXJhbFsibWFqb3IiLCAibWlub3IiLCAicGF0Y2giXQorCithcHAgPSB0eXBlci5UeXBlcigpCisKKworZGVmIHBhcnNlX3ZlcnNpb24odmVyc2lvbjogc3RyKSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKKyAgICBtYXRjaCA9IHJlLmZ1bGxtYXRjaChyIlxkK1wuXGQrXC5cZCsiLCB2ZXJzaW9uKQorICAgIGlmIG5vdCBtYXRjaDoKKyAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgdmVyc2lvbjoge3ZlcnNpb24hcn0uIEV4cGVjdGVkIGZvcm1hdDogWC5ZLloiKQorICAgIG1ham9yLCBtaW5vciwgcGF0Y2ggPSB2ZXJzaW9uLnNwbGl0KCIuIikKKyAgICByZXR1cm4gaW50KG1ham9yKSwgaW50KG1pbm9yKSwgaW50KHBhdGNoKQorCisKK2RlZiBnZXRfY3VycmVudF92ZXJzaW9uKGNvbnRlbnQ6IHN0ciwgdmVyc2lvbl9maWxlOiBQYXRoKSAtPiBzdHI6CisgICAgbWF0Y2hlcyA9IGxpc3QoVkVSU0lPTl9QQVRURVJOLmZpbmRpdGVyKGNvbnRlbnQpKQorICAgIGlmIGxlbihtYXRjaGVzKSAhPSAxOgorICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCisgICAgICAgICAgICBmIkV4cGVjdGVkIGV4YWN0bHkgb25lIF9fdmVyc2lvbl9fIGFzc2lnbm1lbnQgaW4ge3ZlcnNpb25fZmlsZX0sICIKKyAgICAgICAgICAgIGYiZm91bmQge2xlbihtYXRjaGVzKX0iCisgICAgICAgICkKKyAgICByZXR1cm4gbWF0Y2hlc1swXS5ncm91cCgxKQorCisKK2RlZiBidW1wX3ZlcnNpb24odmVyc2lvbjogc3RyLCBidW1wOiBCdW1wVHlwZSkgLT4gc3RyOgorICAgIG1ham9yLCBtaW5vciwgcGF0Y2ggPSBwYXJzZV92ZXJzaW9uKHZlcnNpb24pCisgICAgaWYgYnVtcCA9PSAibWFqb3IiOgorICAgICAgICByZXR1cm4gZiJ7bWFqb3IgKyAxfS4wLjAiCisgICAgaWYgYnVtcCA9PSAibWlub3IiOgorICAgICAgICByZXR1cm4gZiJ7bWFqb3J9LnttaW5vciArIDF9LjAiCisgICAgcmV0dXJuIGYie21ham9yfS57bWlub3J9LntwYXRjaCArIDF9IgorCisKK2RlZiB1cGRhdGVfdmVyc2lvbl9maWxlKGNvbnRlbnQ6IHN0ciwgdmVyc2lvbjogc3RyLCB2ZXJzaW9uX2ZpbGU6IFBhdGgpIC0+IHN0cjoKKyAgICBjdXJyZW50X3ZlcnNpb24gPSBnZXRfY3VycmVudF92ZXJzaW9uKGNvbnRlbnQsIHZlcnNpb25fZmlsZSkKKyAgICBpZiBwYXJzZV92ZXJzaW9uKHZlcnNpb24pIDw9IHBhcnNlX3ZlcnNpb24oY3VycmVudF92ZXJzaW9uKToKKyAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAorICAgICAgICAgICAgZiJOZXcgdmVyc2lvbiB7dmVyc2lvbn0gbXVzdCBiZSBncmVhdGVyIHRoYW4gY3VycmVudCB2ZXJzaW9uIHtjdXJyZW50X3ZlcnNpb259IgorICAgICAgICApCisgICAgcmV0dXJuIFZFUlNJT05fUEFUVEVSTi5zdWIoZidfX3ZlcnNpb25fXyA9ICJ7dmVyc2lvbn0iJywgY29udGVudCwgY291bnQ9MSkKKworCitkZWYgdXBkYXRlX3JlbGVhc2Vfbm90ZXMoCisgICAgY29udGVudDogc3RyLCB2ZXJzaW9uOiBzdHIsIHJlbGVhc2VfZGF0ZTogZGF0ZSwgcmVsZWFzZV9ub3Rlc19maWxlOiBQYXRoCispIC0+IHN0cjoKKyAgICBpZiBub3QgY29udGVudC5zdGFydHN3aXRoKFJFTEVBU0VfTk9URVNfSEVBREVSKToKKyAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAorICAgICAgICAgICAgZiJ7cmVsZWFzZV9ub3Rlc19maWxlfSBtdXN0IHN0YXJ0IHdpdGgge1JFTEVBU0VfTk9URVNfSEVBREVSIXJ9IgorICAgICAgICApCisgICAgaWYgcmUuc2VhcmNoKHJmIl4jIyB7cmUuZXNjYXBlKHZlcnNpb24pfSg/OiBcKFteKV0rXCkpPyQiLCBjb250ZW50LCByZS5NKToKKyAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUmVsZWFzZSBub3RlcyBhbHJlYWR5IGNvbnRhaW4gYSBzZWN0aW9uIGZvciB7dmVyc2lvbn0iKQorCisgICAgbGF0ZXN0X2hlYWRlciA9IGYie1JFTEVBU0VfTk9URVNfSEVBREVSfXtMQVRFU1RfQ0hBTkdFU19IRUFERVJ9XG4iCisgICAgaWYgbm90IGNvbnRlbnQuc3RhcnRzd2l0aChsYXRlc3RfaGVhZGVyKToKKyAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie3JlbGVhc2Vfbm90ZXNfZmlsZX0gbXVzdCBzdGFydCB3aXRoIHtsYXRlc3RfaGVhZGVyIXJ9IikKKworICAgIHJlbGVhc2VfaGVhZGVyID0gZiIjIyB7dmVyc2lvbn0gKHtyZWxlYXNlX2RhdGUuaXNvZm9ybWF0KCl9KSIKKyAgICByZXR1cm4gY29udGVudC5yZXBsYWNlKAorICAgICAgICBsYXRlc3RfaGVhZGVyLAorICAgICAgICBmIntSRUxFQVNFX05PVEVTX0hFQURFUn17TEFURVNUX0NIQU5HRVNfSEVBREVSfVxuXG57cmVsZWFzZV9oZWFkZXJ9XG4iLAorICAgICAgICAxLAorICAgICkKKworCitkZWYgZ2V0X3JlbGVhc2Vfbm90ZXNfYm9keShjb250ZW50OiBzdHIsIHZlcnNpb246IHN0ciwgcmVsZWFzZV9ub3Rlc19maWxlOiBQYXRoKSAtPiBzdHI6CisgICAgdmVyc2lvbl9oZWFkaW5nID0gcmUuY29tcGlsZShyZiIoP20pXiMjIHtyZS5lc2NhcGUodmVyc2lvbil9KD86IFwoW14pXStcKSk/JCIpCisgICAgbWF0Y2ggPSB2ZXJzaW9uX2hlYWRpbmcuc2VhcmNoKGNvbnRlbnQpCisgICAgaWYgbm90IG1hdGNoOgorICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCisgICAgICAgICAgICBmIkNvdWxkIG5vdCBmaW5kIHJlbGVhc2Ugbm90ZXMgc2VjdGlvbiBmb3Ige3ZlcnNpb259IGluIHtyZWxlYXNlX25vdGVzX2ZpbGV9IgorICAgICAgICApCisKKyAgICBuZXh0X21hdGNoID0gVkVSU0lPTl9IRUFESU5HX1BBVFRFUk4uc2VhcmNoKGNvbnRlbnQsIG1hdGNoLmVuZCgpKQorICAgIGVuZCA9IG5leHRfbWF0Y2guc3RhcnQoKSBpZiBuZXh0X21hdGNoIGVsc2UgbGVuKGNvbnRlbnQpCisgICAgYm9keSA9IGNvbnRlbnRbbWF0Y2guZW5kKCkgOiBlbmRdLnN0cmlwKCkKKyAgICBpZiBub3QgYm9keToKKyAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAorICAgICAgICAgICAgZiJSZWxlYXNlIG5vdGVzIHNlY3Rpb24gZm9yIHt2ZXJzaW9ufSBpbiB7cmVsZWFzZV9ub3Rlc19maWxlfSBpcyBlbXB0eSIKKyAgICAgICAgKQorICAgIHJldHVybiBmIntib2R5fVxuIgorCisKK0BhcHAuY29tbWFuZCgpCitkZWYgcHJlcGFyZSgKKyAgICBidW1wOiBBbm5vdGF0ZWRbCisgICAgICAgIEJ1bXBUeXBlLAorICAgICAgICB0eXBlci5Bcmd1bWVudCgKKyAgICAgICAgICAgIGVudnZhcj0iUFJFUEFSRV9SRUxFQVNFX0JVTVAiLAorICAgICAgICAgICAgaGVscD0iVGhlIHJlbGVhc2UgYnVtcCB0byBtYWtlOiBtYWpvciwgbWlub3IsIG9yIHBhdGNoLiIsCisgICAgICAgICksCisgICAgXSwKKyAgICB2ZXJzaW9uX2ZpbGU6IEFubm90YXRlZFsKKyAgICAgICAgUGF0aCwKKyAgICAgICAgdHlwZXIuT3B0aW9uKAorICAgICAgICAgICAgZW52dmFyPSJQUkVQQVJFX1JFTEVBU0VfVkVSU0lPTl9GSUxFIiwKKyAgICAgICAgICAgIGV4aXN0cz1UcnVlLAorICAgICAgICAgICAgZmlsZV9va2F5PVRydWUsCisgICAgICAgICAgICBkaXJfb2theT1GYWxzZSwKKyAgICAgICAgICAgIHJlYWRhYmxlPVRydWUsCisgICAgICAgICAgICB3cml0YWJsZT1UcnVlLAorICAgICAgICAgICAgaGVscD0iUGF0aCB0byB0aGUgUHl0aG9uIGZpbGUgY29udGFpbmluZyB0aGUgX192ZXJzaW9uX18gYXNzaWdubWVudC4iLAorICAgICAgICApLAorICAgIF0sCisgICAgcmVsZWFzZV9ub3Rlc19maWxlOiBBbm5vdGF0ZWRbCisgICAgICAgIFBhdGgsCisgICAgICAgIHR5cGVyLk9wdGlvbigKKyAgICAgICAgICAgIGVudnZhcj0iUFJFUEFSRV9SRUxFQVNFX1JFTEVBU0VfTk9URVNfRklMRSIsCisgICAgICAgICAgICBleGlzdHM9VHJ1ZSwKKyAgICAgICAgICAgIGZpbGVfb2theT1UcnVlLAorICAgICAgICAgICAgZGlyX29rYXk9RmFsc2UsCisgICAgICAgICAgICByZWFkYWJsZT1UcnVlLAorICAgICAgICAgICAgd3JpdGFibGU9VHJ1ZSwKKyAgICAgICAgICAgIGhlbHA9IlBhdGggdG8gdGhlIHJlbGVhc2Ugbm90ZXMgTWFya2Rvd24gZmlsZS4iLAorICAgICAgICApLAorICAgIF0sCisgICAgcmVsZWFzZV9kYXRlOiBBbm5vdGF0ZWRbCisgICAgICAgIHN0ciwKKyAgICAgICAgdHlwZXIuT3B0aW9uKAorICAgICAgICAgICAgIi0tZGF0ZSIsCisgICAgICAgICAgICBlbnZ2YXI9IlBSRVBBUkVfUkVMRUFTRV9EQVRFIiwKKyAgICAgICAgICAgIGhlbHA9IlJlbGVhc2UgZGF0ZSBpbiBZWVlZLU1NLUREIGZvcm1hdC4gRGVmYXVsdHMgdG8gdG9kYXkuIiwKKyAgICAgICAgKSwKKyAgICBdID0gZGF0ZS50b2RheSgpLmlzb2Zvcm1hdCgpLAorKSAtPiBOb25lOgorICAgIHBhcnNlZF9yZWxlYXNlX2RhdGUgPSBkYXRlLmZyb21pc29mb3JtYXQocmVsZWFzZV9kYXRlIG9yIGRhdGUudG9kYXkoKS5pc29mb3JtYXQoKSkKKworICAgIHZlcnNpb25fZmlsZV9jb250ZW50ID0gdmVyc2lvbl9maWxlLnJlYWRfdGV4dCgpCisgICAgcmVsZWFzZV9ub3Rlc19jb250ZW50ID0gcmVsZWFzZV9ub3Rlc19maWxlLnJlYWRfdGV4dCgpCisgICAgdmVyc2lvbiA9IGJ1bXBfdmVyc2lvbigKKyAgICAgICAgZ2V0X2N1cnJlbnRfdmVyc2lvbih2ZXJzaW9uX2ZpbGVfY29udGVudCwgdmVyc2lvbl9maWxlKSwgYnVtcAorICAgICkKKworICAgIHZlcnNpb25fZmlsZS53cml0ZV90ZXh0KAorICAgICAgICB1cGRhdGVfdmVyc2lvbl9maWxlKHZlcnNpb25fZmlsZV9jb250ZW50LCB2ZXJzaW9uLCB2ZXJzaW9uX2ZpbGUpCisgICAgKQorICAgIHJlbGVhc2Vfbm90ZXNfZmlsZS53cml0ZV90ZXh0KAorICAgICAgICB1cGRhdGVfcmVsZWFzZV9ub3RlcygKKyAgICAgICAgICAgIHJlbGVhc2Vfbm90ZXNfY29udGVudCwgdmVyc2lvbiwgcGFyc2VkX3JlbGVhc2VfZGF0ZSwgcmVsZWFzZV9ub3Rlc19maWxlCisgICAgICAgICkKKyAgICApCisKKyAgICB0eXBlci5lY2hvKGYiUHJlcGFyZWQgcmVsZWFzZSB7dmVyc2lvbn0gKHtwYXJzZWRfcmVsZWFzZV9kYXRlLmlzb2Zvcm1hdCgpfSkiKQorCisKK0BhcHAuY29tbWFuZCgpCitkZWYgY3VycmVudF92ZXJzaW9uKAorICAgIHZlcnNpb25fZmlsZTogQW5ub3RhdGVkWworICAgICAgICBQYXRoLAorICAgICAgICB0eXBlci5PcHRpb24oCisgICAgICAgICAgICBlbnZ2YXI9IlBSRVBBUkVfUkVMRUFTRV9WRVJTSU9OX0ZJTEUiLAorICAgICAgICAgICAgZXhpc3RzPVRydWUsCisgICAgICAgICAgICBmaWxlX29rYXk9VHJ1ZSwKKyAgICAgICAgICAgIGRpcl9va2F5PUZhbHNlLAorICAgICAgICAgICAgcmVhZGFibGU9VHJ1ZSwKKyAgICAgICAgICAgIGhlbHA9IlBhdGggdG8gdGhlIFB5dGhvbiBmaWxlIGNvbnRhaW5pbmcgdGhlIF9fdmVyc2lvbl9fIGFzc2lnbm1lbnQuIiwKKyAgICAgICAgKSwKKyAgICBdLAorKSAtPiBOb25lOgorICAgIHR5cGVyLmVjaG8oZ2V0X2N1cnJlbnRfdmVyc2lvbih2ZXJzaW9uX2ZpbGUucmVhZF90ZXh0KCksIHZlcnNpb25fZmlsZSkpCisKKworQGFwcC5jb21tYW5kKCkKK2RlZiByZWxlYXNlX25vdGVzKAorICAgIHZlcnNpb25fZmlsZTogQW5ub3RhdGVkWworICAgICAgICBQYXRoLAorICAgICAgICB0eXBlci5PcHRpb24oCisgICAgICAgICAgICBlbnZ2YXI9IlBSRVBBUkVfUkVMRUFTRV9WRVJTSU9OX0ZJTEUiLAorICAgICAgICAgICAgZXhpc3RzPVRydWUsCisgICAgICAgICAgICBmaWxlX29rYXk9VHJ1ZSwKKyAgICAgICAgICAgIGRpcl9va2F5PUZhbHNlLAorICAgICAgICAgICAgcmVhZGFibGU9VHJ1ZSwKKyAgICAgICAgICAgIGhlbHA9IlBhdGggdG8gdGhlIFB5dGhvbiBmaWxlIGNvbnRhaW5pbmcgdGhlIF9fdmVyc2lvbl9fIGFzc2lnbm1lbnQuIiwKKyAgICAgICAgKSwKKyAgICBdLAorICAgIHJlbGVhc2Vfbm90ZXNfZmlsZTogQW5ub3RhdGVkWworICAgICAgICBQYXRoLAorICAgICAgICB0eXBlci5PcHRpb24oCisgICAgICAgICAgICBlbnZ2YXI9IlBSRVBBUkVfUkVMRUFTRV9SRUxFQVNFX05PVEVTX0ZJTEUiLAorICAgICAgICAgICAgZXhpc3RzPVRydWUsCisgICAgICAgICAgICBmaWxlX29rYXk9VHJ1ZSwKKyAgICAgICAgICAgIGRpcl9va2F5PUZhbHNlLAorICAgICAgICAgICAgcmVhZGFibGU9VHJ1ZSwKKyAgICAgICAgICAgIGhlbHA9IlBhdGggdG8gdGhlIHJlbGVhc2Ugbm90ZXMgTWFya2Rvd24gZmlsZS4iLAorICAgICAgICApLAorICAgIF0sCispIC0+IE5vbmU6CisgICAgdmVyc2lvbiA9IGdldF9jdXJyZW50X3ZlcnNpb24odmVyc2lvbl9maWxlLnJlYWRfdGV4dCgpLCB2ZXJzaW9uX2ZpbGUpCisgICAgdHlwZXIuZWNobygKKyAgICAgICAgZ2V0X3JlbGVhc2Vfbm90ZXNfYm9keSgKKyAgICAgICAgICAgIHJlbGVhc2Vfbm90ZXNfZmlsZS5yZWFkX3RleHQoKSwgdmVyc2lvbiwgcmVsZWFzZV9ub3Rlc19maWxlCisgICAgICAgICksCisgICAgICAgIG5sPUZhbHNlLAorICAgICkKKworCitpZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgorICAgIGFwcCgpCg==", "fastapi_15588": "LS0tIGEvZmFzdGFwaS9zc2UucHkKKysrIGIvZmFzdGFwaS9zc2UucHkKQEAgLTMzLDEwICszMywyMCBAQCBjbGFzcyBFdmVudFNvdXJjZVJlc3BvbnNlKFN0cmVhbWluZ1Jlc3BvbnNlKToKICAgICBtZWRpYV90eXBlID0gInRleHQvZXZlbnQtc3RyZWFtIgogCiAKLWRlZiBfY2hlY2tfaWRfbm9fbnVsbCh2OiBzdHIgfCBOb25lKSAtPiBzdHIgfCBOb25lOgorZGVmIF9jaGVja19zaW5nbGVfbGluZSh2OiBzdHIgfCBOb25lLCBmaWVsZF9uYW1lOiBzdHIpIC0+IHN0ciB8IE5vbmU6CisgICAgaWYgdiBpcyBub3QgTm9uZSBhbmQgKCJcciIgaW4gdiBvciAiXG4iIGluIHYpOgorICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU1NFICd7ZmllbGRfbmFtZX0nIG11c3QgYmUgYSBzaW5nbGUgbGluZSIpCisgICAgcmV0dXJuIHYKKworCitkZWYgX2NoZWNrX2V2ZW50X3NpbmdsZV9saW5lKHY6IHN0ciB8IE5vbmUpIC0+IHN0ciB8IE5vbmU6CisgICAgcmV0dXJuIF9jaGVja19zaW5nbGVfbGluZSh2LCAiZXZlbnQiKQorCisKK2RlZiBfY2hlY2tfaWRfdmFsaWQodjogc3RyIHwgTm9uZSkgLT4gc3RyIHwgTm9uZToKICAgICBpZiB2IGlzIG5vdCBOb25lIGFuZCAiXDAiIGluIHY6CiAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNTRSAnaWQnIG11c3Qgbm90IGNvbnRhaW4gbnVsbCBjaGFyYWN0ZXJzIikKLSAgICByZXR1cm4gdgorICAgIHJldHVybiBfY2hlY2tfc2luZ2xlX2xpbmUodiwgImlkIikKIAogCiBjbGFzcyBTZXJ2ZXJTZW50RXZlbnQoQmFzZU1vZGVsKToKQEAgLTg2LDI0ICs5NiwyNyBAQCBjbGFzcyBTZXJ2ZXJTZW50RXZlbnQoQmFzZU1vZGVsKToKICAgICBdID0gTm9uZQogICAgIGV2ZW50OiBBbm5vdGF0ZWRbCiAgICAgICAgIHN0ciB8IE5vbmUsCisgICAgICAgIEFmdGVyVmFsaWRhdG9yKF9jaGVja19ldmVudF9zaW5nbGVfbGluZSksCiAgICAgICAgIERvYygKICAgICAgICAgICAgICIiIgogICAgICAgICAgICAgT3B0aW9uYWwgZXZlbnQgdHlwZSBuYW1lLgogCiAgICAgICAgICAgICBNYXBzIHRvIGBhZGRFdmVudExpc3RlbmVyKGV2ZW50LCAuLi4pYCBvbiB0aGUgYnJvd3Nlci4gV2hlbiBvbWl0dGVkLAotICAgICAgICAgICAgdGhlIGJyb3dzZXIgZGlzcGF0Y2hlcyBvbiB0aGUgZ2VuZXJpYyBgbWVzc2FnZWAgZXZlbnQuCisgICAgICAgICAgICB0aGUgYnJvd3NlciBkaXNwYXRjaGVzIG9uIHRoZSBnZW5lcmljIGBtZXNzYWdlYCBldmVudC4gTXVzdCBiZSBhCisgICAgICAgICAgICBzaW5nbGUgbGluZS4KICAgICAgICAgICAgICIiIgogICAgICAgICApLAogICAgIF0gPSBOb25lCiAgICAgaWQ6IEFubm90YXRlZFsKICAgICAgICAgc3RyIHwgTm9uZSwKLSAgICAgICAgQWZ0ZXJWYWxpZGF0b3IoX2NoZWNrX2lkX25vX251bGwpLAorICAgICAgICBBZnRlclZhbGlkYXRvcihfY2hlY2tfaWRfdmFsaWQpLAogICAgICAgICBEb2MoCiAgICAgICAgICAgICAiIiIKICAgICAgICAgICAgIE9wdGlvbmFsIGV2ZW50IElELgogCiAgICAgICAgICAgICBUaGUgYnJvd3NlciBzZW5kcyB0aGlzIHZhbHVlIGJhY2sgYXMgdGhlIGBMYXN0LUV2ZW50LUlEYCBoZWFkZXIgb24KLSAgICAgICAgICAgIGF1dG9tYXRpYyByZWNvbm5lY3Rpb24uICoqTXVzdCBub3QgY29udGFpbiBudWxsIChgXFwwYCkgY2hhcmFjdGVycy4qKgorICAgICAgICAgICAgYXV0b21hdGljIHJlY29ubmVjdGlvbi4gKipNdXN0IGJlIGEgc2luZ2xlIGxpbmUqKiBhbmQgbXVzdCBub3QgY29udGFpbgorICAgICAgICAgICAgbnVsbCAoYFxcMGApIGNoYXJhY3RlcnMuCiAgICAgICAgICAgICAiIiIKICAgICAgICAgKSwKICAgICBdID0gTm9uZQo=", "fastapi_15589": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTgyNiw2ICs4MjYsMTAgQEAgZGVmIHJlcXVlc3RfcGFyYW1zX3RvX2FyZ3MoCiAgICAgICAgIGlmIHZhbHVlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgcGFyYW1zX3RvX3Byb2Nlc3NbZ2V0X3ZhbGlkYXRpb25fYWxpYXMoZmllbGQpXSA9IHZhbHVlCiAgICAgICAgIHByb2Nlc3NlZF9rZXlzLmFkZChhbGlhcyBvciBnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkpCisgICAgICAgICMgRm9yIGhlYWRlcnMgd2l0aCBjb252ZXJ0X3VuZGVyc2NvcmVzPVRydWUsIG1hcmsgYm90aCB0aGUgY29udmVydGVkCisgICAgICAgICMgaGVhZGVyIG5hbWUgYW5kIHRoZSBvcmlnaW5hbCBmaWVsZCBhbGlhcyBhcyBwcm9jZXNzZWQgdG8gYXZvaWQKKyAgICAgICAgIyBhY2NlcHRpbmcgdGhlIG9yaWdpbmFsIGFsaWFzIGFzIGFuIGV4dHJhIGhlYWRlci4KKyAgICAgICAgcHJvY2Vzc2VkX2tleXMuYWRkKGdldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKSkKIAogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfcGFyYW1zLmtleXMoKToKICAgICAgICAgaWYga2V5IG5vdCBpbiBwcm9jZXNzZWRfa2V5czoK", "fastapi_15030": "...", "fastapi_14962": "LS0tIGEvZG9jc19zcmMvY3VzdG9tX3Jlc3BvbnNlL3R1dG9yaWFsMDEwX3B5MzEwLnB5CisrKyBiL2RvY3Nfc3JjL2N1c3RvbV9yZXNwb25zZS90dXRvcmlhbDAxMF9weTMxMC5weQpAQCAtMSw5ICsxLDkgQEAKIGZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSQotZnJvbSBmYXN0YXBpLnJlc3BvbnNlcyBpbXBvcnQgT1JKU09OUmVzcG9uc2UKK2Zyb20gZmFzdGFwaS5yZXNwb25zZXMgaW1wb3J0IEhUTUxSZXNwb25zZQogCi1hcHAgPSBGYXN0QVBJKGRlZmF1bHRfcmVzcG9uc2VfY2xhc3M9T1JKU09OUmVzcG9uc2UpCithcHAgPSBGYXN0QVBJKGRlZmF1bHRfcmVzcG9uc2VfY2xhc3M9SFRNTFJlc3BvbnNlKQogCiAKIEBhcHAuZ2V0KCIvaXRlbXMvIikKIGFzeW5jIGRlZiByZWFkX2l0ZW1zKCk6Ci0gICAgcmV0dXJuIFt7Iml0ZW1faWQiOiAiRm9vIn1dCisgICAgcmV0dXJuICI8aDE+SXRlbXM8L2gxPjxwPlRoaXMgaXMgYSBsaXN0IG9mIGl0ZW1zLjwvcD4iCi0tLSBhL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQorKysgYi9mYXN0YXBpL19jb21wYXQvdjIucHkKQEAgLTE5OSw2ICsxOTksMzIgQEAgZGVmIHNlcmlhbGl6ZSgKICAgICAgICAgICAgIGV4Y2x1ZGVfbm9uZT1leGNsdWRlX25vbmUsCiAgICAgICAgICkKIAorICAgIGRlZiBzZXJpYWxpemVfanNvbigKKyAgICAgICAgc2VsZiwKKyAgICAgICAgdmFsdWU6IEFueSwKKyAgICAgICAgKiwKKyAgICAgICAgaW5jbHVkZTogSW5jRXggfCBOb25lID0gTm9uZSwKKyAgICAgICAgZXhjbHVkZTogSW5jRXggfCBOb25lID0gTm9uZSwKKyAgICAgICAgYnlfYWxpYXM6IGJvb2wgPSBUcnVlLAorICAgICAgICBleGNsdWRlX3Vuc2V0OiBib29sID0gRmFsc2UsCisgICAgICAgIGV4Y2x1ZGVfZGVmYXVsdHM6IGJvb2wgPSBGYWxzZSwKKyAgICAgICAgZXhjbHVkZV9ub25lOiBib29sID0gRmFsc2UsCisgICAgKSAtPiBieXRlczoKKyAgICAgICAgIyBXaGF0IGNhbGxzIHRoaXMgY29kZSBwYXNzZXMgYSB2YWx1ZSB0aGF0IGFscmVhZHkgY2FsbGVkCisgICAgICAgICMgc2VsZi5fdHlwZV9hZGFwdGVyLnZhbGlkYXRlX3B5dGhvbih2YWx1ZSkKKyAgICAgICAgIyBUaGlzIHVzZXMgUHlkYW50aWMncyBkdW1wX2pzb24oKSB3aGljaCBzZXJpYWxpemVzIGRpcmVjdGx5IHRvIEpTT04KKyAgICAgICAgIyBieXRlcyBpbiBvbmUgcGFzcyAodmlhIFJ1c3QpLCBhdm9pZGluZyB0aGUgaW50ZXJtZWRpYXRlIFB5dGhvbiBkaWN0CisgICAgICAgICMgc3RlcCBvZiBkdW1wX3B5dGhvbihtb2RlPSJqc29uIikgKyBqc29uLmR1bXBzKCkuCisgICAgICAgIHJldHVybiBzZWxmLl90eXBlX2FkYXB0ZXIuZHVtcF9qc29uKAorICAgICAgICAgICAgdmFsdWUsCisgICAgICAgICAgICBpbmNsdWRlPWluY2x1ZGUsCisgICAgICAgICAgICBleGNsdWRlPWV4Y2x1ZGUsCisgICAgICAgICAgICBieV9hbGlhcz1ieV9hbGlhcywKKyAgICAgICAgICAgIGV4Y2x1ZGVfdW5zZXQ9ZXhjbHVkZV91bnNldCwKKyAgICAgICAgICAgIGV4Y2x1ZGVfZGVmYXVsdHM9ZXhjbHVkZV9kZWZhdWx0cywKKyAgICAgICAgICAgIGV4Y2x1ZGVfbm9uZT1leGNsdWRlX25vbmUsCisgICAgICAgICkKKwogICAgIGRlZiBfX2hhc2hfXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgICMgRWFjaCBNb2RlbEZpZWxkIGlzIHVuaXF1ZSBmb3Igb3VyIHB1cnBvc2VzLCB0byBhbGxvdyBtYWtpbmcgYSBkaWN0IGZyb20KICAgICAgICAgIyBNb2RlbEZpZWxkIHRvIGl0cyBKU09OIFNjaGVtYS4KLS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtMjcxLDYgKzI3MSw3IEBAIGFzeW5jIGRlZiBzZXJpYWxpemVfcmVzcG9uc2UoCiAgICAgZXhjbHVkZV9ub25lOiBib29sID0gRmFsc2UsCiAgICAgaXNfY29yb3V0aW5lOiBib29sID0gVHJ1ZSwKICAgICBlbmRwb2ludF9jdHg6IEVuZHBvaW50Q29udGV4dCB8IE5vbmUgPSBOb25lLAorICAgIGR1bXBfanNvbjogYm9vbCA9IEZhbHNlLAogKSAtPiBBbnk6CiAgICAgaWYgZmllbGQ6CiAgICAgICAgIGlmIGlzX2Nvcm91dGluZToKQEAgLTI4Niw4ICsyODcsOCBAQCBhc3luYyBkZWYgc2VyaWFsaXplX3Jlc3BvbnNlKAogICAgICAgICAgICAgICAgIGJvZHk9cmVzcG9uc2VfY29udGVudCwKICAgICAgICAgICAgICAgICBlbmRwb2ludF9jdHg9Y3R4LAogICAgICAgICAgICAgKQotCi0gICAgICAgIHJldHVybiBmaWVsZC5zZXJpYWxpemUoCisgICAgICAgIHNlcmlhbGl6ZXIgPSBmaWVsZC5zZXJpYWxpemVfanNvbiBpZiBkdW1wX2pzb24gZWxzZSBmaWVsZC5zZXJpYWxpemUKKyAgICAgICAgcmV0dXJuIHNlcmlhbGl6ZXIoCiAgICAgICAgICAgICB2YWx1ZSwKICAgICAgICAgICAgIGluY2x1ZGU9aW5jbHVkZSwKICAgICAgICAgICAgIGV4Y2x1ZGU9ZXhjbHVkZSwKQEAgLTQ0Myw2ICs0NDQsMTQgQEAgYXN5bmMgZGVmIGFwcChyZXF1ZXN0OiBSZXF1ZXN0KSAtPiBSZXNwb25zZToKICAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2VfYXJnc1sic3RhdHVzX2NvZGUiXSA9IGN1cnJlbnRfc3RhdHVzX2NvZGUKICAgICAgICAgICAgICAgICBpZiBzb2x2ZWRfcmVzdWx0LnJlc3BvbnNlLnN0YXR1c19jb2RlOgogICAgICAgICAgICAgICAgICAgICByZXNwb25zZV9hcmdzWyJzdGF0dXNfY29kZSJdID0gc29sdmVkX3Jlc3VsdC5yZXNwb25zZS5zdGF0dXNfY29kZQorICAgICAgICAgICAgICAgICMgVXNlIHRoZSBmYXN0IHBhdGggKGR1bXBfanNvbikgd2hlbiBubyBjdXN0b20gcmVzcG9uc2UKKyAgICAgICAgICAgICAgICAjIGNsYXNzIHdhcyBzZXQgYW5kIGEgcmVzcG9uc2UgZmllbGQgd2l0aCBhIFR5cGVBZGFwdGVyCisgICAgICAgICAgICAgICAgIyBleGlzdHMuIFNlcmlhbGl6ZXMgZGlyZWN0bHkgdG8gSlNPTiBieXRlcyB2aWEgUHlkYW50aWMncworICAgICAgICAgICAgICAgICMgUnVzdCBjb3JlLCBza2lwcGluZyB0aGUgaW50ZXJtZWRpYXRlIFB5dGhvbiBkaWN0ICsKKyAgICAgICAgICAgICAgICAjIGpzb24uZHVtcHMoKSBzdGVwLgorICAgICAgICAgICAgICAgIHVzZV9kdW1wX2pzb24gPSByZXNwb25zZV9maWVsZCBpcyBub3QgTm9uZSBhbmQgaXNpbnN0YW5jZSgKKyAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2VfY2xhc3MsIERlZmF1bHRQbGFjZWhvbGRlcgorICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICBjb250ZW50ID0gYXdhaXQgc2VyaWFsaXplX3Jlc3BvbnNlKAogICAgICAgICAgICAgICAgICAgICBmaWVsZD1yZXNwb25zZV9maWVsZCwKICAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2VfY29udGVudD1yYXdfcmVzcG9uc2UsCkBAIC00NTQsOCArNDYzLDE2IEBAIGFzeW5jIGRlZiBhcHAocmVxdWVzdDogUmVxdWVzdCkgLT4gUmVzcG9uc2U6CiAgICAgICAgICAgICAgICAgICAgIGV4Y2x1ZGVfbm9uZT1yZXNwb25zZV9tb2RlbF9leGNsdWRlX25vbmUsCiAgICAgICAgICAgICAgICAgICAgIGlzX2Nvcm91dGluZT1pc19jb3JvdXRpbmUsCiAgICAgICAgICAgICAgICAgICAgIGVuZHBvaW50X2N0eD1lbmRwb2ludF9jdHgsCisgICAgICAgICAgICAgICAgICAgIGR1bXBfanNvbj11c2VfZHVtcF9qc29uLAogICAgICAgICAgICAgICAgICkKLSAgICAgICAgICAgICAgICByZXNwb25zZSA9IGFjdHVhbF9yZXNwb25zZV9jbGFzcyhjb250ZW50LCAqKnJlc3BvbnNlX2FyZ3MpCisgICAgICAgICAgICAgICAgaWYgdXNlX2R1bXBfanNvbjoKKyAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2UgPSBSZXNwb25zZSgKKyAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRlbnQ9Y29udGVudCwKKyAgICAgICAgICAgICAgICAgICAgICAgIG1lZGlhX3R5cGU9ImFwcGxpY2F0aW9uL2pzb24iLAorICAgICAgICAgICAgICAgICAgICAgICAgKipyZXNwb25zZV9hcmdzLAorICAgICAgICAgICAgICAgICAgICApCisgICAgICAgICAgICAgICAgZWxzZToKKyAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2UgPSBhY3R1YWxfcmVzcG9uc2VfY2xhc3MoY29udGVudCwgKipyZXNwb25zZV9hcmdzKQogICAgICAgICAgICAgICAgIGlmIG5vdCBpc19ib2R5X2FsbG93ZWRfZm9yX3N0YXR1c19jb2RlKHJlc3BvbnNlLnN0YXR1c19jb2RlKToKICAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2UuYm9keSA9IGIiIgogICAgICAgICAgICAgICAgIHJlc3BvbnNlLmhlYWRlcnMucmF3LmV4dGVuZChzb2x2ZWRfcmVzdWx0LnJlc3BvbnNlLmhlYWRlcnMucmF3KQo=", "fastapi_15280": "LS0tIGEvZG9jc19zcmMvdmliZS90dXRvcmlhbDAwMV9weTMxMC5weQorKysgYi9kb2NzX3NyYy92aWJlL3R1dG9yaWFsMDAxX3B5MzEwLnB5CkBAIC0wLDAgKzEsMTIgQEAKK2Zyb20gdHlwaW5nIGltcG9ydCBBbnkKKworZnJvbSBmYXN0YXBpIGltcG9ydCBGYXN0QVBJCisKK2FwcCA9IEZhc3RBUEkoKQorCisKK0BhcHAudmliZSgKKyAgICAiL3ZpYmUvIiwKKyAgICBwcm9tcHQ9InBscyByZXR1cm4ganNvbiBvZiB1c2VycyBmcm9tIGRhdGFiYXNlLiBtYWtlIG5vIG1pc3Rha2VzIiwKKykKK2FzeW5jIGRlZiBhaV92aWJlcyhib2R5OiBBbnkpOiAuLi4KLS0tIGEvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKKysrIGIvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKQEAgLTEwLDcgKzEwLDExIEBACiAgICAgcmVxdWVzdF92YWxpZGF0aW9uX2V4Y2VwdGlvbl9oYW5kbGVyLAogICAgIHdlYnNvY2tldF9yZXF1ZXN0X3ZhbGlkYXRpb25fZXhjZXB0aW9uX2hhbmRsZXIsCiApCi1mcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgUmVxdWVzdFZhbGlkYXRpb25FcnJvciwgV2ViU29ja2V0UmVxdWVzdFZhbGlkYXRpb25FcnJvcgorZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0ICgKKyAgICBGYXN0QVBJRXJyb3IsCisgICAgUmVxdWVzdFZhbGlkYXRpb25FcnJvciwKKyAgICBXZWJTb2NrZXRSZXF1ZXN0VmFsaWRhdGlvbkVycm9yLAorKQogZnJvbSBmYXN0YXBpLmxvZ2dlciBpbXBvcnQgbG9nZ2VyCiBmcm9tIGZhc3RhcGkubWlkZGxld2FyZS5hc3luY2V4aXRzdGFjayBpbXBvcnQgQXN5bmNFeGl0U3RhY2tNaWRkbGV3YXJlCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5kb2NzIGltcG9ydCAoCkBAIC00NTU5LDYgKzQ1NjMsNjAgQEAgZGVmIHRyYWNlX2l0ZW0oaXRlbV9pZDogc3RyKToKICAgICAgICAgICAgIGdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbj1nZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb24sCiAgICAgICAgICkKIAorICAgIGRlZiB2aWJlKAorICAgICAgICBzZWxmLAorICAgICAgICBwYXRoOiBBbm5vdGF0ZWRbCisgICAgICAgICAgICBzdHIsCisgICAgICAgICAgICBEb2MoCisgICAgICAgICAgICAgICAgIiIiCisgICAgICAgICAgICAgICAgVGhlIFVSTCBwYXRoIHRvIGJlIHVzZWQgZm9yIHRoaXMgKnBhdGggb3BlcmF0aW9uKi4KKworICAgICAgICAgICAgICAgIEZvciBleGFtcGxlLCBpbiBgaHR0cDovL2V4YW1wbGUuY29tL3ZpYmVzYCwgdGhlIHBhdGggaXMgYC92aWJlc2AuCisgICAgICAgICAgICAgICAgIiIiCisgICAgICAgICAgICApLAorICAgICAgICBdLAorICAgICAgICAqLAorICAgICAgICBwcm9tcHQ6IEFubm90YXRlZFsKKyAgICAgICAgICAgIHN0ciwKKyAgICAgICAgICAgIERvYygKKyAgICAgICAgICAgICAgICAiIiIKKyAgICAgICAgICAgICAgICBUaGUgcHJvbXB0IHRvIHNlbmQgdG8gdGhlIExMTSBwcm92aWRlciBhbG9uZyB3aXRoIHRoZSBwYXlsb2FkLgorCisgICAgICAgICAgICAgICAgVGhpcyB0ZWxscyB0aGUgTExNIHdoYXQgdG8gZG8gd2l0aCB0aGUgcmVxdWVzdCBib2R5LgorICAgICAgICAgICAgICAgICIiIgorICAgICAgICAgICAgKSwKKyAgICAgICAgXSA9ICIiLAorICAgICkgLT4gQ2FsbGFibGVbW0RlY29yYXRlZENhbGxhYmxlXSwgRGVjb3JhdGVkQ2FsbGFibGVdOgorICAgICAgICAiIiIKKyAgICAgICAgQWRkIGEgKnZpYmUgY29kaW5nIHBhdGggb3BlcmF0aW9uKiB0aGF0IHJlY2VpdmVzIGFueSBIVFRQIG1ldGhvZAorICAgICAgICBhbmQgYW55IHBheWxvYWQuCisKKyAgICAgICAgSXQncyBpbnRlbmRlZCB0byByZWNlaXZlIHRoZSByZXF1ZXN0IGFuZCBzZW5kIHRoZSBwYXlsb2FkIGRpcmVjdGx5CisgICAgICAgIHRvIGFuIExMTSBwcm92aWRlciwgYW5kIHJldHVybiB0aGUgcmVzcG9uc2UgYXMgaXMuCisKKyAgICAgICAgRW1icmFjZSB0aGUgZnJlZWRvbSBhbmQgZmxleGliaWxpdHkgb2Ygbm90IGhhdmluZyBhbnkgZGF0YSB2YWxpZGF0aW9uLAorICAgICAgICBkb2N1bWVudGF0aW9uLCBvciBzZXJpYWxpemF0aW9uLgorCisgICAgICAgICMjIEV4YW1wbGUKKworICAgICAgICBgYGBweXRob24KKyAgICAgICAgZnJvbSB0eXBpbmcgaW1wb3J0IEFueQorCisgICAgICAgIGZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSQorCisgICAgICAgIGFwcCA9IEZhc3RBUEkoKQorCisKKyAgICAgICAgQGFwcC52aWJlKAorICAgICAgICAgICAgIi92aWJlLyIsCisgICAgICAgICAgICBwcm9tcHQ9InBscyByZXR1cm4ganNvbiBvZiB1c2VycyBmcm9tIGRhdGFiYXNlLiBtYWtlIG5vIG1pc3Rha2VzIiwKKyAgICAgICAgKQorICAgICAgICBhc3luYyBkZWYgYWlfdmliZXMoYm9keTogQW55KToKKyAgICAgICAgICAgIC4uLgorICAgICAgICBgYGAKKyAgICAgICAgIiIiCisgICAgICAgIHJhaXNlIEZhc3RBUElFcnJvcigiQXJlIHlvdSBraWRkaW5nIG1lPyBIYXBweSBBcHJpbCBGb29sJ3MiKQorCiAgICAgZGVmIHdlYnNvY2tldF9yb3V0ZSgKICAgICAgICAgc2VsZiwgcGF0aDogc3RyLCBuYW1lOiBzdHIgfCBOb25lID0gTm9uZQogICAgICkgLT4gQ2FsbGFibGVbW0RlY29yYXRlZENhbGxhYmxlXSwgRGVjb3JhdGVkQ2FsbGFibGVdOgo=", "fastapi_14964": "LS0tIGEvZmFzdGFwaS9yZXNwb25zZXMucHkKKysrIGIvZmFzdGFwaS9yZXNwb25zZXMucHkKQEAgLTEsMTIgKzEsMTQgQEAKIGZyb20gdHlwaW5nIGltcG9ydCBBbnkKIAorZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0IEZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcKIGZyb20gc3RhcmxldHRlLnJlc3BvbnNlcyBpbXBvcnQgRmlsZVJlc3BvbnNlIGFzIEZpbGVSZXNwb25zZSAgIyBub3FhCiBmcm9tIHN0YXJsZXR0ZS5yZXNwb25zZXMgaW1wb3J0IEhUTUxSZXNwb25zZSBhcyBIVE1MUmVzcG9uc2UgICMgbm9xYQogZnJvbSBzdGFybGV0dGUucmVzcG9uc2VzIGltcG9ydCBKU09OUmVzcG9uc2UgYXMgSlNPTlJlc3BvbnNlICAjIG5vcWEKIGZyb20gc3RhcmxldHRlLnJlc3BvbnNlcyBpbXBvcnQgUGxhaW5UZXh0UmVzcG9uc2UgYXMgUGxhaW5UZXh0UmVzcG9uc2UgICMgbm9xYQogZnJvbSBzdGFybGV0dGUucmVzcG9uc2VzIGltcG9ydCBSZWRpcmVjdFJlc3BvbnNlIGFzIFJlZGlyZWN0UmVzcG9uc2UgICMgbm9xYQogZnJvbSBzdGFybGV0dGUucmVzcG9uc2VzIGltcG9ydCBSZXNwb25zZSBhcyBSZXNwb25zZSAgIyBub3FhCiBmcm9tIHN0YXJsZXR0ZS5yZXNwb25zZXMgaW1wb3J0IFN0cmVhbWluZ1Jlc3BvbnNlIGFzIFN0cmVhbWluZ1Jlc3BvbnNlICAjIG5vcWEKK2Zyb20gdHlwaW5nX2V4dGVuc2lvbnMgaW1wb3J0IGRlcHJlY2F0ZWQKIAogdHJ5OgogICAgIGltcG9ydCB1anNvbgpAQCAtMjAsMjUgKzIyLDU5IEBACiAgICAgb3Jqc29uID0gTm9uZSAgIyB0eXBlOiBpZ25vcmUKIAogCitAZGVwcmVjYXRlZCgKKyAgICAiVUpTT05SZXNwb25zZSBpcyBkZXByZWNhdGVkLCBGYXN0QVBJIG5vdyBzZXJpYWxpemVzIGRhdGEgZGlyZWN0bHkgdG8gSlNPTiAiCisgICAgImJ5dGVzIHZpYSBQeWRhbnRpYyB3aGVuIGEgcmV0dXJuIHR5cGUgb3IgcmVzcG9uc2UgbW9kZWwgaXMgc2V0LCB3aGljaCBpcyAiCisgICAgImZhc3RlciBhbmQgZG9lc24ndCBuZWVkIGEgY3VzdG9tIHJlc3BvbnNlIGNsYXNzLiBSZWFkIG1vcmUgaW4gdGhlIEZhc3RBUEkgIgorICAgICJkb2NzOiBodHRwczovL2Zhc3RhcGkudGlhbmdvbG8uY29tL2FkdmFuY2VkL2N1c3RvbS1yZXNwb25zZS8jb3Jqc29uLW9yLXJlc3BvbnNlLW1vZGVsICIKKyAgICAiYW5kIGh0dHBzOi8vZmFzdGFwaS50aWFuZ29sby5jb20vdHV0b3JpYWwvcmVzcG9uc2UtbW9kZWwvIiwKKyAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAorICAgIHN0YWNrbGV2ZWw9MiwKKykKIGNsYXNzIFVKU09OUmVzcG9uc2UoSlNPTlJlc3BvbnNlKToKLSAgICAiIiIKLSAgICBKU09OIHJlc3BvbnNlIHVzaW5nIHRoZSBoaWdoLXBlcmZvcm1hbmNlIHVqc29uIGxpYnJhcnkgdG8gc2VyaWFsaXplIGRhdGEgdG8gSlNPTi4KKyAgICAiIiJKU09OIHJlc3BvbnNlIHVzaW5nIHRoZSB1anNvbiBsaWJyYXJ5IHRvIHNlcmlhbGl6ZSBkYXRhIHRvIEpTT04uCisKKyAgICAqKkRlcHJlY2F0ZWQqKjogYFVKU09OUmVzcG9uc2VgIGlzIGRlcHJlY2F0ZWQuIEZhc3RBUEkgbm93IHNlcmlhbGl6ZXMgZGF0YQorICAgIGRpcmVjdGx5IHRvIEpTT04gYnl0ZXMgdmlhIFB5ZGFudGljIHdoZW4gYSByZXR1cm4gdHlwZSBvciByZXNwb25zZSBtb2RlbCBpcworICAgIHNldCwgd2hpY2ggaXMgZmFzdGVyIGFuZCBkb2Vzbid0IG5lZWQgYSBjdXN0b20gcmVzcG9uc2UgY2xhc3MuCisKKyAgICBSZWFkIG1vcmUgaW4gdGhlCisgICAgW0Zhc3RBUEkgZG9jcyBmb3IgQ3VzdG9tIFJlc3BvbnNlXShodHRwczovL2Zhc3RhcGkudGlhbmdvbG8uY29tL2FkdmFuY2VkL2N1c3RvbS1yZXNwb25zZS8jb3Jqc29uLW9yLXJlc3BvbnNlLW1vZGVsKQorICAgIGFuZCB0aGUKKyAgICBbRmFzdEFQSSBkb2NzIGZvciBSZXNwb25zZSBNb2RlbF0oaHR0cHM6Ly9mYXN0YXBpLnRpYW5nb2xvLmNvbS90dXRvcmlhbC9yZXNwb25zZS1tb2RlbC8pLgogCi0gICAgUmVhZCBtb3JlIGFib3V0IGl0IGluIHRoZQotICAgIFtGYXN0QVBJIGRvY3MgZm9yIEN1c3RvbSBSZXNwb25zZSAtIEhUTUwsIFN0cmVhbSwgRmlsZSwgb3RoZXJzXShodHRwczovL2Zhc3RhcGkudGlhbmdvbG8uY29tL2FkdmFuY2VkL2N1c3RvbS1yZXNwb25zZS8pLgorICAgICoqTm90ZSoqOiBgdWpzb25gIGlzIG5vdCBpbmNsdWRlZCB3aXRoIEZhc3RBUEkgYW5kIG11c3QgYmUgaW5zdGFsbGVkCisgICAgc2VwYXJhdGVseSwgZS5nLiBgcGlwIGluc3RhbGwgdWpzb25gLgogICAgICIiIgogCiAgICAgZGVmIHJlbmRlcihzZWxmLCBjb250ZW50OiBBbnkpIC0+IGJ5dGVzOgogICAgICAgICBhc3NlcnQgdWpzb24gaXMgbm90IE5vbmUsICJ1anNvbiBtdXN0IGJlIGluc3RhbGxlZCB0byB1c2UgVUpTT05SZXNwb25zZSIKICAgICAgICAgcmV0dXJuIHVqc29uLmR1bXBzKGNvbnRlbnQsIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCJ1dGYtOCIpCiAKIAorQGRlcHJlY2F0ZWQoCisgICAgIk9SSlNPTlJlc3BvbnNlIGlzIGRlcHJlY2F0ZWQsIEZhc3RBUEkgbm93IHNlcmlhbGl6ZXMgZGF0YSBkaXJlY3RseSB0byBKU09OICIKKyAgICAiYnl0ZXMgdmlhIFB5ZGFudGljIHdoZW4gYSByZXR1cm4gdHlwZSBvciByZXNwb25zZSBtb2RlbCBpcyBzZXQsIHdoaWNoIGlzICIKKyAgICAiZmFzdGVyIGFuZCBkb2Vzbid0IG5lZWQgYSBjdXN0b20gcmVzcG9uc2UgY2xhc3MuIFJlYWQgbW9yZSBpbiB0aGUgRmFzdEFQSSAiCisgICAgImRvY3M6IGh0dHBzOi8vZmFzdGFwaS50aWFuZ29sby5jb20vYWR2YW5jZWQvY3VzdG9tLXJlc3BvbnNlLyNvcmpzb24tb3ItcmVzcG9uc2UtbW9kZWwgIgorICAgICJhbmQgaHR0cHM6Ly9mYXN0YXBpLnRpYW5nb2xvLmNvbS90dXRvcmlhbC9yZXNwb25zZS1tb2RlbC8iLAorICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCisgICAgc3RhY2tsZXZlbD0yLAorKQogY2xhc3MgT1JKU09OUmVzcG9uc2UoSlNPTlJlc3BvbnNlKToKLSAgICAiIiIKLSAgICBKU09OIHJlc3BvbnNlIHVzaW5nIHRoZSBoaWdoLXBlcmZvcm1hbmNlIG9yanNvbiBsaWJyYXJ5IHRvIHNlcmlhbGl6ZSBkYXRhIHRvIEpTT04uCisgICAgIiIiSlNPTiByZXNwb25zZSB1c2luZyB0aGUgb3Jqc29uIGxpYnJhcnkgdG8gc2VyaWFsaXplIGRhdGEgdG8gSlNPTi4KKworICAgICoqRGVwcmVjYXRlZCoqOiBgT1JKU09OUmVzcG9uc2VgIGlzIGRlcHJlY2F0ZWQuIEZhc3RBUEkgbm93IHNlcmlhbGl6ZXMgZGF0YQorICAgIGRpcmVjdGx5IHRvIEpTT04gYnl0ZXMgdmlhIFB5ZGFudGljIHdoZW4gYSByZXR1cm4gdHlwZSBvciByZXNwb25zZSBtb2RlbCBpcworICAgIHNldCwgd2hpY2ggaXMgZmFzdGVyIGFuZCBkb2Vzbid0IG5lZWQgYSBjdXN0b20gcmVzcG9uc2UgY2xhc3MuCisKKyAgICBSZWFkIG1vcmUgaW4gdGhlCisgICAgW0Zhc3RBUEkgZG9jcyBmb3IgQ3VzdG9tIFJlc3BvbnNlXShodHRwczovL2Zhc3RhcGkudGlhbmdvbG8uY29tL2FkdmFuY2VkL2N1c3RvbS1yZXNwb25zZS8jb3Jqc29uLW9yLXJlc3BvbnNlLW1vZGVsKQorICAgIGFuZCB0aGUKKyAgICBbRmFzdEFQSSBkb2NzIGZvciBSZXNwb25zZSBNb2RlbF0oaHR0cHM6Ly9mYXN0YXBpLnRpYW5nb2xvLmNvbS90dXRvcmlhbC9yZXNwb25zZS1tb2RlbC8pLgogCi0gICAgUmVhZCBtb3JlIGFib3V0IGl0IGluIHRoZQotICAgIFtGYXN0QVBJIGRvY3MgZm9yIEN1c3RvbSBSZXNwb25zZSAtIEhUTUwsIFN0cmVhbSwgRmlsZSwgb3RoZXJzXShodHRwczovL2Zhc3RhcGkudGlhbmdvbG8uY29tL2FkdmFuY2VkL2N1c3RvbS1yZXNwb25zZS8pLgorICAgICoqTm90ZSoqOiBgb3Jqc29uYCBpcyBub3QgaW5jbHVkZWQgd2l0aCBGYXN0QVBJIGFuZCBtdXN0IGJlIGluc3RhbGxlZAorICAgIHNlcGFyYXRlbHksIGUuZy4gYHBpcCBpbnN0YWxsIG9yanNvbmAuCiAgICAgIiIiCiAKICAgICBkZWYgcmVuZGVyKHNlbGYsIGNvbnRlbnQ6IEFueSkgLT4gYnl0ZXM6Cg==", "fastapi_15023": "LS0tIGEvZG9jc19zcmMvc3RyZWFtX2RhdGEvdHV0b3JpYWwwMDJfcHkzMTAucHkKKysrIGIvZG9jc19zcmMvc3RyZWFtX2RhdGEvdHV0b3JpYWwwMDJfcHkzMTAucHkKQEAgLTIyLDIzICsyMiwzMyBAQCBjbGFzcyBQTkdTdHJlYW1pbmdSZXNwb25zZShTdHJlYW1pbmdSZXNwb25zZSk6CiAKIEBhcHAuZ2V0KCIvaW1hZ2Uvc3RyZWFtIiwgcmVzcG9uc2VfY2xhc3M9UE5HU3RyZWFtaW5nUmVzcG9uc2UpCiBhc3luYyBkZWYgc3RyZWFtX2ltYWdlKCkgLT4gQXN5bmNJdGVyYWJsZVtieXRlc106Ci0gICAgZm9yIGNodW5rIGluIHJlYWRfaW1hZ2UoKToKLSAgICAgICAgeWllbGQgY2h1bmsKKyAgICB3aXRoIHJlYWRfaW1hZ2UoKSBhcyBpbWFnZV9maWxlOgorICAgICAgICBmb3IgY2h1bmsgaW4gaW1hZ2VfZmlsZToKKyAgICAgICAgICAgIHlpZWxkIGNodW5rCiAKIAogQGFwcC5nZXQoIi9pbWFnZS9zdHJlYW0tbm8tYXN5bmMiLCByZXNwb25zZV9jbGFzcz1QTkdTdHJlYW1pbmdSZXNwb25zZSkKIGRlZiBzdHJlYW1faW1hZ2Vfbm9fYXN5bmMoKSAtPiBJdGVyYWJsZVtieXRlc106Ci0gICAgZm9yIGNodW5rIGluIHJlYWRfaW1hZ2UoKToKLSAgICAgICAgeWllbGQgY2h1bmsKKyAgICB3aXRoIHJlYWRfaW1hZ2UoKSBhcyBpbWFnZV9maWxlOgorICAgICAgICBmb3IgY2h1bmsgaW4gaW1hZ2VfZmlsZToKKyAgICAgICAgICAgIHlpZWxkIGNodW5rCisKKworQGFwcC5nZXQoIi9pbWFnZS9zdHJlYW0tbm8tYXN5bmMteWllbGQtZnJvbSIsIHJlc3BvbnNlX2NsYXNzPVBOR1N0cmVhbWluZ1Jlc3BvbnNlKQorZGVmIHN0cmVhbV9pbWFnZV9ub19hc3luY195aWVsZF9mcm9tKCkgLT4gSXRlcmFibGVbYnl0ZXNdOgorICAgIHdpdGggcmVhZF9pbWFnZSgpIGFzIGltYWdlX2ZpbGU6CisgICAgICAgIHlpZWxkIGZyb20gaW1hZ2VfZmlsZQogCiAKIEBhcHAuZ2V0KCIvaW1hZ2Uvc3RyZWFtLW5vLWFubm90YXRpb24iLCByZXNwb25zZV9jbGFzcz1QTkdTdHJlYW1pbmdSZXNwb25zZSkKIGFzeW5jIGRlZiBzdHJlYW1faW1hZ2Vfbm9fYW5ub3RhdGlvbigpOgotICAgIGZvciBjaHVuayBpbiByZWFkX2ltYWdlKCk6Ci0gICAgICAgIHlpZWxkIGNodW5rCisgICAgd2l0aCByZWFkX2ltYWdlKCkgYXMgaW1hZ2VfZmlsZToKKyAgICAgICAgZm9yIGNodW5rIGluIGltYWdlX2ZpbGU6CisgICAgICAgICAgICB5aWVsZCBjaHVuawogCiAKIEBhcHAuZ2V0KCIvaW1hZ2Uvc3RyZWFtLW5vLWFzeW5jLW5vLWFubm90YXRpb24iLCByZXNwb25zZV9jbGFzcz1QTkdTdHJlYW1pbmdSZXNwb25zZSkKIGRlZiBzdHJlYW1faW1hZ2Vfbm9fYXN5bmNfbm9fYW5ub3RhdGlvbigpOgotICAgIGZvciBjaHVuayBpbiByZWFkX2ltYWdlKCk6Ci0gICAgICAgIHlpZWxkIGNodW5rCisgICAgd2l0aCByZWFkX2ltYWdlKCkgYXMgaW1hZ2VfZmlsZToKKyAgICAgICAgZm9yIGNodW5rIGluIGltYWdlX2ZpbGU6CisgICAgICAgICAgICB5aWVsZCBjaHVuawo=", "fastapi_14986": "LS0tIGEvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKKysrIGIvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKQEAgLTExMDEsMTYgKzExMDEsMTggQEAgZGVmIG9wZW5hcGkoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAKICAgICBkZWYgc2V0dXAoc2VsZikgLT4gTm9uZToKICAgICAgICAgaWYgc2VsZi5vcGVuYXBpX3VybDoKLSAgICAgICAgICAgIHVybHMgPSAoc2VydmVyX2RhdGEuZ2V0KCJ1cmwiKSBmb3Igc2VydmVyX2RhdGEgaW4gc2VsZi5zZXJ2ZXJzKQotICAgICAgICAgICAgc2VydmVyX3VybHMgPSB7dXJsIGZvciB1cmwgaW4gdXJscyBpZiB1cmx9CiAKICAgICAgICAgICAgIGFzeW5jIGRlZiBvcGVuYXBpKHJlcTogUmVxdWVzdCkgLT4gSlNPTlJlc3BvbnNlOgogICAgICAgICAgICAgICAgIHJvb3RfcGF0aCA9IHJlcS5zY29wZS5nZXQoInJvb3RfcGF0aCIsICIiKS5yc3RyaXAoIi8iKQotICAgICAgICAgICAgICAgIGlmIHJvb3RfcGF0aCBub3QgaW4gc2VydmVyX3VybHM6Ci0gICAgICAgICAgICAgICAgICAgIGlmIHJvb3RfcGF0aCBhbmQgc2VsZi5yb290X3BhdGhfaW5fc2VydmVyczoKLSAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuc2VydmVycy5pbnNlcnQoMCwgeyJ1cmwiOiByb290X3BhdGh9KQotICAgICAgICAgICAgICAgICAgICAgICAgc2VydmVyX3VybHMuYWRkKHJvb3RfcGF0aCkKLSAgICAgICAgICAgICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKHNlbGYub3BlbmFwaSgpKQorICAgICAgICAgICAgICAgIHNjaGVtYSA9IHNlbGYub3BlbmFwaSgpCisgICAgICAgICAgICAgICAgaWYgcm9vdF9wYXRoIGFuZCBzZWxmLnJvb3RfcGF0aF9pbl9zZXJ2ZXJzOgorICAgICAgICAgICAgICAgICAgICBzZXJ2ZXJfdXJscyA9IHtzLmdldCgidXJsIikgZm9yIHMgaW4gc2NoZW1hLmdldCgic2VydmVycyIsIFtdKX0KKyAgICAgICAgICAgICAgICAgICAgaWYgcm9vdF9wYXRoIG5vdCBpbiBzZXJ2ZXJfdXJsczoKKyAgICAgICAgICAgICAgICAgICAgICAgIHNjaGVtYSA9IGRpY3Qoc2NoZW1hKQorICAgICAgICAgICAgICAgICAgICAgICAgc2NoZW1hWyJzZXJ2ZXJzIl0gPSBbeyJ1cmwiOiByb290X3BhdGh9XSArIHNjaGVtYS5nZXQoCisgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNlcnZlcnMiLCBbXQorICAgICAgICAgICAgICAgICAgICAgICAgKQorICAgICAgICAgICAgICAgIHJldHVybiBKU09OUmVzcG9uc2Uoc2NoZW1hKQogCiAgICAgICAgICAgICBzZWxmLmFkZF9yb3V0ZShzZWxmLm9wZW5hcGlfdXJsLCBvcGVuYXBpLCBpbmNsdWRlX2luX3NjaGVtYT1GYWxzZSkKICAgICAgICAgaWYgc2VsZi5vcGVuYXBpX3VybCBhbmQgc2VsZi5kb2NzX3VybDoKLS0tIGEvZmFzdGFwaS9vcGVuYXBpL2RvY3MucHkKKysrIGIvZmFzdGFwaS9vcGVuYXBpL2RvY3MucHkKQEAgLTUsNiArNSwyMCBAQAogZnJvbSBmYXN0YXBpLmVuY29kZXJzIGltcG9ydCBqc29uYWJsZV9lbmNvZGVyCiBmcm9tIHN0YXJsZXR0ZS5yZXNwb25zZXMgaW1wb3J0IEhUTUxSZXNwb25zZQogCisKK2RlZiBfaHRtbF9zYWZlX2pzb24odmFsdWU6IEFueSkgLT4gc3RyOgorICAgICIiIlNlcmlhbGl6ZSBhIHZhbHVlIHRvIEpTT04gd2l0aCBIVE1MIHNwZWNpYWwgY2hhcmFjdGVycyBlc2NhcGVkLgorCisgICAgVGhpcyBwcmV2ZW50cyBpbmplY3Rpb24gd2hlbiB0aGUgSlNPTiBpcyBlbWJlZGRlZCBpbnNpZGUgYSA8c2NyaXB0PiB0YWcuCisgICAgIiIiCisgICAgcmV0dXJuICgKKyAgICAgICAganNvbi5kdW1wcyh2YWx1ZSkKKyAgICAgICAgLnJlcGxhY2UoIjwiLCAiXFx1MDAzYyIpCisgICAgICAgIC5yZXBsYWNlKCI+IiwgIlxcdTAwM2UiKQorICAgICAgICAucmVwbGFjZSgiJiIsICJcXHUwMDI2IikKKyAgICApCisKKwogc3dhZ2dlcl91aV9kZWZhdWx0X3BhcmFtZXRlcnM6IEFubm90YXRlZFsKICAgICBkaWN0W3N0ciwgQW55XSwKICAgICBEb2MoCkBAIC0xNTUsNyArMTY5LDcgQEAgZGVmIGdldF9zd2FnZ2VyX3VpX2h0bWwoCiAgICAgIiIiCiAKICAgICBmb3Iga2V5LCB2YWx1ZSBpbiBjdXJyZW50X3N3YWdnZXJfdWlfcGFyYW1ldGVycy5pdGVtcygpOgotICAgICAgICBodG1sICs9IGYie2pzb24uZHVtcHMoa2V5KX06IHtqc29uLmR1bXBzKGpzb25hYmxlX2VuY29kZXIodmFsdWUpKX0sXG4iCisgICAgICAgIGh0bWwgKz0gZiJ7X2h0bWxfc2FmZV9qc29uKGtleSl9OiB7X2h0bWxfc2FmZV9qc29uKGpzb25hYmxlX2VuY29kZXIodmFsdWUpKX0sXG4iCiAKICAgICBpZiBvYXV0aDJfcmVkaXJlY3RfdXJsOgogICAgICAgICBodG1sICs9IGYib2F1dGgyUmVkaXJlY3RVcmw6IHdpbmRvdy5sb2NhdGlvbi5vcmlnaW4gKyAne29hdXRoMl9yZWRpcmVjdF91cmx9JywiCkBAIC0xNjksNyArMTgzLDcgQEAgZGVmIGdldF9zd2FnZ2VyX3VpX2h0bWwoCiAKICAgICBpZiBpbml0X29hdXRoOgogICAgICAgICBodG1sICs9IGYiIiIKLSAgICAgICAgdWkuaW5pdE9BdXRoKHtqc29uLmR1bXBzKGpzb25hYmxlX2VuY29kZXIoaW5pdF9vYXV0aCkpfSkKKyAgICAgICAgdWkuaW5pdE9BdXRoKHtfaHRtbF9zYWZlX2pzb24oanNvbmFibGVfZW5jb2Rlcihpbml0X29hdXRoKSl9KQogICAgICAgICAiIiIKIAogICAgIGh0bWwgKz0gIiIiCg==", "fastapi_14978": "LS0tIGEvZG9jc19zcmMvc3RyaWN0X2NvbnRlbnRfdHlwZS90dXRvcmlhbDAwMV9weTMxMC5weQorKysgYi9kb2NzX3NyYy9zdHJpY3RfY29udGVudF90eXBlL3R1dG9yaWFsMDAxX3B5MzEwLnB5CkBAIC0wLDAgKzEsMTQgQEAKK2Zyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSQorZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsCisKK2FwcCA9IEZhc3RBUEkoc3RyaWN0X2NvbnRlbnRfdHlwZT1GYWxzZSkKKworCitjbGFzcyBJdGVtKEJhc2VNb2RlbCk6CisgICAgbmFtZTogc3RyCisgICAgcHJpY2U6IGZsb2F0CisKKworQGFwcC5wb3N0KCIvaXRlbXMvIikKK2FzeW5jIGRlZiBjcmVhdGVfaXRlbShpdGVtOiBJdGVtKToKKyAgICByZXR1cm4gaXRlbQotLS0gYS9mYXN0YXBpL2FwcGxpY2F0aW9ucy5weQorKysgYi9mYXN0YXBpL2FwcGxpY2F0aW9ucy5weQpAQCAtODQwLDYgKzg0MCwyOSBAQCBjbGFzcyBJdGVtKEJhc2VNb2RlbCk6CiAgICAgICAgICAgICAgICAgIiIiCiAgICAgICAgICAgICApLAogICAgICAgICBdID0gTm9uZSwKKyAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZTogQW5ub3RhdGVkWworICAgICAgICAgICAgYm9vbCwKKyAgICAgICAgICAgIERvYygKKyAgICAgICAgICAgICAgICAiIiIKKyAgICAgICAgICAgICAgICBFbmFibGUgc3RyaWN0IGNoZWNraW5nIGZvciByZXF1ZXN0IENvbnRlbnQtVHlwZSBoZWFkZXJzLgorCisgICAgICAgICAgICAgICAgV2hlbiBgVHJ1ZWAgKHRoZSBkZWZhdWx0KSwgcmVxdWVzdHMgd2l0aCBhIGJvZHkgdGhhdCBkbyBub3QgaW5jbHVkZQorICAgICAgICAgICAgICAgIGEgYENvbnRlbnQtVHlwZWAgaGVhZGVyIHdpbGwgKipub3QqKiBiZSBwYXJzZWQgYXMgSlNPTi4KKworICAgICAgICAgICAgICAgIFRoaXMgcHJldmVudHMgcG90ZW50aWFsIGNyb3NzLXNpdGUgcmVxdWVzdCBmb3JnZXJ5IChDU1JGKSBhdHRhY2tzCisgICAgICAgICAgICAgICAgdGhhdCBleHBsb2l0IHRoZSBicm93c2VyJ3MgYWJpbGl0eSB0byBzZW5kIHJlcXVlc3RzIHdpdGhvdXQgYQorICAgICAgICAgICAgICAgIENvbnRlbnQtVHlwZSBoZWFkZXIsIGJ5cGFzc2luZyBDT1JTIHByZWZsaWdodCBjaGVja3MuIEluIHBhcnRpY3VsYXIKKyAgICAgICAgICAgICAgICBhcHBsaWNhYmxlIGZvciBhcHBzIHRoYXQgbmVlZCB0byBiZSBydW4gbG9jYWxseSAoaW4gbG9jYWxob3N0KS4KKworICAgICAgICAgICAgICAgIFdoZW4gYEZhbHNlYCwgcmVxdWVzdHMgd2l0aG91dCBhIGBDb250ZW50LVR5cGVgIGhlYWRlciB3aWxsIGhhdmUKKyAgICAgICAgICAgICAgICB0aGVpciBib2R5IHBhcnNlZCBhcyBKU09OLCB3aGljaCBtYWludGFpbnMgY29tcGF0aWJpbGl0eSB3aXRoCisgICAgICAgICAgICAgICAgY2VydGFpbiBjbGllbnRzIHRoYXQgZG9uJ3Qgc2VuZCBgQ29udGVudC1UeXBlYCBoZWFkZXJzLgorCisgICAgICAgICAgICAgICAgUmVhZCBtb3JlIGFib3V0IGl0IGluIHRoZQorICAgICAgICAgICAgICAgIFtGYXN0QVBJIGRvY3MgZm9yIFN0cmljdCBDb250ZW50LVR5cGVdKGh0dHBzOi8vZmFzdGFwaS50aWFuZ29sby5jb20vYWR2YW5jZWQvc3RyaWN0LWNvbnRlbnQtdHlwZS8pLgorICAgICAgICAgICAgICAgICIiIgorICAgICAgICAgICAgKSwKKyAgICAgICAgXSA9IFRydWUsCiAgICAgICAgICoqZXh0cmE6IEFubm90YXRlZFsKICAgICAgICAgICAgIEFueSwKICAgICAgICAgICAgIERvYygKQEAgLTk3NCw2ICs5OTcsNyBAQCBjbGFzcyBJdGVtKEJhc2VNb2RlbCk6CiAgICAgICAgICAgICBpbmNsdWRlX2luX3NjaGVtYT1pbmNsdWRlX2luX3NjaGVtYSwKICAgICAgICAgICAgIHJlc3BvbnNlcz1yZXNwb25zZXMsCiAgICAgICAgICAgICBnZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb249Z2VuZXJhdGVfdW5pcXVlX2lkX2Z1bmN0aW9uLAorICAgICAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZT1zdHJpY3RfY29udGVudF90eXBlLAogICAgICAgICApCiAgICAgICAgIHNlbGYuZXhjZXB0aW9uX2hhbmRsZXJzOiBkaWN0WwogICAgICAgICAgICAgQW55LCBDYWxsYWJsZVtbUmVxdWVzdCwgQW55XSwgUmVzcG9uc2UgfCBBd2FpdGFibGVbUmVzcG9uc2VdXQotLS0gYS9mYXN0YXBpL3JvdXRpbmcucHkKKysrIGIvZmFzdGFwaS9yb3V0aW5nLnB5CkBAIC0zMjksNiArMzI5LDcgQEAgZGVmIGdldF9yZXF1ZXN0X2hhbmRsZXIoCiAgICAgcmVzcG9uc2VfbW9kZWxfZXhjbHVkZV9ub25lOiBib29sID0gRmFsc2UsCiAgICAgZGVwZW5kZW5jeV9vdmVycmlkZXNfcHJvdmlkZXI6IEFueSB8IE5vbmUgPSBOb25lLAogICAgIGVtYmVkX2JvZHlfZmllbGRzOiBib29sID0gRmFsc2UsCisgICAgc3RyaWN0X2NvbnRlbnRfdHlwZTogYm9vbCB8IERlZmF1bHRQbGFjZWhvbGRlciA9IERlZmF1bHQoVHJ1ZSksCiApIC0+IENhbGxhYmxlW1tSZXF1ZXN0XSwgQ29yb3V0aW5lW0FueSwgQW55LCBSZXNwb25zZV1dOgogICAgIGFzc2VydCBkZXBlbmRhbnQuY2FsbCBpcyBub3QgTm9uZSwgImRlcGVuZGFudC5jYWxsIG11c3QgYmUgYSBmdW5jdGlvbiIKICAgICBpc19jb3JvdXRpbmUgPSBkZXBlbmRhbnQuaXNfY29yb3V0aW5lX2NhbGxhYmxlCkBAIC0zMzcsNiArMzM4LDEwIEBAIGRlZiBnZXRfcmVxdWVzdF9oYW5kbGVyKAogICAgICAgICBhY3R1YWxfcmVzcG9uc2VfY2xhc3M6IHR5cGVbUmVzcG9uc2VdID0gcmVzcG9uc2VfY2xhc3MudmFsdWUKICAgICBlbHNlOgogICAgICAgICBhY3R1YWxfcmVzcG9uc2VfY2xhc3MgPSByZXNwb25zZV9jbGFzcworICAgIGlmIGlzaW5zdGFuY2Uoc3RyaWN0X2NvbnRlbnRfdHlwZSwgRGVmYXVsdFBsYWNlaG9sZGVyKToKKyAgICAgICAgYWN0dWFsX3N0cmljdF9jb250ZW50X3R5cGU6IGJvb2wgPSBzdHJpY3RfY29udGVudF90eXBlLnZhbHVlCisgICAgZWxzZToKKyAgICAgICAgYWN0dWFsX3N0cmljdF9jb250ZW50X3R5cGUgPSBzdHJpY3RfY29udGVudF90eXBlCiAKICAgICBhc3luYyBkZWYgYXBwKHJlcXVlc3Q6IFJlcXVlc3QpIC0+IFJlc3BvbnNlOgogICAgICAgICByZXNwb25zZTogUmVzcG9uc2UgfCBOb25lID0gTm9uZQpAQCAtMzcwLDcgKzM3NSw4IEBAIGFzeW5jIGRlZiBhcHAocmVxdWVzdDogUmVxdWVzdCkgLT4gUmVzcG9uc2U6CiAgICAgICAgICAgICAgICAgICAgICAgICBqc29uX2JvZHk6IEFueSA9IFVuZGVmaW5lZAogICAgICAgICAgICAgICAgICAgICAgICAgY29udGVudF90eXBlX3ZhbHVlID0gcmVxdWVzdC5oZWFkZXJzLmdldCgiY29udGVudC10eXBlIikKICAgICAgICAgICAgICAgICAgICAgICAgIGlmIG5vdCBjb250ZW50X3R5cGVfdmFsdWU6Ci0gICAgICAgICAgICAgICAgICAgICAgICAgICAganNvbl9ib2R5ID0gYXdhaXQgcmVxdWVzdC5qc29uKCkKKyAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBub3QgYWN0dWFsX3N0cmljdF9jb250ZW50X3R5cGU6CisgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGpzb25fYm9keSA9IGF3YWl0IHJlcXVlc3QuanNvbigpCiAgICAgICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1lc3NhZ2UgPSBlbWFpbC5tZXNzYWdlLk1lc3NhZ2UoKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1lc3NhZ2VbImNvbnRlbnQtdHlwZSJdID0gY29udGVudF90eXBlX3ZhbHVlCkBAIC01OTksNiArNjA1LDcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICBvcGVuYXBpX2V4dHJhOiBkaWN0W3N0ciwgQW55XSB8IE5vbmUgPSBOb25lLAogICAgICAgICBnZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb246IENhbGxhYmxlW1siQVBJUm91dGUiXSwgc3RyXQogICAgICAgICB8IERlZmF1bHRQbGFjZWhvbGRlciA9IERlZmF1bHQoZ2VuZXJhdGVfdW5pcXVlX2lkKSwKKyAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZTogYm9vbCB8IERlZmF1bHRQbGFjZWhvbGRlciA9IERlZmF1bHQoVHJ1ZSksCiAgICAgKSAtPiBOb25lOgogICAgICAgICBzZWxmLnBhdGggPSBwYXRoCiAgICAgICAgIHNlbGYuZW5kcG9pbnQgPSBlbmRwb2ludApAQCAtNjI1LDYgKzYzMiw3IEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgc2VsZi5jYWxsYmFja3MgPSBjYWxsYmFja3MKICAgICAgICAgc2VsZi5vcGVuYXBpX2V4dHJhID0gb3BlbmFwaV9leHRyYQogICAgICAgICBzZWxmLmdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbiA9IGdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbgorICAgICAgICBzZWxmLnN0cmljdF9jb250ZW50X3R5cGUgPSBzdHJpY3RfY29udGVudF90eXBlCiAgICAgICAgIHNlbGYudGFncyA9IHRhZ3Mgb3IgW10KICAgICAgICAgc2VsZi5yZXNwb25zZXMgPSByZXNwb25zZXMgb3Ige30KICAgICAgICAgc2VsZi5uYW1lID0gZ2V0X25hbWUoZW5kcG9pbnQpIGlmIG5hbWUgaXMgTm9uZSBlbHNlIG5hbWUKQEAgLTcxMyw2ICs3MjEsNyBAQCBkZWYgZ2V0X3JvdXRlX2hhbmRsZXIoc2VsZikgLT4gQ2FsbGFibGVbW1JlcXVlc3RdLCBDb3JvdXRpbmVbQW55LCBBbnksIFJlc3BvbnNlXQogICAgICAgICAgICAgcmVzcG9uc2VfbW9kZWxfZXhjbHVkZV9ub25lPXNlbGYucmVzcG9uc2VfbW9kZWxfZXhjbHVkZV9ub25lLAogICAgICAgICAgICAgZGVwZW5kZW5jeV9vdmVycmlkZXNfcHJvdmlkZXI9c2VsZi5kZXBlbmRlbmN5X292ZXJyaWRlc19wcm92aWRlciwKICAgICAgICAgICAgIGVtYmVkX2JvZHlfZmllbGRzPXNlbGYuX2VtYmVkX2JvZHlfZmllbGRzLAorICAgICAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZT1zZWxmLnN0cmljdF9jb250ZW50X3R5cGUsCiAgICAgICAgICkKIAogICAgIGRlZiBtYXRjaGVzKHNlbGYsIHNjb3BlOiBTY29wZSkgLT4gdHVwbGVbTWF0Y2gsIFNjb3BlXToKQEAgLTk2Myw2ICs5NzIsMjkgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICAgICAgICAgICIiIgogICAgICAgICAgICAgKSwKICAgICAgICAgXSA9IERlZmF1bHQoZ2VuZXJhdGVfdW5pcXVlX2lkKSwKKyAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZTogQW5ub3RhdGVkWworICAgICAgICAgICAgYm9vbCwKKyAgICAgICAgICAgIERvYygKKyAgICAgICAgICAgICAgICAiIiIKKyAgICAgICAgICAgICAgICBFbmFibGUgc3RyaWN0IGNoZWNraW5nIGZvciByZXF1ZXN0IENvbnRlbnQtVHlwZSBoZWFkZXJzLgorCisgICAgICAgICAgICAgICAgV2hlbiBgVHJ1ZWAgKHRoZSBkZWZhdWx0KSwgcmVxdWVzdHMgd2l0aCBhIGJvZHkgdGhhdCBkbyBub3QgaW5jbHVkZQorICAgICAgICAgICAgICAgIGEgYENvbnRlbnQtVHlwZWAgaGVhZGVyIHdpbGwgKipub3QqKiBiZSBwYXJzZWQgYXMgSlNPTi4KKworICAgICAgICAgICAgICAgIFRoaXMgcHJldmVudHMgcG90ZW50aWFsIGNyb3NzLXNpdGUgcmVxdWVzdCBmb3JnZXJ5IChDU1JGKSBhdHRhY2tzCisgICAgICAgICAgICAgICAgdGhhdCBleHBsb2l0IHRoZSBicm93c2VyJ3MgYWJpbGl0eSB0byBzZW5kIHJlcXVlc3RzIHdpdGhvdXQgYQorICAgICAgICAgICAgICAgIENvbnRlbnQtVHlwZSBoZWFkZXIsIGJ5cGFzc2luZyBDT1JTIHByZWZsaWdodCBjaGVja3MuIEluIHBhcnRpY3VsYXIKKyAgICAgICAgICAgICAgICBhcHBsaWNhYmxlIGZvciBhcHBzIHRoYXQgbmVlZCB0byBiZSBydW4gbG9jYWxseSAoaW4gbG9jYWxob3N0KS4KKworICAgICAgICAgICAgICAgIFdoZW4gYEZhbHNlYCwgcmVxdWVzdHMgd2l0aG91dCBhIGBDb250ZW50LVR5cGVgIGhlYWRlciB3aWxsIGhhdmUKKyAgICAgICAgICAgICAgICB0aGVpciBib2R5IHBhcnNlZCBhcyBKU09OLCB3aGljaCBtYWludGFpbnMgY29tcGF0aWJpbGl0eSB3aXRoCisgICAgICAgICAgICAgICAgY2VydGFpbiBjbGllbnRzIHRoYXQgZG9uJ3Qgc2VuZCBgQ29udGVudC1UeXBlYCBoZWFkZXJzLgorCisgICAgICAgICAgICAgICAgUmVhZCBtb3JlIGFib3V0IGl0IGluIHRoZQorICAgICAgICAgICAgICAgIFtGYXN0QVBJIGRvY3MgZm9yIFN0cmljdCBDb250ZW50LVR5cGVdKGh0dHBzOi8vZmFzdGFwaS50aWFuZ29sby5jb20vYWR2YW5jZWQvc3RyaWN0LWNvbnRlbnQtdHlwZS8pLgorICAgICAgICAgICAgICAgICIiIgorICAgICAgICAgICAgKSwKKyAgICAgICAgXSA9IERlZmF1bHQoVHJ1ZSksCiAgICAgKSAtPiBOb25lOgogICAgICAgICAjIERldGVybWluZSB0aGUgbGlmZXNwYW4gY29udGV4dCB0byB1c2UKICAgICAgICAgaWYgbGlmZXNwYW4gaXMgTm9uZToKQEAgLTEwMDksNiArMTA0MSw3IEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgc2VsZi5yb3V0ZV9jbGFzcyA9IHJvdXRlX2NsYXNzCiAgICAgICAgIHNlbGYuZGVmYXVsdF9yZXNwb25zZV9jbGFzcyA9IGRlZmF1bHRfcmVzcG9uc2VfY2xhc3MKICAgICAgICAgc2VsZi5nZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb24gPSBnZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb24KKyAgICAgICAgc2VsZi5zdHJpY3RfY29udGVudF90eXBlID0gc3RyaWN0X2NvbnRlbnRfdHlwZQogCiAgICAgZGVmIHJvdXRlKAogICAgICAgICBzZWxmLApAQCAtMTA1OSw2ICsxMDkyLDcgQEAgZGVmIGFkZF9hcGlfcm91dGUoCiAgICAgICAgIG9wZW5hcGlfZXh0cmE6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgIGdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbjogQ2FsbGFibGVbW0FQSVJvdXRlXSwgc3RyXQogICAgICAgICB8IERlZmF1bHRQbGFjZWhvbGRlciA9IERlZmF1bHQoZ2VuZXJhdGVfdW5pcXVlX2lkKSwKKyAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZTogYm9vbCB8IERlZmF1bHRQbGFjZWhvbGRlciA9IERlZmF1bHQoVHJ1ZSksCiAgICAgKSAtPiBOb25lOgogICAgICAgICByb3V0ZV9jbGFzcyA9IHJvdXRlX2NsYXNzX292ZXJyaWRlIG9yIHNlbGYucm91dGVfY2xhc3MKICAgICAgICAgcmVzcG9uc2VzID0gcmVzcG9uc2VzIG9yIHt9CkBAIC0xMTA1LDYgKzExMzksOSBAQCBkZWYgYWRkX2FwaV9yb3V0ZSgKICAgICAgICAgICAgIGNhbGxiYWNrcz1jdXJyZW50X2NhbGxiYWNrcywKICAgICAgICAgICAgIG9wZW5hcGlfZXh0cmE9b3BlbmFwaV9leHRyYSwKICAgICAgICAgICAgIGdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbj1jdXJyZW50X2dlbmVyYXRlX3VuaXF1ZV9pZCwKKyAgICAgICAgICAgIHN0cmljdF9jb250ZW50X3R5cGU9Z2V0X3ZhbHVlX29yX2RlZmF1bHQoCisgICAgICAgICAgICAgICAgc3RyaWN0X2NvbnRlbnRfdHlwZSwgc2VsZi5zdHJpY3RfY29udGVudF90eXBlCisgICAgICAgICAgICApLAogICAgICAgICApCiAgICAgICAgIHNlbGYucm91dGVzLmFwcGVuZChyb3V0ZSkKIApAQCAtMTQ4MCw2ICsxNTE3LDExIEBAIGRlZiByZWFkX3VzZXJzKCk6CiAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrcz1jdXJyZW50X2NhbGxiYWNrcywKICAgICAgICAgICAgICAgICAgICAgb3BlbmFwaV9leHRyYT1yb3V0ZS5vcGVuYXBpX2V4dHJhLAogICAgICAgICAgICAgICAgICAgICBnZW5lcmF0ZV91bmlxdWVfaWRfZnVuY3Rpb249Y3VycmVudF9nZW5lcmF0ZV91bmlxdWVfaWQsCisgICAgICAgICAgICAgICAgICAgIHN0cmljdF9jb250ZW50X3R5cGU9Z2V0X3ZhbHVlX29yX2RlZmF1bHQoCisgICAgICAgICAgICAgICAgICAgICAgICByb3V0ZS5zdHJpY3RfY29udGVudF90eXBlLAorICAgICAgICAgICAgICAgICAgICAgICAgcm91dGVyLnN0cmljdF9jb250ZW50X3R5cGUsCisgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLnN0cmljdF9jb250ZW50X3R5cGUsCisgICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgZWxpZiBpc2luc3RhbmNlKHJvdXRlLCByb3V0aW5nLlJvdXRlKToKICAgICAgICAgICAgICAgICBtZXRob2RzID0gbGlzdChyb3V0ZS5tZXRob2RzIG9yIFtdKQo=", "fastapi_14258": "LS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtMTM5Myw2ICsxMzkzLDEwIEBAIGRlZiByZWFkX3VzZXJzKCk6CiAgICAgICAgIGFwcC5pbmNsdWRlX3JvdXRlcihpbnRlcm5hbF9yb3V0ZXIpCiAgICAgICAgIGBgYAogICAgICAgICAiIiIKKyAgICAgICAgYXNzZXJ0IHNlbGYgaXMgbm90IHJvdXRlciwgKAorICAgICAgICAgICAgIkNhbm5vdCBpbmNsdWRlIHRoZSBzYW1lIEFQSVJvdXRlciBpbnN0YW5jZSBpbnRvIGl0c2VsZi4gIgorICAgICAgICAgICAgIkRpZCB5b3UgbWVhbiB0byBpbmNsdWRlIGEgZGlmZmVyZW50IHJvdXRlcj8iCisgICAgICAgICkKICAgICAgICAgaWYgcHJlZml4OgogICAgICAgICAgICAgYXNzZXJ0IHByZWZpeC5zdGFydHN3aXRoKCIvIiksICJBIHBhdGggcHJlZml4IG11c3Qgc3RhcnQgd2l0aCAnLyciCiAgICAgICAgICAgICBhc3NlcnQgbm90IHByZWZpeC5lbmRzd2l0aCgiLyIpLCAoCg==", "fastapi_14851": "LS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtMSwyMiArMSwzMSBAQAoraW1wb3J0IGNvbnRleHRsaWIKIGltcG9ydCBlbWFpbC5tZXNzYWdlCiBpbXBvcnQgZnVuY3Rvb2xzCiBpbXBvcnQgaW5zcGVjdAogaW1wb3J0IGpzb24KK2ltcG9ydCB0eXBlcwogZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0ICgKICAgICBBc3luY0l0ZXJhdG9yLAogICAgIEF3YWl0YWJsZSwKICAgICBDb2xsZWN0aW9uLAogICAgIENvcm91dGluZSwKKyAgICBHZW5lcmF0b3IsCiAgICAgTWFwcGluZywKICAgICBTZXF1ZW5jZSwKICkKLWZyb20gY29udGV4dGxpYiBpbXBvcnQgQXN5bmNFeGl0U3RhY2ssIGFzeW5jY29udGV4dG1hbmFnZXIKK2Zyb20gY29udGV4dGxpYiBpbXBvcnQgKAorICAgIEFic3RyYWN0QXN5bmNDb250ZXh0TWFuYWdlciwKKyAgICBBYnN0cmFjdENvbnRleHRNYW5hZ2VyLAorICAgIEFzeW5jRXhpdFN0YWNrLAorICAgIGFzeW5jY29udGV4dG1hbmFnZXIsCispCiBmcm9tIGVudW0gaW1wb3J0IEVudW0sIEludEVudW0KIGZyb20gdHlwaW5nIGltcG9ydCAoCiAgICAgQW5ub3RhdGVkLAogICAgIEFueSwKICAgICBDYWxsYWJsZSwKICAgICBPcHRpb25hbCwKKyAgICBUeXBlVmFyLAogICAgIFVuaW9uLAogKQogCkBAIC0xNDMsNiArMTUyLDUwIEBAIGFzeW5jIGRlZiBhcHAoc2NvcGU6IFNjb3BlLCByZWNlaXZlOiBSZWNlaXZlLCBzZW5kOiBTZW5kKSAtPiBOb25lOgogICAgIHJldHVybiBhcHAKIAogCitfVCA9IFR5cGVWYXIoIl9UIikKKworCisjIFZlbmRvcmVkIGZyb20gc3RhcmxldHRlLnJvdXRpbmcgdG8gYXZvaWQgaW1wb3J0aW5nIHByaXZhdGUgc3ltYm9scworY2xhc3MgX0FzeW5jTGlmdENvbnRleHRNYW5hZ2VyKEFic3RyYWN0QXN5bmNDb250ZXh0TWFuYWdlcltfVF0pOgorICAgICIiIgorICAgIFdyYXBzIGEgc3luY2hyb25vdXMgY29udGV4dCBtYW5hZ2VyIHRvIG1ha2UgaXQgYXN5bmMuCisKKyAgICBUaGlzIGlzIHZlbmRvcmVkIGZyb20gU3RhcmxldHRlIHRvIGF2b2lkIGltcG9ydGluZyBwcml2YXRlIHN5bWJvbHMuCisgICAgIiIiCisKKyAgICBkZWYgX19pbml0X18oc2VsZiwgY206IEFic3RyYWN0Q29udGV4dE1hbmFnZXJbX1RdKSAtPiBOb25lOgorICAgICAgICBzZWxmLl9jbSA9IGNtCisKKyAgICBhc3luYyBkZWYgX19hZW50ZXJfXyhzZWxmKSAtPiBfVDoKKyAgICAgICAgcmV0dXJuIHNlbGYuX2NtLl9fZW50ZXJfXygpCisKKyAgICBhc3luYyBkZWYgX19hZXhpdF9fKAorICAgICAgICBzZWxmLAorICAgICAgICBleGNfdHlwZTogT3B0aW9uYWxbdHlwZVtCYXNlRXhjZXB0aW9uXV0sCisgICAgICAgIGV4Y192YWx1ZTogT3B0aW9uYWxbQmFzZUV4Y2VwdGlvbl0sCisgICAgICAgIHRyYWNlYmFjazogT3B0aW9uYWxbdHlwZXMuVHJhY2ViYWNrVHlwZV0sCisgICAgKSAtPiBPcHRpb25hbFtib29sXToKKyAgICAgICAgcmV0dXJuIHNlbGYuX2NtLl9fZXhpdF9fKGV4Y190eXBlLCBleGNfdmFsdWUsIHRyYWNlYmFjaykKKworCisjIFZlbmRvcmVkIGZyb20gc3RhcmxldHRlLnJvdXRpbmcgdG8gYXZvaWQgaW1wb3J0aW5nIHByaXZhdGUgc3ltYm9scworZGVmIF93cmFwX2dlbl9saWZlc3Bhbl9jb250ZXh0KAorICAgIGxpZmVzcGFuX2NvbnRleHQ6IENhbGxhYmxlW1tBbnldLCBHZW5lcmF0b3JbQW55LCBBbnksIEFueV1dLAorKSAtPiBDYWxsYWJsZVtbQW55XSwgQWJzdHJhY3RBc3luY0NvbnRleHRNYW5hZ2VyW0FueV1dOgorICAgICIiIgorICAgIFdyYXAgYSBnZW5lcmF0b3ItYmFzZWQgbGlmZXNwYW4gY29udGV4dCBpbnRvIGFuIGFzeW5jIGNvbnRleHQgbWFuYWdlci4KKworICAgIFRoaXMgaXMgdmVuZG9yZWQgZnJvbSBTdGFybGV0dGUgdG8gYXZvaWQgaW1wb3J0aW5nIHByaXZhdGUgc3ltYm9scy4KKyAgICAiIiIKKyAgICBjbWdyID0gY29udGV4dGxpYi5jb250ZXh0bWFuYWdlcihsaWZlc3Bhbl9jb250ZXh0KQorCisgICAgQGZ1bmN0b29scy53cmFwcyhjbWdyKQorICAgIGRlZiB3cmFwcGVyKGFwcDogQW55KSAtPiBfQXN5bmNMaWZ0Q29udGV4dE1hbmFnZXJbQW55XToKKyAgICAgICAgcmV0dXJuIF9Bc3luY0xpZnRDb250ZXh0TWFuYWdlcihjbWdyKGFwcCkpCisKKyAgICByZXR1cm4gd3JhcHBlcgorCisKIGRlZiBfbWVyZ2VfbGlmZXNwYW5fY29udGV4dCgKICAgICBvcmlnaW5hbF9jb250ZXh0OiBMaWZlc3BhbltBbnldLCBuZXN0ZWRfY29udGV4dDogTGlmZXNwYW5bQW55XQogKSAtPiBMaWZlc3BhbltBbnldOgpAQCAtMTYwLDYgKzIxMywzMCBAQCBhc3luYyBkZWYgbWVyZ2VkX2xpZmVzcGFuKAogICAgIHJldHVybiBtZXJnZWRfbGlmZXNwYW4gICMgdHlwZTogaWdub3JlW3JldHVybi12YWx1ZV0KIAogCitjbGFzcyBfRGVmYXVsdExpZmVzcGFuOgorICAgICIiIgorICAgIERlZmF1bHQgbGlmZXNwYW4gY29udGV4dCBtYW5hZ2VyIHRoYXQgcnVucyBvbl9zdGFydHVwIGFuZCBvbl9zaHV0ZG93biBoYW5kbGVycy4KKworICAgIFRoaXMgaXMgYSBjb3B5IG9mIHRoZSBTdGFybGV0dGUgX0RlZmF1bHRMaWZlc3BhbiBjbGFzcyB0aGF0IHdhcyByZW1vdmVkCisgICAgaW4gU3RhcmxldHRlLiBGYXN0QVBJIGtlZXBzIGl0IHRvIG1haW50YWluIGJhY2t3YXJkIGNvbXBhdGliaWxpdHkgd2l0aAorICAgIG9uX3N0YXJ0dXAgYW5kIG9uX3NodXRkb3duIGV2ZW50IGhhbmRsZXJzLgorCisgICAgUmVmOiBodHRwczovL2dpdGh1Yi5jb20vS2x1ZGV4L3N0YXJsZXR0ZS9wdWxsLzMxMTcKKyAgICAiIiIKKworICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb3V0ZXI6ICJBUElSb3V0ZXIiKSAtPiBOb25lOgorICAgICAgICBzZWxmLl9yb3V0ZXIgPSByb3V0ZXIKKworICAgIGFzeW5jIGRlZiBfX2FlbnRlcl9fKHNlbGYpIC0+IE5vbmU6CisgICAgICAgIGF3YWl0IHNlbGYuX3JvdXRlci5fc3RhcnR1cCgpCisKKyAgICBhc3luYyBkZWYgX19hZXhpdF9fKHNlbGYsICpleGNfaW5mbzogb2JqZWN0KSAtPiBOb25lOgorICAgICAgICBhd2FpdCBzZWxmLl9yb3V0ZXIuX3NodXRkb3duKCkKKworICAgIGRlZiBfX2NhbGxfXyhzZWxmOiBfVCwgYXBwOiBvYmplY3QpIC0+IF9UOgorICAgICAgICByZXR1cm4gc2VsZgorCisKICMgQ2FjaGUgZm9yIGVuZHBvaW50IGNvbnRleHQgdG8gYXZvaWQgcmUtZXh0cmFjdGluZyBvbiBldmVyeSByZXF1ZXN0CiBfZW5kcG9pbnRfY29udGV4dF9jYWNoZTogZGljdFtpbnQsIEVuZHBvaW50Q29udGV4dF0gPSB7fQogCkBAIC05MDMsMTMgKzk4MCwzMyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICAgICApLAogICAgICAgICBdID0gRGVmYXVsdChnZW5lcmF0ZV91bmlxdWVfaWQpLAogICAgICkgLT4gTm9uZToKKyAgICAgICAgIyBIYW5kbGUgb25fc3RhcnR1cC9vbl9zaHV0ZG93biBsb2NhbGx5IHNpbmNlIFN0YXJsZXR0ZSByZW1vdmVkIHN1cHBvcnQKKyAgICAgICAgIyBSZWY6IGh0dHBzOi8vZ2l0aHViLmNvbS9LbHVkZXgvc3RhcmxldHRlL3B1bGwvMzExNworICAgICAgICAjIFRPRE86IGRlcHJlY2F0ZSB0aGlzIG9uY2UgdGhlIGxpZmVzcGFuIChvciBhbHRlcm5hdGl2ZSkgaW50ZXJmYWNlIGlzIGltcHJvdmVkCisgICAgICAgIHNlbGYub25fc3RhcnR1cDogbGlzdFtDYWxsYWJsZVtbXSwgQW55XV0gPSAoCisgICAgICAgICAgICBbXSBpZiBvbl9zdGFydHVwIGlzIE5vbmUgZWxzZSBsaXN0KG9uX3N0YXJ0dXApCisgICAgICAgICkKKyAgICAgICAgc2VsZi5vbl9zaHV0ZG93bjogbGlzdFtDYWxsYWJsZVtbXSwgQW55XV0gPSAoCisgICAgICAgICAgICBbXSBpZiBvbl9zaHV0ZG93biBpcyBOb25lIGVsc2UgbGlzdChvbl9zaHV0ZG93bikKKyAgICAgICAgKQorCisgICAgICAgICMgRGV0ZXJtaW5lIHRoZSBsaWZlc3BhbiBjb250ZXh0IHRvIHVzZQorICAgICAgICBpZiBsaWZlc3BhbiBpcyBOb25lOgorICAgICAgICAgICAgIyBVc2UgdGhlIGRlZmF1bHQgbGlmZXNwYW4gdGhhdCBydW5zIG9uX3N0YXJ0dXAvb25fc2h1dGRvd24gaGFuZGxlcnMKKyAgICAgICAgICAgIGxpZmVzcGFuX2NvbnRleHQ6IExpZmVzcGFuW0FueV0gPSBfRGVmYXVsdExpZmVzcGFuKHNlbGYpCisgICAgICAgIGVsaWYgaW5zcGVjdC5pc2FzeW5jZ2VuZnVuY3Rpb24obGlmZXNwYW4pOgorICAgICAgICAgICAgbGlmZXNwYW5fY29udGV4dCA9IGFzeW5jY29udGV4dG1hbmFnZXIobGlmZXNwYW4pCisgICAgICAgIGVsaWYgaW5zcGVjdC5pc2dlbmVyYXRvcmZ1bmN0aW9uKGxpZmVzcGFuKToKKyAgICAgICAgICAgIGxpZmVzcGFuX2NvbnRleHQgPSBfd3JhcF9nZW5fbGlmZXNwYW5fY29udGV4dChsaWZlc3BhbikKKyAgICAgICAgZWxzZToKKyAgICAgICAgICAgIGxpZmVzcGFuX2NvbnRleHQgPSBsaWZlc3BhbgorICAgICAgICBzZWxmLmxpZmVzcGFuX2NvbnRleHQgPSBsaWZlc3Bhbl9jb250ZXh0CisKICAgICAgICAgc3VwZXIoKS5fX2luaXRfXygKICAgICAgICAgICAgIHJvdXRlcz1yb3V0ZXMsCiAgICAgICAgICAgICByZWRpcmVjdF9zbGFzaGVzPXJlZGlyZWN0X3NsYXNoZXMsCiAgICAgICAgICAgICBkZWZhdWx0PWRlZmF1bHQsCi0gICAgICAgICAgICBvbl9zdGFydHVwPW9uX3N0YXJ0dXAsCi0gICAgICAgICAgICBvbl9zaHV0ZG93bj1vbl9zaHV0ZG93biwKLSAgICAgICAgICAgIGxpZmVzcGFuPWxpZmVzcGFuLAorICAgICAgICAgICAgbGlmZXNwYW49bGlmZXNwYW5fY29udGV4dCwKICAgICAgICAgKQogICAgICAgICBpZiBwcmVmaXg6CiAgICAgICAgICAgICBhc3NlcnQgcHJlZml4LnN0YXJ0c3dpdGgoIi8iKSwgIkEgcGF0aCBwcmVmaXggbXVzdCBzdGFydCB3aXRoICcvJyIKQEAgLTQ0NzMsNiArNDU3MCw1OCBAQCBkZWYgdHJhY2VfaXRlbShpdGVtX2lkOiBzdHIpOgogICAgICAgICAgICAgZ2VuZXJhdGVfdW5pcXVlX2lkX2Z1bmN0aW9uPWdlbmVyYXRlX3VuaXF1ZV9pZF9mdW5jdGlvbiwKICAgICAgICAgKQogCisgICAgIyBUT0RPOiByZW1vdmUgdGhpcyBvbmNlIHRoZSBsaWZlc3BhbiAob3IgYWx0ZXJuYXRpdmUpIGludGVyZmFjZSBpcyBpbXByb3ZlZAorICAgIGFzeW5jIGRlZiBfc3RhcnR1cChzZWxmKSAtPiBOb25lOgorICAgICAgICAiIiIKKyAgICAgICAgUnVuIGFueSBgLm9uX3N0YXJ0dXBgIGV2ZW50IGhhbmRsZXJzLgorCisgICAgICAgIFRoaXMgbWV0aG9kIGlzIGtlcHQgZm9yIGJhY2t3YXJkIGNvbXBhdGliaWxpdHkgYWZ0ZXIgU3RhcmxldHRlIHJlbW92ZWQKKyAgICAgICAgc3VwcG9ydCBmb3Igb25fc3RhcnR1cC9vbl9zaHV0ZG93biBoYW5kbGVycy4KKworICAgICAgICBSZWY6IGh0dHBzOi8vZ2l0aHViLmNvbS9LbHVkZXgvc3RhcmxldHRlL3B1bGwvMzExNworICAgICAgICAiIiIKKyAgICAgICAgZm9yIGhhbmRsZXIgaW4gc2VsZi5vbl9zdGFydHVwOgorICAgICAgICAgICAgaWYgaXNfYXN5bmNfY2FsbGFibGUoaGFuZGxlcik6CisgICAgICAgICAgICAgICAgYXdhaXQgaGFuZGxlcigpCisgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgIGhhbmRsZXIoKQorCisgICAgIyBUT0RPOiByZW1vdmUgdGhpcyBvbmNlIHRoZSBsaWZlc3BhbiAob3IgYWx0ZXJuYXRpdmUpIGludGVyZmFjZSBpcyBpbXByb3ZlZAorICAgIGFzeW5jIGRlZiBfc2h1dGRvd24oc2VsZikgLT4gTm9uZToKKyAgICAgICAgIiIiCisgICAgICAgIFJ1biBhbnkgYC5vbl9zaHV0ZG93bmAgZXZlbnQgaGFuZGxlcnMuCisKKyAgICAgICAgVGhpcyBtZXRob2QgaXMga2VwdCBmb3IgYmFja3dhcmQgY29tcGF0aWJpbGl0eSBhZnRlciBTdGFybGV0dGUgcmVtb3ZlZAorICAgICAgICBzdXBwb3J0IGZvciBvbl9zdGFydHVwL29uX3NodXRkb3duIGhhbmRsZXJzLgorCisgICAgICAgIFJlZjogaHR0cHM6Ly9naXRodWIuY29tL0tsdWRleC9zdGFybGV0dGUvcHVsbC8zMTE3CisgICAgICAgICIiIgorICAgICAgICBmb3IgaGFuZGxlciBpbiBzZWxmLm9uX3NodXRkb3duOgorICAgICAgICAgICAgaWYgaXNfYXN5bmNfY2FsbGFibGUoaGFuZGxlcik6CisgICAgICAgICAgICAgICAgYXdhaXQgaGFuZGxlcigpCisgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgIGhhbmRsZXIoKQorCisgICAgIyBUT0RPOiByZW1vdmUgdGhpcyBvbmNlIHRoZSBsaWZlc3BhbiAob3IgYWx0ZXJuYXRpdmUpIGludGVyZmFjZSBpcyBpbXByb3ZlZAorICAgIGRlZiBhZGRfZXZlbnRfaGFuZGxlcigKKyAgICAgICAgc2VsZiwKKyAgICAgICAgZXZlbnRfdHlwZTogc3RyLAorICAgICAgICBmdW5jOiBDYWxsYWJsZVtbXSwgQW55XSwKKyAgICApIC0+IE5vbmU6CisgICAgICAgICIiIgorICAgICAgICBBZGQgYW4gZXZlbnQgaGFuZGxlciBmdW5jdGlvbiBmb3Igc3RhcnR1cCBvciBzaHV0ZG93bi4KKworICAgICAgICBUaGlzIG1ldGhvZCBpcyBrZXB0IGZvciBiYWNrd2FyZCBjb21wYXRpYmlsaXR5IGFmdGVyIFN0YXJsZXR0ZSByZW1vdmVkCisgICAgICAgIHN1cHBvcnQgZm9yIG9uX3N0YXJ0dXAvb25fc2h1dGRvd24gaGFuZGxlcnMuCisKKyAgICAgICAgUmVmOiBodHRwczovL2dpdGh1Yi5jb20vS2x1ZGV4L3N0YXJsZXR0ZS9wdWxsLzMxMTcKKyAgICAgICAgIiIiCisgICAgICAgIGFzc2VydCBldmVudF90eXBlIGluICgic3RhcnR1cCIsICJzaHV0ZG93biIpCisgICAgICAgIGlmIGV2ZW50X3R5cGUgPT0gInN0YXJ0dXAiOgorICAgICAgICAgICAgc2VsZi5vbl9zdGFydHVwLmFwcGVuZChmdW5jKQorICAgICAgICBlbHNlOgorICAgICAgICAgICAgc2VsZi5vbl9zaHV0ZG93bi5hcHBlbmQoZnVuYykKKwogICAgIEBkZXByZWNhdGVkKAogICAgICAgICAiIiIKICAgICAgICAgb25fZXZlbnQgaXMgZGVwcmVjYXRlZCwgdXNlIGxpZmVzcGFuIGV2ZW50IGhhbmRsZXJzIGluc3RlYWQuCg==", "fastapi_14873": "LS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtOTUyLDE2ICs5NTIsNiBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICAgICApLAogICAgICAgICBdID0gRGVmYXVsdChnZW5lcmF0ZV91bmlxdWVfaWQpLAogICAgICkgLT4gTm9uZToKLSAgICAgICAgIyBIYW5kbGUgb25fc3RhcnR1cC9vbl9zaHV0ZG93biBsb2NhbGx5IHNpbmNlIFN0YXJsZXR0ZSByZW1vdmVkIHN1cHBvcnQKLSAgICAgICAgIyBSZWY6IGh0dHBzOi8vZ2l0aHViLmNvbS9LbHVkZXgvc3RhcmxldHRlL3B1bGwvMzExNwotICAgICAgICAjIFRPRE86IGRlcHJlY2F0ZSB0aGlzIG9uY2UgdGhlIGxpZmVzcGFuIChvciBhbHRlcm5hdGl2ZSkgaW50ZXJmYWNlIGlzIGltcHJvdmVkCi0gICAgICAgIHNlbGYub25fc3RhcnR1cDogbGlzdFtDYWxsYWJsZVtbXSwgQW55XV0gPSAoCi0gICAgICAgICAgICBbXSBpZiBvbl9zdGFydHVwIGlzIE5vbmUgZWxzZSBsaXN0KG9uX3N0YXJ0dXApCi0gICAgICAgICkKLSAgICAgICAgc2VsZi5vbl9zaHV0ZG93bjogbGlzdFtDYWxsYWJsZVtbXSwgQW55XV0gPSAoCi0gICAgICAgICAgICBbXSBpZiBvbl9zaHV0ZG93biBpcyBOb25lIGVsc2UgbGlzdChvbl9zaHV0ZG93bikKLSAgICAgICAgKQotCiAgICAgICAgICMgRGV0ZXJtaW5lIHRoZSBsaWZlc3BhbiBjb250ZXh0IHRvIHVzZQogICAgICAgICBpZiBsaWZlc3BhbiBpcyBOb25lOgogICAgICAgICAgICAgIyBVc2UgdGhlIGRlZmF1bHQgbGlmZXNwYW4gdGhhdCBydW5zIG9uX3N0YXJ0dXAvb25fc2h1dGRvd24gaGFuZGxlcnMKQEAgLTk4NSw2ICs5NzUsMTcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICAgICAgYXNzZXJ0IG5vdCBwcmVmaXguZW5kc3dpdGgoIi8iKSwgKAogICAgICAgICAgICAgICAgICJBIHBhdGggcHJlZml4IG11c3Qgbm90IGVuZCB3aXRoICcvJywgYXMgdGhlIHJvdXRlcyB3aWxsIHN0YXJ0IHdpdGggJy8nIgogICAgICAgICAgICAgKQorCisgICAgICAgICMgSGFuZGxlIG9uX3N0YXJ0dXAvb25fc2h1dGRvd24gbG9jYWxseSBzaW5jZSBTdGFybGV0dGUgcmVtb3ZlZCBzdXBwb3J0CisgICAgICAgICMgUmVmOiBodHRwczovL2dpdGh1Yi5jb20vS2x1ZGV4L3N0YXJsZXR0ZS9wdWxsLzMxMTcKKyAgICAgICAgIyBUT0RPOiBkZXByZWNhdGUgdGhpcyBvbmNlIHRoZSBsaWZlc3BhbiAob3IgYWx0ZXJuYXRpdmUpIGludGVyZmFjZSBpcyBpbXByb3ZlZAorICAgICAgICBzZWxmLm9uX3N0YXJ0dXA6IGxpc3RbQ2FsbGFibGVbW10sIEFueV1dID0gKAorICAgICAgICAgICAgW10gaWYgb25fc3RhcnR1cCBpcyBOb25lIGVsc2UgbGlzdChvbl9zdGFydHVwKQorICAgICAgICApCisgICAgICAgIHNlbGYub25fc2h1dGRvd246IGxpc3RbQ2FsbGFibGVbW10sIEFueV1dID0gKAorICAgICAgICAgICAgW10gaWYgb25fc2h1dGRvd24gaXMgTm9uZSBlbHNlIGxpc3Qob25fc2h1dGRvd24pCisgICAgICAgICkKKwogICAgICAgICBzZWxmLnByZWZpeCA9IHByZWZpeAogICAgICAgICBzZWxmLnRhZ3M6IGxpc3RbVW5pb25bc3RyLCBFbnVtXV0gPSB0YWdzIG9yIFtdCiAgICAgICAgIHNlbGYuZGVwZW5kZW5jaWVzID0gbGlzdChkZXBlbmRlbmNpZXMgb3IgW10pCg==", "fastapi_13920": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTY2LDYgKzY2LDcgQEAKIGZyb20gc3RhcmxldHRlLnJlc3BvbnNlcyBpbXBvcnQgUmVzcG9uc2UKIGZyb20gc3RhcmxldHRlLndlYnNvY2tldHMgaW1wb3J0IFdlYlNvY2tldAogZnJvbSB0eXBpbmdfZXh0ZW5zaW9ucyBpbXBvcnQgTGl0ZXJhbCwgZ2V0X2FyZ3MsIGdldF9vcmlnaW4KK2Zyb20gdHlwaW5nX2luc3BlY3Rpb24udHlwaW5nX29iamVjdHMgaW1wb3J0IGlzX3R5cGVhbGlhc3R5cGUKIAogbXVsdGlwYXJ0X25vdF9pbnN0YWxsZWRfZXJyb3IgPSAoCiAgICAgJ0Zvcm0gZGF0YSByZXF1aXJlcyAicHl0aG9uLW11bHRpcGFydCIgdG8gYmUgaW5zdGFsbGVkLiBcbicKQEAgLTM3MCw2ICszNzEsOSBAQCBkZWYgYW5hbHl6ZV9wYXJhbSgKICAgICBkZXBlbmRzID0gTm9uZQogICAgIHR5cGVfYW5ub3RhdGlvbjogQW55ID0gQW55CiAgICAgdXNlX2Fubm90YXRpb246IEFueSA9IEFueQorICAgIGlmIGlzX3R5cGVhbGlhc3R5cGUoYW5ub3RhdGlvbik6CisgICAgICAgICMgdW5wYWNrIGluIGNhc2UgUEVQIDY5NSB0eXBlIHN5bnRheCBpcyB1c2VkCisgICAgICAgIGFubm90YXRpb24gPSBhbm5vdGF0aW9uLl9fdmFsdWVfXwogICAgIGlmIGFubm90YXRpb24gaXMgbm90IGluc3BlY3QuU2lnbmF0dXJlLmVtcHR5OgogICAgICAgICB1c2VfYW5ub3RhdGlvbiA9IGFubm90YXRpb24KICAgICAgICAgdHlwZV9hbm5vdGF0aW9uID0gYW5ub3RhdGlvbgo=", "fastapi_14794": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTQ0OSw3ICs0NDksOSBAQCBkZWYgYW5hbHl6ZV9wYXJhbSgKICAgICAgICAgZGVwZW5kcyA9IGRhdGFjbGFzc2VzLnJlcGxhY2UoZGVwZW5kcywgZGVwZW5kZW5jeT10eXBlX2Fubm90YXRpb24pCiAKICAgICAjIEhhbmRsZSBub24tcGFyYW0gdHlwZSBhbm5vdGF0aW9ucyBsaWtlIFJlcXVlc3QKLSAgICBpZiBsZW5pZW50X2lzc3ViY2xhc3MoCisgICAgIyBPbmx5IGFwcGx5IHNwZWNpYWwgaGFuZGxpbmcgd2hlbiB0aGVyZSdzIG5vIGV4cGxpY2l0IERlcGVuZHMgLSBpZiB0aGVyZSdzIGEgRGVwZW5kcywKKyAgICAjIHRoZSBkZXBlbmRlbmN5IHdpbGwgYmUgY2FsbGVkIGFuZCBpdHMgcmV0dXJuIHZhbHVlIHVzZWQgaW5zdGVhZCBvZiB0aGUgc3BlY2lhbCBpbmplY3Rpb24KKyAgICBpZiBkZXBlbmRzIGlzIE5vbmUgYW5kIGxlbmllbnRfaXNzdWJjbGFzcygKICAgICAgICAgdHlwZV9hbm5vdGF0aW9uLAogICAgICAgICAoCiAgICAgICAgICAgICBSZXF1ZXN0LApAQCAtNDYwLDcgKzQ2Miw2IEBAIGRlZiBhbmFseXplX3BhcmFtKAogICAgICAgICAgICAgU2VjdXJpdHlTY29wZXMsCiAgICAgICAgICksCiAgICAgKToKLSAgICAgICAgYXNzZXJ0IGRlcGVuZHMgaXMgTm9uZSwgZiJDYW5ub3Qgc3BlY2lmeSBgRGVwZW5kc2AgZm9yIHR5cGUge3R5cGVfYW5ub3RhdGlvbiFyfSIKICAgICAgICAgYXNzZXJ0IGZpZWxkX2luZm8gaXMgTm9uZSwgKAogICAgICAgICAgICAgZiJDYW5ub3Qgc3BlY2lmeSBGYXN0QVBJIGFubm90YXRpb24gZm9yIHR5cGUge3R5cGVfYW5ub3RhdGlvbiFyfSIKICAgICAgICAgKQo=", "fastapi_14786": "LS0tIGEvZmFzdGFwaS9zZWN1cml0eS91dGlscy5weQorKysgYi9mYXN0YXBpL3NlY3VyaXR5L3V0aWxzLnB5CkBAIC03LDQgKzcsNCBAQCBkZWYgZ2V0X2F1dGhvcml6YXRpb25fc2NoZW1lX3BhcmFtKAogICAgIGlmIG5vdCBhdXRob3JpemF0aW9uX2hlYWRlcl92YWx1ZToKICAgICAgICAgcmV0dXJuICIiLCAiIgogICAgIHNjaGVtZSwgXywgcGFyYW0gPSBhdXRob3JpemF0aW9uX2hlYWRlcl92YWx1ZS5wYXJ0aXRpb24oIiAiKQotICAgIHJldHVybiBzY2hlbWUsIHBhcmFtCisgICAgcmV0dXJuIHNjaGVtZSwgcGFyYW0uc3RyaXAoKQo=", "fastapi_14492": "LS0tIGEvZG9jc19zcmMvbWV0YWRhdGEvdHV0b3JpYWwwMDFfMV9weTM5LnB5CisrKyBiL2RvY3Nfc3JjL21ldGFkYXRhL3R1dG9yaWFsMDAxXzFfcHkzOS5weQpAQCAtMjgsNyArMjgsNyBAQAogICAgIH0sCiAgICAgbGljZW5zZV9pbmZvPXsKICAgICAgICAgIm5hbWUiOiAiQXBhY2hlIDIuMCIsCi0gICAgICAgICJpZGVudGlmaWVyIjogIk1JVCIsCisgICAgICAgICJpZGVudGlmaWVyIjogIkFwYWNoZS0yLjAiLAogICAgIH0sCiApCiAK", "fastapi_14479": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTU1Miw3ICs1NTIsNyBAQCBkZWYgYW5hbHl6ZV9wYXJhbSgKICAgICAgICAgICAgICAgICAgICAgIyBGb3IgUHlkYW50aWMgdjEKICAgICAgICAgICAgICAgICAgICAgYW5kIGdldGF0dHIoZmllbGQsICJzaGFwZSIsIDEpID09IDEKICAgICAgICAgICAgICAgICApCi0gICAgICAgICAgICApCisgICAgICAgICAgICApLCBmIlF1ZXJ5IHBhcmFtZXRlciB7cGFyYW1fbmFtZSFyfSBtdXN0IGJlIG9uZSBvZiB0aGUgc3VwcG9ydGVkIHR5cGVzIgogCiAgICAgcmV0dXJuIFBhcmFtRGV0YWlscyh0eXBlX2Fubm90YXRpb249dHlwZV9hbm5vdGF0aW9uLCBkZXBlbmRzPWRlcGVuZHMsIGZpZWxkPWZpZWxkKQogCg==", "fastapi_14463": "LS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtMSwzICsxLDQgQEAKK2ltcG9ydCBjb3B5CiBpbXBvcnQgaHR0cC5jbGllbnQKIGltcG9ydCBpbnNwZWN0CiBpbXBvcnQgd2FybmluZ3MKQEAgLTM3Nyw3ICszNzgsNyBAQCBkZWYgZ2V0X29wZW5hcGlfcGF0aCgKICAgICAgICAgICAgICAgICAgICAgYWRkaXRpb25hbF9zdGF0dXNfY29kZSwKICAgICAgICAgICAgICAgICAgICAgYWRkaXRpb25hbF9yZXNwb25zZSwKICAgICAgICAgICAgICAgICApIGluIHJvdXRlLnJlc3BvbnNlcy5pdGVtcygpOgotICAgICAgICAgICAgICAgICAgICBwcm9jZXNzX3Jlc3BvbnNlID0gYWRkaXRpb25hbF9yZXNwb25zZS5jb3B5KCkKKyAgICAgICAgICAgICAgICAgICAgcHJvY2Vzc19yZXNwb25zZSA9IGNvcHkuZGVlcGNvcHkoYWRkaXRpb25hbF9yZXNwb25zZSkKICAgICAgICAgICAgICAgICAgICAgcHJvY2Vzc19yZXNwb25zZS5wb3AoIm1vZGVsIiwgTm9uZSkKICAgICAgICAgICAgICAgICAgICAgc3RhdHVzX2NvZGVfa2V5ID0gc3RyKGFkZGl0aW9uYWxfc3RhdHVzX2NvZGUpLnVwcGVyKCkKICAgICAgICAgICAgICAgICAgICAgaWYgc3RhdHVzX2NvZGVfa2V5ID09ICJERUZBVUxUIjoK", "fastapi_12942": "LS0tIGEvZmFzdGFwaS9fY29tcGF0LnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC5weQpAQCAtNTkwLDYgKzU5MCw5IEBAIGRlZiBmaWVsZF9hbm5vdGF0aW9uX2lzX2NvbXBsZXgoYW5ub3RhdGlvbjogVW5pb25bVHlwZVtBbnldLCBOb25lXSkgLT4gYm9vbDoKICAgICBpZiBvcmlnaW4gaXMgVW5pb24gb3Igb3JpZ2luIGlzIFVuaW9uVHlwZToKICAgICAgICAgcmV0dXJuIGFueShmaWVsZF9hbm5vdGF0aW9uX2lzX2NvbXBsZXgoYXJnKSBmb3IgYXJnIGluIGdldF9hcmdzKGFubm90YXRpb24pKQogCisgICAgaWYgb3JpZ2luIGlzIEFubm90YXRlZDoKKyAgICAgICAgcmV0dXJuIGZpZWxkX2Fubm90YXRpb25faXNfY29tcGxleChnZXRfYXJncyhhbm5vdGF0aW9uKVswXSkKKwogICAgIHJldHVybiAoCiAgICAgICAgIF9hbm5vdGF0aW9uX2lzX2NvbXBsZXgoYW5ub3RhdGlvbikKICAgICAgICAgb3IgX2Fubm90YXRpb25faXNfY29tcGxleChvcmlnaW4pCg==", "fastapi_14099": "...", "fastapi_14609": "...", "fastapi_14372": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTEsMyArMSw0IEBACitpbXBvcnQgZGF0YWNsYXNzZXMKIGltcG9ydCBpbnNwZWN0CiBmcm9tIGNvbnRleHRsaWIgaW1wb3J0IEFzeW5jRXhpdFN0YWNrLCBjb250ZXh0bWFuYWdlcgogZnJvbSBjb3B5IGltcG9ydCBjb3B5LCBkZWVwY29weQpAQCAtNDI4LDcgKzQyOSw3IEBAIGRlZiBhbmFseXplX3BhcmFtKAogICAgIGlmIGRlcGVuZHMgaXMgbm90IE5vbmUgYW5kIGRlcGVuZHMuZGVwZW5kZW5jeSBpcyBOb25lOgogICAgICAgICAjIENvcHkgYGRlcGVuZHNgIGJlZm9yZSBtdXRhdGluZyBpdAogICAgICAgICBkZXBlbmRzID0gY29weShkZXBlbmRzKQotICAgICAgICBkZXBlbmRzLmRlcGVuZGVuY3kgPSB0eXBlX2Fubm90YXRpb24KKyAgICAgICAgZGVwZW5kcyA9IGRhdGFjbGFzc2VzLnJlcGxhY2UoZGVwZW5kcywgZGVwZW5kZW5jeT10eXBlX2Fubm90YXRpb24pCiAKICAgICAjIEhhbmRsZSBub24tcGFyYW0gdHlwZSBhbm5vdGF0aW9ucyBsaWtlIFJlcXVlc3QKICAgICBpZiBsZW5pZW50X2lzc3ViY2xhc3MoCi0tLSBhL2Zhc3RhcGkvcGFyYW1zLnB5CisrKyBiL2Zhc3RhcGkvcGFyYW1zLnB5CkBAIC03NjIsMTMgKzc2MiwxMyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICkKIAogCi1AZGF0YWNsYXNzCitAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQogY2xhc3MgRGVwZW5kczoKICAgICBkZXBlbmRlbmN5OiBPcHRpb25hbFtDYWxsYWJsZVsuLi4sIEFueV1dID0gTm9uZQogICAgIHVzZV9jYWNoZTogYm9vbCA9IFRydWUKICAgICBzY29wZTogVW5pb25bTGl0ZXJhbFsiZnVuY3Rpb24iLCAicmVxdWVzdCJdLCBOb25lXSA9IE5vbmUKIAogCi1AZGF0YWNsYXNzCitAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQogY2xhc3MgU2VjdXJpdHkoRGVwZW5kcyk6CiAgICAgc2NvcGVzOiBPcHRpb25hbFtTZXF1ZW5jZVtzdHJdXSA9IE5vbmUK", "fastapi_14349": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMjYyLDEyICsyNjIsMTIgQEAgZGVmIF9yZXBsYWNlX3JlZnMoCiAgICAgbmV3X3NjaGVtYSA9IGRlZXBjb3B5KHNjaGVtYSkKICAgICBmb3Iga2V5LCB2YWx1ZSBpbiBuZXdfc2NoZW1hLml0ZW1zKCk6CiAgICAgICAgIGlmIGtleSA9PSAiJHJlZiI6Ci0gICAgICAgICAgICByZWZfbmFtZSA9IHNjaGVtYVsiJHJlZiJdLnNwbGl0KCIvIilbLTFdCi0gICAgICAgICAgICBpZiByZWZfbmFtZSBpbiBvbGRfbmFtZV90b19uZXdfbmFtZV9tYXA6Ci0gICAgICAgICAgICAgICAgbmV3X25hbWUgPSBvbGRfbmFtZV90b19uZXdfbmFtZV9tYXBbcmVmX25hbWVdCi0gICAgICAgICAgICAgICAgbmV3X3NjaGVtYVsiJHJlZiJdID0gUkVGX1RFTVBMQVRFLmZvcm1hdChtb2RlbD1uZXdfbmFtZSkKLSAgICAgICAgICAgIGVsc2U6Ci0gICAgICAgICAgICAgICAgbmV3X3NjaGVtYVsiJHJlZiJdID0gc2NoZW1hWyIkcmVmIl0KKyAgICAgICAgICAgIHZhbHVlID0gc2NoZW1hWyIkcmVmIl0KKyAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CisgICAgICAgICAgICAgICAgcmVmX25hbWUgPSBzY2hlbWFbIiRyZWYiXS5zcGxpdCgiLyIpWy0xXQorICAgICAgICAgICAgICAgIGlmIHJlZl9uYW1lIGluIG9sZF9uYW1lX3RvX25ld19uYW1lX21hcDoKKyAgICAgICAgICAgICAgICAgICAgbmV3X25hbWUgPSBvbGRfbmFtZV90b19uZXdfbmFtZV9tYXBbcmVmX25hbWVdCisgICAgICAgICAgICAgICAgICAgIG5ld19zY2hlbWFbIiRyZWYiXSA9IFJFRl9URU1QTEFURS5mb3JtYXQobW9kZWw9bmV3X25hbWUpCiAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KToKICAgICAgICAgICAgIG5ld19zY2hlbWFba2V5XSA9IF9yZXBsYWNlX3JlZnMoCg==", "fastapi_14512": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMTgsNyArMTgsNyBAQAogZnJvbSBmYXN0YXBpLl9jb21wYXQgaW1wb3J0IG1heV92MSwgc2hhcmVkCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5jb25zdGFudHMgaW1wb3J0IFJFRl9URU1QTEFURQogZnJvbSBmYXN0YXBpLnR5cGVzIGltcG9ydCBJbmNFeCwgTW9kZWxOYW1lTWFwLCBVbmlvblR5cGUKLWZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgQ29uZmlnRGljdCwgVHlwZUFkYXB0ZXIsIGNyZWF0ZV9tb2RlbAorZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBDb25maWdEaWN0LCBGaWVsZCwgVHlwZUFkYXB0ZXIsIGNyZWF0ZV9tb2RlbAogZnJvbSBweWRhbnRpYyBpbXBvcnQgUHlkYW50aWNTY2hlbWFHZW5lcmF0aW9uRXJyb3IgYXMgUHlkYW50aWNTY2hlbWFHZW5lcmF0aW9uRXJyb3IKIGZyb20gcHlkYW50aWMgaW1wb3J0IFB5ZGFudGljVW5kZWZpbmVkQW5ub3RhdGlvbiBhcyBQeWRhbnRpY1VuZGVmaW5lZEFubm90YXRpb24KIGZyb20gcHlkYW50aWMgaW1wb3J0IFZhbGlkYXRpb25FcnJvciBhcyBWYWxpZGF0aW9uRXJyb3IKQEAgLTUwLDYgKzUwLDQ1IEBACiBldmFsdWF0ZV9mb3J3YXJkcmVmID0gZXZhbF90eXBlX2xlbmllbnQKIFZhbGlkYXRvciA9IEFueQogCisjIFRPRE86IHJlbW92ZSB3aGVuIGRyb3BwaW5nIHN1cHBvcnQgZm9yIFB5ZGFudGljIDwgdjIuMTIuMworX0F0dHJzID0geworICAgICJkZWZhdWx0IjogLi4uLAorICAgICJkZWZhdWx0X2ZhY3RvcnkiOiBOb25lLAorICAgICJhbGlhcyI6IE5vbmUsCisgICAgImFsaWFzX3ByaW9yaXR5IjogTm9uZSwKKyAgICAidmFsaWRhdGlvbl9hbGlhcyI6IE5vbmUsCisgICAgInNlcmlhbGl6YXRpb25fYWxpYXMiOiBOb25lLAorICAgICJ0aXRsZSI6IE5vbmUsCisgICAgImZpZWxkX3RpdGxlX2dlbmVyYXRvciI6IE5vbmUsCisgICAgImRlc2NyaXB0aW9uIjogTm9uZSwKKyAgICAiZXhhbXBsZXMiOiBOb25lLAorICAgICJleGNsdWRlIjogTm9uZSwKKyAgICAiZXhjbHVkZV9pZiI6IE5vbmUsCisgICAgImRpc2NyaW1pbmF0b3IiOiBOb25lLAorICAgICJkZXByZWNhdGVkIjogTm9uZSwKKyAgICAianNvbl9zY2hlbWFfZXh0cmEiOiBOb25lLAorICAgICJmcm96ZW4iOiBOb25lLAorICAgICJ2YWxpZGF0ZV9kZWZhdWx0IjogTm9uZSwKKyAgICAicmVwciI6IFRydWUsCisgICAgImluaXQiOiBOb25lLAorICAgICJpbml0X3ZhciI6IE5vbmUsCisgICAgImt3X29ubHkiOiBOb25lLAorfQorCisKKyMgVE9ETzogcmVtb3ZlIHdoZW4gZHJvcHBpbmcgc3VwcG9ydCBmb3IgUHlkYW50aWMgPCB2Mi4xMi4zCitkZWYgYXNkaWN0KGZpZWxkX2luZm86IEZpZWxkSW5mbykgLT4gRGljdFtzdHIsIEFueV06CisgICAgYXR0cmlidXRlcyA9IHt9CisgICAgZm9yIGF0dHIgaW4gX0F0dHJzOgorICAgICAgICB2YWx1ZSA9IGdldGF0dHIoZmllbGRfaW5mbywgYXR0ciwgVW5kZWZpbmVkKQorICAgICAgICBpZiB2YWx1ZSBpcyBub3QgVW5kZWZpbmVkOgorICAgICAgICAgICAgYXR0cmlidXRlc1thdHRyXSA9IHZhbHVlCisgICAgcmV0dXJuIHsKKyAgICAgICAgImFubm90YXRpb24iOiBmaWVsZF9pbmZvLmFubm90YXRpb24sCisgICAgICAgICJtZXRhZGF0YSI6IGZpZWxkX2luZm8ubWV0YWRhdGEsCisgICAgICAgICJhdHRyaWJ1dGVzIjogYXR0cmlidXRlcywKKyAgICB9CisKIAogY2xhc3MgQmFzZUNvbmZpZzoKICAgICBwYXNzCkBAIC05NSwxMCArMTM0LDE1IEBAIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgICAgICAgICAgd2FybmluZ3Muc2ltcGxlZmlsdGVyKAogICAgICAgICAgICAgICAgICAgICAiaWdub3JlIiwgY2F0ZWdvcnk9VW5zdXBwb3J0ZWRGaWVsZEF0dHJpYnV0ZVdhcm5pbmcKICAgICAgICAgICAgICAgICApCisgICAgICAgICAgICAjIFRPRE86IHJlbW92ZSBhZnRlciBkcm9wcGluZyBzdXBwb3J0IGZvciBQeXRob24gMy44IGFuZAorICAgICAgICAgICAgIyBzZXR0aW5nIHRoZSBtaW4gUHlkYW50aWMgdG8gdjIuMTIuMyB0aGF0IGFkZHMgYXNkaWN0KCkKKyAgICAgICAgICAgIGZpZWxkX2RpY3QgPSBhc2RpY3Qoc2VsZi5maWVsZF9pbmZvKQogICAgICAgICAgICAgYW5ub3RhdGVkX2FyZ3MgPSAoCi0gICAgICAgICAgICAgICAgc2VsZi5maWVsZF9pbmZvLmFubm90YXRpb24sCi0gICAgICAgICAgICAgICAgKnNlbGYuZmllbGRfaW5mby5tZXRhZGF0YSwKLSAgICAgICAgICAgICAgICBzZWxmLmZpZWxkX2luZm8sCisgICAgICAgICAgICAgICAgZmllbGRfZGljdFsiYW5ub3RhdGlvbiJdLAorICAgICAgICAgICAgICAgICpmaWVsZF9kaWN0WyJtZXRhZGF0YSJdLAorICAgICAgICAgICAgICAgICMgdGhpcyBGaWVsZEluZm8gbmVlZHMgdG8gYmUgY3JlYXRlZCBhZ2FpbiBzbyB0aGF0IGl0IGRvZXNuJ3QgaW5jbHVkZQorICAgICAgICAgICAgICAgICMgdGhlIG9sZCBmaWVsZCBpbmZvIG1ldGFkYXRhIGFuZCBvbmx5IHRoZSByZXN0IG9mIHRoZSBhdHRyaWJ1dGVzCisgICAgICAgICAgICAgICAgRmllbGQoKipmaWVsZF9kaWN0WyJhdHRyaWJ1dGVzIl0pLAogICAgICAgICAgICAgKQogICAgICAgICAgICAgc2VsZi5fdHlwZV9hZGFwdGVyOiBUeXBlQWRhcHRlcltBbnldID0gVHlwZUFkYXB0ZXIoCiAgICAgICAgICAgICAgICAgQW5ub3RhdGVkW2Fubm90YXRlZF9hcmdzXSwK", "fastapi_14487": "LS0tIGEvZG9jc19zcmMvaGFuZGxpbmdfZXJyb3JzL3R1dG9yaWFsMDA0LnB5CisrKyBiL2RvY3Nfc3JjL2hhbmRsaW5nX2Vycm9ycy90dXRvcmlhbDAwNC5weQpAQCAtMTIsOCArMTIsMTEgQEAgYXN5bmMgZGVmIGh0dHBfZXhjZXB0aW9uX2hhbmRsZXIocmVxdWVzdCwgZXhjKToKIAogCiBAYXBwLmV4Y2VwdGlvbl9oYW5kbGVyKFJlcXVlc3RWYWxpZGF0aW9uRXJyb3IpCi1hc3luYyBkZWYgdmFsaWRhdGlvbl9leGNlcHRpb25faGFuZGxlcihyZXF1ZXN0LCBleGMpOgotICAgIHJldHVybiBQbGFpblRleHRSZXNwb25zZShzdHIoZXhjKSwgc3RhdHVzX2NvZGU9NDAwKQorYXN5bmMgZGVmIHZhbGlkYXRpb25fZXhjZXB0aW9uX2hhbmRsZXIocmVxdWVzdCwgZXhjOiBSZXF1ZXN0VmFsaWRhdGlvbkVycm9yKToKKyAgICBtZXNzYWdlID0gIlZhbGlkYXRpb24gZXJyb3JzOiIKKyAgICBmb3IgZXJyb3IgaW4gZXhjLmVycm9ycygpOgorICAgICAgICBtZXNzYWdlICs9IGYiXG5GaWVsZDoge2Vycm9yWydsb2MnXX0sIEVycm9yOiB7ZXJyb3JbJ21zZyddfSIKKyAgICByZXR1cm4gUGxhaW5UZXh0UmVzcG9uc2UobWVzc2FnZSwgc3RhdHVzX2NvZGU9NDAwKQogCiAKIEBhcHAuZ2V0KCIvaXRlbXMve2l0ZW1faWR9IikK", "fastapi_11355": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTEsNSArMSw2IEBACiBpbXBvcnQgZGF0YWNsYXNzZXMKIGltcG9ydCBpbnNwZWN0CitpbXBvcnQgc3lzCiBmcm9tIGNvbnRleHRsaWIgaW1wb3J0IEFzeW5jRXhpdFN0YWNrLCBjb250ZXh0bWFuYWdlcgogZnJvbSBjb3B5IGltcG9ydCBjb3B5LCBkZWVwY29weQogZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCkBAIC0xOTEsNyArMTkyLDEwIEBAIGRlZiBnZXRfZmxhdF9wYXJhbXMoZGVwZW5kYW50OiBEZXBlbmRhbnQpIC0+IExpc3RbTW9kZWxGaWVsZF06CiAKIAogZGVmIGdldF90eXBlZF9zaWduYXR1cmUoY2FsbDogQ2FsbGFibGVbLi4uLCBBbnldKSAtPiBpbnNwZWN0LlNpZ25hdHVyZToKLSAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsKQorICAgIGlmIHN5cy52ZXJzaW9uX2luZm8gPj0gKDMsIDEwKToKKyAgICAgICAgc2lnbmF0dXJlID0gaW5zcGVjdC5zaWduYXR1cmUoY2FsbCwgZXZhbF9zdHI9VHJ1ZSkKKyAgICBlbHNlOgorICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsKQogICAgIHVud3JhcHBlZCA9IGluc3BlY3QudW53cmFwKGNhbGwpCiAgICAgZ2xvYmFsbnMgPSBnZXRhdHRyKHVud3JhcHBlZCwgIl9fZ2xvYmFsc19fIiwge30pCiAgICAgdHlwZWRfcGFyYW1zID0gWwpAQCAtMjE3LDcgKzIyMSwxMCBAQCBkZWYgZ2V0X3R5cGVkX2Fubm90YXRpb24oYW5ub3RhdGlvbjogQW55LCBnbG9iYWxuczogRGljdFtzdHIsIEFueV0pIC0+IEFueToKIAogCiBkZWYgZ2V0X3R5cGVkX3JldHVybl9hbm5vdGF0aW9uKGNhbGw6IENhbGxhYmxlWy4uLiwgQW55XSkgLT4gQW55OgotICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKGNhbGwpCisgICAgaWYgc3lzLnZlcnNpb25faW5mbyA+PSAoMywgMTApOgorICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsLCBldmFsX3N0cj1UcnVlKQorICAgIGVsc2U6CisgICAgICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKGNhbGwpCiAgICAgdW53cmFwcGVkID0gaW5zcGVjdC51bndyYXAoY2FsbCkKICAgICBhbm5vdGF0aW9uID0gc2lnbmF0dXJlLnJldHVybl9hbm5vdGF0aW9uCiAK", "fastapi_14485": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTIwOSwxMSArMjA5LDIxIEBAIGRlZiBnZXRfZmxhdF9wYXJhbXMoZGVwZW5kYW50OiBEZXBlbmRhbnQpIC0+IExpc3RbTW9kZWxGaWVsZF06CiAgICAgcmV0dXJuIHBhdGhfcGFyYW1zICsgcXVlcnlfcGFyYW1zICsgaGVhZGVyX3BhcmFtcyArIGNvb2tpZV9wYXJhbXMKIAogCi1kZWYgZ2V0X3R5cGVkX3NpZ25hdHVyZShjYWxsOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IGluc3BlY3QuU2lnbmF0dXJlOgorZGVmIF9nZXRfc2lnbmF0dXJlKGNhbGw6IENhbGxhYmxlWy4uLiwgQW55XSkgLT4gaW5zcGVjdC5TaWduYXR1cmU6CiAgICAgaWYgc3lzLnZlcnNpb25faW5mbyA+PSAoMywgMTApOgotICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsLCBldmFsX3N0cj1UcnVlKQorICAgICAgICB0cnk6CisgICAgICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsLCBldmFsX3N0cj1UcnVlKQorICAgICAgICBleGNlcHQgTmFtZUVycm9yOgorICAgICAgICAgICAgIyBIYW5kbGUgdHlwZSBhbm5vdGF0aW9ucyB3aXRoIGlmIFRZUEVfQ0hFQ0tJTkcsIG5vdCB1c2VkIGJ5IEZhc3RBUEkKKyAgICAgICAgICAgICMgZS5nLiBkZXBlbmRlbmN5IHJldHVybiB0eXBlcworICAgICAgICAgICAgc2lnbmF0dXJlID0gaW5zcGVjdC5zaWduYXR1cmUoY2FsbCkKICAgICBlbHNlOgogICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsKQorICAgIHJldHVybiBzaWduYXR1cmUKKworCitkZWYgZ2V0X3R5cGVkX3NpZ25hdHVyZShjYWxsOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IGluc3BlY3QuU2lnbmF0dXJlOgorICAgIHNpZ25hdHVyZSA9IF9nZXRfc2lnbmF0dXJlKGNhbGwpCiAgICAgdW53cmFwcGVkID0gaW5zcGVjdC51bndyYXAoY2FsbCkKICAgICBnbG9iYWxucyA9IGdldGF0dHIodW53cmFwcGVkLCAiX19nbG9iYWxzX18iLCB7fSkKICAgICB0eXBlZF9wYXJhbXMgPSBbCkBAIC0yMzksMTAgKzI0OSw3IEBAIGRlZiBnZXRfdHlwZWRfYW5ub3RhdGlvbihhbm5vdGF0aW9uOiBBbnksIGdsb2JhbG5zOiBEaWN0W3N0ciwgQW55XSkgLT4gQW55OgogCiAKIGRlZiBnZXRfdHlwZWRfcmV0dXJuX2Fubm90YXRpb24oY2FsbDogQ2FsbGFibGVbLi4uLCBBbnldKSAtPiBBbnk6Ci0gICAgaWYgc3lzLnZlcnNpb25faW5mbyA+PSAoMywgMTApOgotICAgICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsLCBldmFsX3N0cj1UcnVlKQotICAgIGVsc2U6Ci0gICAgICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKGNhbGwpCisgICAgc2lnbmF0dXJlID0gX2dldF9zaWduYXR1cmUoY2FsbCkKICAgICB1bndyYXBwZWQgPSBpbnNwZWN0LnVud3JhcChjYWxsKQogICAgIGFubm90YXRpb24gPSBzaWduYXR1cmUucmV0dXJuX2Fubm90YXRpb24KIAo=", "fastapi_14482": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMSw3ICsxLDcgQEAKIGltcG9ydCByZQogaW1wb3J0IHdhcm5pbmdzCiBmcm9tIGNvcHkgaW1wb3J0IGNvcHksIGRlZXBjb3B5Ci1mcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKK2Zyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgaXNfZGF0YWNsYXNzCiBmcm9tIGVudW0gaW1wb3J0IEVudW0KIGZyb20gdHlwaW5nIGltcG9ydCAoCiAgICAgQW55LApAQCAtMTgsNyArMTgsNyBAQAogZnJvbSBmYXN0YXBpLl9jb21wYXQgaW1wb3J0IG1heV92MSwgc2hhcmVkCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5jb25zdGFudHMgaW1wb3J0IFJFRl9URU1QTEFURQogZnJvbSBmYXN0YXBpLnR5cGVzIGltcG9ydCBJbmNFeCwgTW9kZWxOYW1lTWFwLCBVbmlvblR5cGUKLWZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgVHlwZUFkYXB0ZXIsIGNyZWF0ZV9tb2RlbAorZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBDb25maWdEaWN0LCBUeXBlQWRhcHRlciwgY3JlYXRlX21vZGVsCiBmcm9tIHB5ZGFudGljIGltcG9ydCBQeWRhbnRpY1NjaGVtYUdlbmVyYXRpb25FcnJvciBhcyBQeWRhbnRpY1NjaGVtYUdlbmVyYXRpb25FcnJvcgogZnJvbSBweWRhbnRpYyBpbXBvcnQgUHlkYW50aWNVbmRlZmluZWRBbm5vdGF0aW9uIGFzIFB5ZGFudGljVW5kZWZpbmVkQW5ub3RhdGlvbgogZnJvbSBweWRhbnRpYyBpbXBvcnQgVmFsaWRhdGlvbkVycm9yIGFzIFZhbGlkYXRpb25FcnJvcgpAQCAtNjQsNiArNjQsNyBAQCBjbGFzcyBNb2RlbEZpZWxkOgogICAgIGZpZWxkX2luZm86IEZpZWxkSW5mbwogICAgIG5hbWU6IHN0cgogICAgIG1vZGU6IExpdGVyYWxbInZhbGlkYXRpb24iLCAic2VyaWFsaXphdGlvbiJdID0gInZhbGlkYXRpb24iCisgICAgY29uZmlnOiBVbmlvbltDb25maWdEaWN0LCBOb25lXSA9IE5vbmUKIAogICAgIEBwcm9wZXJ0eQogICAgIGRlZiBhbGlhcyhzZWxmKSAtPiBzdHI6CkBAIC05NCw4ICs5NSwxNCBAQCBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgICAgICAgICAgIHdhcm5pbmdzLnNpbXBsZWZpbHRlcigKICAgICAgICAgICAgICAgICAgICAgImlnbm9yZSIsIGNhdGVnb3J5PVVuc3VwcG9ydGVkRmllbGRBdHRyaWJ1dGVXYXJuaW5nCiAgICAgICAgICAgICAgICAgKQorICAgICAgICAgICAgYW5ub3RhdGVkX2FyZ3MgPSAoCisgICAgICAgICAgICAgICAgc2VsZi5maWVsZF9pbmZvLmFubm90YXRpb24sCisgICAgICAgICAgICAgICAgKnNlbGYuZmllbGRfaW5mby5tZXRhZGF0YSwKKyAgICAgICAgICAgICAgICBzZWxmLmZpZWxkX2luZm8sCisgICAgICAgICAgICApCiAgICAgICAgICAgICBzZWxmLl90eXBlX2FkYXB0ZXI6IFR5cGVBZGFwdGVyW0FueV0gPSBUeXBlQWRhcHRlcigKLSAgICAgICAgICAgICAgICBBbm5vdGF0ZWRbc2VsZi5maWVsZF9pbmZvLmFubm90YXRpb24sIHNlbGYuZmllbGRfaW5mb10KKyAgICAgICAgICAgICAgICBBbm5vdGF0ZWRbYW5ub3RhdGVkX2FyZ3NdLAorICAgICAgICAgICAgICAgIGNvbmZpZz1zZWxmLmNvbmZpZywKICAgICAgICAgICAgICkKIAogICAgIGRlZiBnZXRfZGVmYXVsdChzZWxmKSAtPiBBbnk6CkBAIC00MTIsMTAgKzQxOSwyMSBAQCBkZWYgY3JlYXRlX2JvZHlfbW9kZWwoCiAKIAogZGVmIGdldF9tb2RlbF9maWVsZHMobW9kZWw6IFR5cGVbQmFzZU1vZGVsXSkgLT4gTGlzdFtNb2RlbEZpZWxkXToKLSAgICByZXR1cm4gWwotICAgICAgICBNb2RlbEZpZWxkKGZpZWxkX2luZm89ZmllbGRfaW5mbywgbmFtZT1uYW1lKQotICAgICAgICBmb3IgbmFtZSwgZmllbGRfaW5mbyBpbiBtb2RlbC5tb2RlbF9maWVsZHMuaXRlbXMoKQotICAgIF0KKyAgICBtb2RlbF9maWVsZHM6IExpc3RbTW9kZWxGaWVsZF0gPSBbXQorICAgIGZvciBuYW1lLCBmaWVsZF9pbmZvIGluIG1vZGVsLm1vZGVsX2ZpZWxkcy5pdGVtcygpOgorICAgICAgICB0eXBlXyA9IGZpZWxkX2luZm8uYW5ub3RhdGlvbgorICAgICAgICBpZiBsZW5pZW50X2lzc3ViY2xhc3ModHlwZV8sIChCYXNlTW9kZWwsIGRpY3QpKSBvciBpc19kYXRhY2xhc3ModHlwZV8pOgorICAgICAgICAgICAgbW9kZWxfY29uZmlnID0gTm9uZQorICAgICAgICBlbHNlOgorICAgICAgICAgICAgbW9kZWxfY29uZmlnID0gbW9kZWwubW9kZWxfY29uZmlnCisgICAgICAgIG1vZGVsX2ZpZWxkcy5hcHBlbmQoCisgICAgICAgICAgICBNb2RlbEZpZWxkKAorICAgICAgICAgICAgICAgIGZpZWxkX2luZm89ZmllbGRfaW5mbywKKyAgICAgICAgICAgICAgICBuYW1lPW5hbWUsCisgICAgICAgICAgICAgICAgY29uZmlnPW1vZGVsX2NvbmZpZywKKyAgICAgICAgICAgICkKKyAgICAgICAgKQorICAgIHJldHVybiBtb2RlbF9maWVsZHMKIAogCiAjIER1cGxpY2F0ZSBvZiBzZXZlcmFsIHNjaGVtYSBmdW5jdGlvbnMgZnJvbSBQeWRhbnRpYyB2MSB0byBtYWtlIHRoZW0gY29tcGF0aWJsZSB3aXRoCg==", "fastapi_14306": "LS0tIGEvZmFzdGFwaS9leGNlcHRpb25zLnB5CisrKyBiL2Zhc3RhcGkvZXhjZXB0aW9ucy5weQpAQCAtMSw0ICsxLDQgQEAKLWZyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QsIE9wdGlvbmFsLCBTZXF1ZW5jZSwgVHlwZSwgVW5pb24KK2Zyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QsIE9wdGlvbmFsLCBTZXF1ZW5jZSwgVHlwZSwgVHlwZWREaWN0LCBVbmlvbgogCiBmcm9tIGFubm90YXRlZF9kb2MgaW1wb3J0IERvYwogZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBjcmVhdGVfbW9kZWwKQEAgLTcsNiArNywxMyBAQAogZnJvbSB0eXBpbmdfZXh0ZW5zaW9ucyBpbXBvcnQgQW5ub3RhdGVkCiAKIAorY2xhc3MgRW5kcG9pbnRDb250ZXh0KFR5cGVkRGljdCwgdG90YWw9RmFsc2UpOgorICAgIGZ1bmN0aW9uOiBzdHIKKyAgICBwYXRoOiBzdHIKKyAgICBmaWxlOiBzdHIKKyAgICBsaW5lOiBpbnQKKworCiBjbGFzcyBIVFRQRXhjZXB0aW9uKFN0YXJsZXR0ZUhUVFBFeGNlcHRpb24pOgogICAgICIiIgogICAgIEFuIEhUVFAgZXhjZXB0aW9uIHlvdSBjYW4gcmFpc2UgaW4geW91ciBvd24gY29kZSB0byBzaG93IGVycm9ycyB0byB0aGUgY2xpZW50LgpAQCAtMTU1LDMwICsxNjIsNzIgQEAgY2xhc3MgRGVwZW5kZW5jeVNjb3BlRXJyb3IoRmFzdEFQSUVycm9yKToKIAogCiBjbGFzcyBWYWxpZGF0aW9uRXhjZXB0aW9uKEV4Y2VwdGlvbik6Ci0gICAgZGVmIF9faW5pdF9fKHNlbGYsIGVycm9yczogU2VxdWVuY2VbQW55XSkgLT4gTm9uZToKKyAgICBkZWYgX19pbml0X18oCisgICAgICAgIHNlbGYsCisgICAgICAgIGVycm9yczogU2VxdWVuY2VbQW55XSwKKyAgICAgICAgKiwKKyAgICAgICAgZW5kcG9pbnRfY3R4OiBPcHRpb25hbFtFbmRwb2ludENvbnRleHRdID0gTm9uZSwKKyAgICApIC0+IE5vbmU6CiAgICAgICAgIHNlbGYuX2Vycm9ycyA9IGVycm9ycworICAgICAgICBzZWxmLmVuZHBvaW50X2N0eCA9IGVuZHBvaW50X2N0eAorCisgICAgICAgIGN0eCA9IGVuZHBvaW50X2N0eCBvciB7fQorICAgICAgICBzZWxmLmVuZHBvaW50X2Z1bmN0aW9uID0gY3R4LmdldCgiZnVuY3Rpb24iKQorICAgICAgICBzZWxmLmVuZHBvaW50X3BhdGggPSBjdHguZ2V0KCJwYXRoIikKKyAgICAgICAgc2VsZi5lbmRwb2ludF9maWxlID0gY3R4LmdldCgiZmlsZSIpCisgICAgICAgIHNlbGYuZW5kcG9pbnRfbGluZSA9IGN0eC5nZXQoImxpbmUiKQogCiAgICAgZGVmIGVycm9ycyhzZWxmKSAtPiBTZXF1ZW5jZVtBbnldOgogICAgICAgICByZXR1cm4gc2VsZi5fZXJyb3JzCiAKKyAgICBkZWYgX2Zvcm1hdF9lbmRwb2ludF9jb250ZXh0KHNlbGYpIC0+IHN0cjoKKyAgICAgICAgaWYgbm90IChzZWxmLmVuZHBvaW50X2ZpbGUgYW5kIHNlbGYuZW5kcG9pbnRfbGluZSBhbmQgc2VsZi5lbmRwb2ludF9mdW5jdGlvbik6CisgICAgICAgICAgICBpZiBzZWxmLmVuZHBvaW50X3BhdGg6CisgICAgICAgICAgICAgICAgcmV0dXJuIGYiXG4gIEVuZHBvaW50OiB7c2VsZi5lbmRwb2ludF9wYXRofSIKKyAgICAgICAgICAgIHJldHVybiAiIgorCisgICAgICAgIGNvbnRleHQgPSBmJ1xuICBGaWxlICJ7c2VsZi5lbmRwb2ludF9maWxlfSIsIGxpbmUge3NlbGYuZW5kcG9pbnRfbGluZX0sIGluIHtzZWxmLmVuZHBvaW50X2Z1bmN0aW9ufScKKyAgICAgICAgaWYgc2VsZi5lbmRwb2ludF9wYXRoOgorICAgICAgICAgICAgY29udGV4dCArPSBmIlxuICAgIHtzZWxmLmVuZHBvaW50X3BhdGh9IgorICAgICAgICByZXR1cm4gY29udGV4dAorCisgICAgZGVmIF9fc3RyX18oc2VsZikgLT4gc3RyOgorICAgICAgICBtZXNzYWdlID0gZiJ7bGVuKHNlbGYuX2Vycm9ycyl9IHZhbGlkYXRpb24gZXJyb3J7J3MnIGlmIGxlbihzZWxmLl9lcnJvcnMpICE9IDEgZWxzZSAnJ306XG4iCisgICAgICAgIGZvciBlcnIgaW4gc2VsZi5fZXJyb3JzOgorICAgICAgICAgICAgbWVzc2FnZSArPSBmIiAge2Vycn1cbiIKKyAgICAgICAgbWVzc2FnZSArPSBzZWxmLl9mb3JtYXRfZW5kcG9pbnRfY29udGV4dCgpCisgICAgICAgIHJldHVybiBtZXNzYWdlLnJzdHJpcCgpCisKIAogY2xhc3MgUmVxdWVzdFZhbGlkYXRpb25FcnJvcihWYWxpZGF0aW9uRXhjZXB0aW9uKToKLSAgICBkZWYgX19pbml0X18oc2VsZiwgZXJyb3JzOiBTZXF1ZW5jZVtBbnldLCAqLCBib2R5OiBBbnkgPSBOb25lKSAtPiBOb25lOgotICAgICAgICBzdXBlcigpLl9faW5pdF9fKGVycm9ycykKKyAgICBkZWYgX19pbml0X18oCisgICAgICAgIHNlbGYsCisgICAgICAgIGVycm9yczogU2VxdWVuY2VbQW55XSwKKyAgICAgICAgKiwKKyAgICAgICAgYm9keTogQW55ID0gTm9uZSwKKyAgICAgICAgZW5kcG9pbnRfY3R4OiBPcHRpb25hbFtFbmRwb2ludENvbnRleHRdID0gTm9uZSwKKyAgICApIC0+IE5vbmU6CisgICAgICAgIHN1cGVyKCkuX19pbml0X18oZXJyb3JzLCBlbmRwb2ludF9jdHg9ZW5kcG9pbnRfY3R4KQogICAgICAgICBzZWxmLmJvZHkgPSBib2R5CiAKIAogY2xhc3MgV2ViU29ja2V0UmVxdWVzdFZhbGlkYXRpb25FcnJvcihWYWxpZGF0aW9uRXhjZXB0aW9uKToKLSAgICBwYXNzCisgICAgZGVmIF9faW5pdF9fKAorICAgICAgICBzZWxmLAorICAgICAgICBlcnJvcnM6IFNlcXVlbmNlW0FueV0sCisgICAgICAgICosCisgICAgICAgIGVuZHBvaW50X2N0eDogT3B0aW9uYWxbRW5kcG9pbnRDb250ZXh0XSA9IE5vbmUsCisgICAgKSAtPiBOb25lOgorICAgICAgICBzdXBlcigpLl9faW5pdF9fKGVycm9ycywgZW5kcG9pbnRfY3R4PWVuZHBvaW50X2N0eCkKIAogCiBjbGFzcyBSZXNwb25zZVZhbGlkYXRpb25FcnJvcihWYWxpZGF0aW9uRXhjZXB0aW9uKToKLSAgICBkZWYgX19pbml0X18oc2VsZiwgZXJyb3JzOiBTZXF1ZW5jZVtBbnldLCAqLCBib2R5OiBBbnkgPSBOb25lKSAtPiBOb25lOgotICAgICAgICBzdXBlcigpLl9faW5pdF9fKGVycm9ycykKKyAgICBkZWYgX19pbml0X18oCisgICAgICAgIHNlbGYsCisgICAgICAgIGVycm9yczogU2VxdWVuY2VbQW55XSwKKyAgICAgICAgKiwKKyAgICAgICAgYm9keTogQW55ID0gTm9uZSwKKyAgICAgICAgZW5kcG9pbnRfY3R4OiBPcHRpb25hbFtFbmRwb2ludENvbnRleHRdID0gTm9uZSwKKyAgICApIC0+IE5vbmU6CisgICAgICAgIHN1cGVyKCkuX19pbml0X18oZXJyb3JzLCBlbmRwb2ludF9jdHg9ZW5kcG9pbnRfY3R4KQogICAgICAgICBzZWxmLmJvZHkgPSBib2R5Ci0KLSAgICBkZWYgX19zdHJfXyhzZWxmKSAtPiBzdHI6Ci0gICAgICAgIG1lc3NhZ2UgPSBmIntsZW4oc2VsZi5fZXJyb3JzKX0gdmFsaWRhdGlvbiBlcnJvcnM6XG4iCi0gICAgICAgIGZvciBlcnIgaW4gc2VsZi5fZXJyb3JzOgotICAgICAgICAgICAgbWVzc2FnZSArPSBmIiAge2Vycn1cbiIKLSAgICAgICAgcmV0dXJuIG1lc3NhZ2UKLS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtNDYsNiArNDYsNyBAQAogKQogZnJvbSBmYXN0YXBpLmVuY29kZXJzIGltcG9ydCBqc29uYWJsZV9lbmNvZGVyCiBmcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgKAorICAgIEVuZHBvaW50Q29udGV4dCwKICAgICBGYXN0QVBJRXJyb3IsCiAgICAgUmVxdWVzdFZhbGlkYXRpb25FcnJvciwKICAgICBSZXNwb25zZVZhbGlkYXRpb25FcnJvciwKQEAgLTIxMiw2ICsyMTMsMzMgQEAgYXN5bmMgZGVmIG1lcmdlZF9saWZlc3BhbigKICAgICByZXR1cm4gbWVyZ2VkX2xpZmVzcGFuICAjIHR5cGU6IGlnbm9yZVtyZXR1cm4tdmFsdWVdCiAKIAorIyBDYWNoZSBmb3IgZW5kcG9pbnQgY29udGV4dCB0byBhdm9pZCByZS1leHRyYWN0aW5nIG9uIGV2ZXJ5IHJlcXVlc3QKK19lbmRwb2ludF9jb250ZXh0X2NhY2hlOiBEaWN0W2ludCwgRW5kcG9pbnRDb250ZXh0XSA9IHt9CisKKworZGVmIF9leHRyYWN0X2VuZHBvaW50X2NvbnRleHQoZnVuYzogQW55KSAtPiBFbmRwb2ludENvbnRleHQ6CisgICAgIiIiRXh0cmFjdCBlbmRwb2ludCBjb250ZXh0IHdpdGggY2FjaGluZyB0byBhdm9pZCByZXBlYXRlZCBmaWxlIEkvTy4iIiIKKyAgICBmdW5jX2lkID0gaWQoZnVuYykKKworICAgIGlmIGZ1bmNfaWQgaW4gX2VuZHBvaW50X2NvbnRleHRfY2FjaGU6CisgICAgICAgIHJldHVybiBfZW5kcG9pbnRfY29udGV4dF9jYWNoZVtmdW5jX2lkXQorCisgICAgdHJ5OgorICAgICAgICBjdHg6IEVuZHBvaW50Q29udGV4dCA9IHt9CisKKyAgICAgICAgaWYgKHNvdXJjZV9maWxlIDo9IGluc3BlY3QuZ2V0c291cmNlZmlsZShmdW5jKSkgaXMgbm90IE5vbmU6CisgICAgICAgICAgICBjdHhbImZpbGUiXSA9IHNvdXJjZV9maWxlCisgICAgICAgIGlmIChsaW5lX251bWJlciA6PSBpbnNwZWN0LmdldHNvdXJjZWxpbmVzKGZ1bmMpWzFdKSBpcyBub3QgTm9uZToKKyAgICAgICAgICAgIGN0eFsibGluZSJdID0gbGluZV9udW1iZXIKKyAgICAgICAgaWYgKGZ1bmNfbmFtZSA6PSBnZXRhdHRyKGZ1bmMsICJfX25hbWVfXyIsIE5vbmUpKSBpcyBub3QgTm9uZToKKyAgICAgICAgICAgIGN0eFsiZnVuY3Rpb24iXSA9IGZ1bmNfbmFtZQorICAgIGV4Y2VwdCBFeGNlcHRpb246CisgICAgICAgIGN0eCA9IEVuZHBvaW50Q29udGV4dCgpCisKKyAgICBfZW5kcG9pbnRfY29udGV4dF9jYWNoZVtmdW5jX2lkXSA9IGN0eAorICAgIHJldHVybiBjdHgKKworCiBhc3luYyBkZWYgc2VyaWFsaXplX3Jlc3BvbnNlKAogICAgICosCiAgICAgZmllbGQ6IE9wdGlvbmFsW01vZGVsRmllbGRdID0gTm9uZSwKQEAgLTIyMyw2ICsyNTEsNyBAQCBhc3luYyBkZWYgc2VyaWFsaXplX3Jlc3BvbnNlKAogICAgIGV4Y2x1ZGVfZGVmYXVsdHM6IGJvb2wgPSBGYWxzZSwKICAgICBleGNsdWRlX25vbmU6IGJvb2wgPSBGYWxzZSwKICAgICBpc19jb3JvdXRpbmU6IGJvb2wgPSBUcnVlLAorICAgIGVuZHBvaW50X2N0eDogT3B0aW9uYWxbRW5kcG9pbnRDb250ZXh0XSA9IE5vbmUsCiApIC0+IEFueToKICAgICBpZiBmaWVsZDoKICAgICAgICAgZXJyb3JzID0gW10KQEAgLTI0NSw4ICsyNzQsMTEgQEAgYXN5bmMgZGVmIHNlcmlhbGl6ZV9yZXNwb25zZSgKICAgICAgICAgZWxpZiBlcnJvcnNfOgogICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChlcnJvcnNfKQogICAgICAgICBpZiBlcnJvcnM6CisgICAgICAgICAgICBjdHggPSBlbmRwb2ludF9jdHggb3IgRW5kcG9pbnRDb250ZXh0KCkKICAgICAgICAgICAgIHJhaXNlIFJlc3BvbnNlVmFsaWRhdGlvbkVycm9yKAotICAgICAgICAgICAgICAgIGVycm9ycz1fbm9ybWFsaXplX2Vycm9ycyhlcnJvcnMpLCBib2R5PXJlc3BvbnNlX2NvbnRlbnQKKyAgICAgICAgICAgICAgICBlcnJvcnM9X25vcm1hbGl6ZV9lcnJvcnMoZXJyb3JzKSwKKyAgICAgICAgICAgICAgICBib2R5PXJlc3BvbnNlX2NvbnRlbnQsCisgICAgICAgICAgICAgICAgZW5kcG9pbnRfY3R4PWN0eCwKICAgICAgICAgICAgICkKIAogICAgICAgICBpZiBoYXNhdHRyKGZpZWxkLCAic2VyaWFsaXplIik6CkBAIC0zMTgsNiArMzUwLDE4IEBAIGFzeW5jIGRlZiBhcHAocmVxdWVzdDogUmVxdWVzdCkgLT4gUmVzcG9uc2U6CiAgICAgICAgICAgICAiZmFzdGFwaV9taWRkbGV3YXJlX2FzdGFjayBub3QgZm91bmQgaW4gcmVxdWVzdCBzY29wZSIKICAgICAgICAgKQogCisgICAgICAgICMgRXh0cmFjdCBlbmRwb2ludCBjb250ZXh0IGZvciBlcnJvciBtZXNzYWdlcworICAgICAgICBlbmRwb2ludF9jdHggPSAoCisgICAgICAgICAgICBfZXh0cmFjdF9lbmRwb2ludF9jb250ZXh0KGRlcGVuZGFudC5jYWxsKQorICAgICAgICAgICAgaWYgZGVwZW5kYW50LmNhbGwKKyAgICAgICAgICAgIGVsc2UgRW5kcG9pbnRDb250ZXh0KCkKKyAgICAgICAgKQorCisgICAgICAgIGlmIGRlcGVuZGFudC5wYXRoOgorICAgICAgICAgICAgIyBGb3IgbW91bnRlZCBzdWItYXBwcywgaW5jbHVkZSB0aGUgbW91bnQgcGF0aCBwcmVmaXgKKyAgICAgICAgICAgIG1vdW50X3BhdGggPSByZXF1ZXN0LnNjb3BlLmdldCgicm9vdF9wYXRoIiwgIiIpLnJzdHJpcCgiLyIpCisgICAgICAgICAgICBlbmRwb2ludF9jdHhbInBhdGgiXSA9IGYie3JlcXVlc3QubWV0aG9kfSB7bW91bnRfcGF0aH17ZGVwZW5kYW50LnBhdGh9IgorCiAgICAgICAgICMgUmVhZCBib2R5IGFuZCBhdXRvLWNsb3NlIGZpbGVzCiAgICAgICAgIHRyeToKICAgICAgICAgICAgIGJvZHk6IEFueSA9IE5vbmUKQEAgLTM1NSw2ICszOTksNyBAQCBhc3luYyBkZWYgYXBwKHJlcXVlc3Q6IFJlcXVlc3QpIC0+IFJlc3BvbnNlOgogICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICAgXSwKICAgICAgICAgICAgICAgICBib2R5PWUuZG9jLAorICAgICAgICAgICAgICAgIGVuZHBvaW50X2N0eD1lbmRwb2ludF9jdHgsCiAgICAgICAgICAgICApCiAgICAgICAgICAgICByYWlzZSB2YWxpZGF0aW9uX2Vycm9yIGZyb20gZQogICAgICAgICBleGNlcHQgSFRUUEV4Y2VwdGlvbjoKQEAgLTQxNCwxNCArNDU5LDE1IEBAIGFzeW5jIGRlZiBhcHAocmVxdWVzdDogUmVxdWVzdCkgLT4gUmVzcG9uc2U6CiAgICAgICAgICAgICAgICAgICAgIGV4Y2x1ZGVfZGVmYXVsdHM9cmVzcG9uc2VfbW9kZWxfZXhjbHVkZV9kZWZhdWx0cywKICAgICAgICAgICAgICAgICAgICAgZXhjbHVkZV9ub25lPXJlc3BvbnNlX21vZGVsX2V4Y2x1ZGVfbm9uZSwKICAgICAgICAgICAgICAgICAgICAgaXNfY29yb3V0aW5lPWlzX2Nvcm91dGluZSwKKyAgICAgICAgICAgICAgICAgICAgZW5kcG9pbnRfY3R4PWVuZHBvaW50X2N0eCwKICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgcmVzcG9uc2UgPSBhY3R1YWxfcmVzcG9uc2VfY2xhc3MoY29udGVudCwgKipyZXNwb25zZV9hcmdzKQogICAgICAgICAgICAgICAgIGlmIG5vdCBpc19ib2R5X2FsbG93ZWRfZm9yX3N0YXR1c19jb2RlKHJlc3BvbnNlLnN0YXR1c19jb2RlKToKICAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2UuYm9keSA9IGIiIgogICAgICAgICAgICAgICAgIHJlc3BvbnNlLmhlYWRlcnMucmF3LmV4dGVuZChzb2x2ZWRfcmVzdWx0LnJlc3BvbnNlLmhlYWRlcnMucmF3KQogICAgICAgICBpZiBlcnJvcnM6CiAgICAgICAgICAgICB2YWxpZGF0aW9uX2Vycm9yID0gUmVxdWVzdFZhbGlkYXRpb25FcnJvcigKLSAgICAgICAgICAgICAgICBfbm9ybWFsaXplX2Vycm9ycyhlcnJvcnMpLCBib2R5PWJvZHkKKyAgICAgICAgICAgICAgICBfbm9ybWFsaXplX2Vycm9ycyhlcnJvcnMpLCBib2R5PWJvZHksIGVuZHBvaW50X2N0eD1lbmRwb2ludF9jdHgKICAgICAgICAgICAgICkKICAgICAgICAgICAgIHJhaXNlIHZhbGlkYXRpb25fZXJyb3IKIApAQCAtNDM4LDYgKzQ4NCwxNSBAQCBkZWYgZ2V0X3dlYnNvY2tldF9hcHAoCiAgICAgZW1iZWRfYm9keV9maWVsZHM6IGJvb2wgPSBGYWxzZSwKICkgLT4gQ2FsbGFibGVbW1dlYlNvY2tldF0sIENvcm91dGluZVtBbnksIEFueSwgQW55XV06CiAgICAgYXN5bmMgZGVmIGFwcCh3ZWJzb2NrZXQ6IFdlYlNvY2tldCkgLT4gTm9uZToKKyAgICAgICAgZW5kcG9pbnRfY3R4ID0gKAorICAgICAgICAgICAgX2V4dHJhY3RfZW5kcG9pbnRfY29udGV4dChkZXBlbmRhbnQuY2FsbCkKKyAgICAgICAgICAgIGlmIGRlcGVuZGFudC5jYWxsCisgICAgICAgICAgICBlbHNlIEVuZHBvaW50Q29udGV4dCgpCisgICAgICAgICkKKyAgICAgICAgaWYgZGVwZW5kYW50LnBhdGg6CisgICAgICAgICAgICAjIEZvciBtb3VudGVkIHN1Yi1hcHBzLCBpbmNsdWRlIHRoZSBtb3VudCBwYXRoIHByZWZpeAorICAgICAgICAgICAgbW91bnRfcGF0aCA9IHdlYnNvY2tldC5zY29wZS5nZXQoInJvb3RfcGF0aCIsICIiKS5yc3RyaXAoIi8iKQorICAgICAgICAgICAgZW5kcG9pbnRfY3R4WyJwYXRoIl0gPSBmIldTIHttb3VudF9wYXRofXtkZXBlbmRhbnQucGF0aH0iCiAgICAgICAgIGFzeW5jX2V4aXRfc3RhY2sgPSB3ZWJzb2NrZXQuc2NvcGUuZ2V0KCJmYXN0YXBpX2lubmVyX2FzdGFjayIpCiAgICAgICAgIGFzc2VydCBpc2luc3RhbmNlKGFzeW5jX2V4aXRfc3RhY2ssIEFzeW5jRXhpdFN0YWNrKSwgKAogICAgICAgICAgICAgImZhc3RhcGlfaW5uZXJfYXN0YWNrIG5vdCBmb3VuZCBpbiByZXF1ZXN0IHNjb3BlIgpAQCAtNDUxLDcgKzUwNiw4IEBAIGFzeW5jIGRlZiBhcHAod2Vic29ja2V0OiBXZWJTb2NrZXQpIC0+IE5vbmU6CiAgICAgICAgICkKICAgICAgICAgaWYgc29sdmVkX3Jlc3VsdC5lcnJvcnM6CiAgICAgICAgICAgICByYWlzZSBXZWJTb2NrZXRSZXF1ZXN0VmFsaWRhdGlvbkVycm9yKAotICAgICAgICAgICAgICAgIF9ub3JtYWxpemVfZXJyb3JzKHNvbHZlZF9yZXN1bHQuZXJyb3JzKQorICAgICAgICAgICAgICAgIF9ub3JtYWxpemVfZXJyb3JzKHNvbHZlZF9yZXN1bHQuZXJyb3JzKSwKKyAgICAgICAgICAgICAgICBlbmRwb2ludF9jdHg9ZW5kcG9pbnRfY3R4LAogICAgICAgICAgICAgKQogICAgICAgICBhc3NlcnQgZGVwZW5kYW50LmNhbGwgaXMgbm90IE5vbmUsICJkZXBlbmRhbnQuY2FsbCBtdXN0IGJlIGEgZnVuY3Rpb24iCiAgICAgICAgIGF3YWl0IGRlcGVuZGFudC5jYWxsKCoqc29sdmVkX3Jlc3VsdC52YWx1ZXMpCg==", "fastapi_14458": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMTEwLDYgKzExMCw4IEBAIGRlZiBpc19nZW5fY2FsbGFibGUoc2VsZikgLT4gYm9vbDoKICAgICAgICAgICAgIF9pbXBhcnRpYWwoc2VsZi5jYWxsKQogICAgICAgICApIG9yIGluc3BlY3QuaXNnZW5lcmF0b3JmdW5jdGlvbihfdW53cmFwcGVkX2NhbGwoc2VsZi5jYWxsKSk6CiAgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBpZiBpbnNwZWN0LmlzY2xhc3MoX3Vud3JhcHBlZF9jYWxsKHNlbGYuY2FsbCkpOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihfaW1wYXJ0aWFsKHNlbGYuY2FsbCksICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKICAgICAgICAgaWYgZHVuZGVyX2NhbGwgaXMgTm9uZToKICAgICAgICAgICAgIHJldHVybiBGYWxzZSAgIyBwcmFnbWE6IG5vIGNvdmVyCkBAIC0xMzQsNiArMTM2LDggQEAgZGVmIGlzX2FzeW5jX2dlbl9jYWxsYWJsZShzZWxmKSAtPiBib29sOgogICAgICAgICAgICAgX2ltcGFydGlhbChzZWxmLmNhbGwpCiAgICAgICAgICkgb3IgaW5zcGVjdC5pc2FzeW5jZ2VuZnVuY3Rpb24oX3Vud3JhcHBlZF9jYWxsKHNlbGYuY2FsbCkpOgogICAgICAgICAgICAgcmV0dXJuIFRydWUKKyAgICAgICAgaWYgaW5zcGVjdC5pc2NsYXNzKF91bndyYXBwZWRfY2FsbChzZWxmLmNhbGwpKToKKyAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgICBkdW5kZXJfY2FsbCA9IGdldGF0dHIoX2ltcGFydGlhbChzZWxmLmNhbGwpLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0CiAgICAgICAgIGlmIGR1bmRlcl9jYWxsIGlzIE5vbmU6CiAgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgpAQCAtMTYyLDYgKzE2Niw4IEBAIGRlZiBpc19jb3JvdXRpbmVfY2FsbGFibGUoc2VsZikgLT4gYm9vbDoKICAgICAgICAgICAgIF91bndyYXBwZWRfY2FsbChzZWxmLmNhbGwpCiAgICAgICAgICk6CiAgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBpZiBpbnNwZWN0LmlzY2xhc3MoX3Vud3JhcHBlZF9jYWxsKHNlbGYuY2FsbCkpOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihfaW1wYXJ0aWFsKHNlbGYuY2FsbCksICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKICAgICAgICAgaWYgZHVuZGVyX2NhbGwgaXMgTm9uZToKICAgICAgICAgICAgIHJldHVybiBGYWxzZSAgIyBwcmFnbWE6IG5vIGNvdmVyCkBAIC0xNzYsNyArMTgyLDYgQEAgZGVmIGlzX2Nvcm91dGluZV9jYWxsYWJsZShzZWxmKSAtPiBib29sOgogICAgICAgICAgICAgX2ltcGFydGlhbChkdW5kZXJfdW53cmFwcGVkX2NhbGwpCiAgICAgICAgICkgb3IgaXNjb3JvdXRpbmVmdW5jdGlvbihfdW53cmFwcGVkX2NhbGwoZHVuZGVyX3Vud3JhcHBlZF9jYWxsKSk6CiAgICAgICAgICAgICByZXR1cm4gVHJ1ZQotICAgICAgICAjIGlmIGluc3BlY3QuaXNjbGFzcyhzZWxmLmNhbGwpOiBGYWxzZSwgY292ZXJlZCBieSBkZWZhdWx0IHJldHVybgogICAgICAgICByZXR1cm4gRmFsc2UKIAogICAgIEBjYWNoZWRfcHJvcGVydHkK", "fastapi_14459": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMiw3ICsyLDcgQEAKIGltcG9ydCBzeXMKIGZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKIGZyb20gZnVuY3Rvb2xzIGltcG9ydCBjYWNoZWRfcHJvcGVydHksIHBhcnRpYWwKLWZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBMaXN0LCBPcHRpb25hbCwgU2VxdWVuY2UsIFVuaW9uCitmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZSwgTGlzdCwgT3B0aW9uYWwsIFVuaW9uCiAKIGZyb20gZmFzdGFwaS5fY29tcGF0IGltcG9ydCBNb2RlbEZpZWxkCiBmcm9tIGZhc3RhcGkuc2VjdXJpdHkuYmFzZSBpbXBvcnQgU2VjdXJpdHlCYXNlCkBAIC0yOCwxMiArMjgsNiBAQCBkZWYgX2ltcGFydGlhbChmdW5jOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IENhbGxhYmxlWy4uLiwgQW55XToKICAgICByZXR1cm4gZnVuYwogCiAKLUBkYXRhY2xhc3MKLWNsYXNzIFNlY3VyaXR5UmVxdWlyZW1lbnQ6Ci0gICAgc2VjdXJpdHlfc2NoZW1lOiBTZWN1cml0eUJhc2UKLSAgICBzY29wZXM6IE9wdGlvbmFsW1NlcXVlbmNlW3N0cl1dID0gTm9uZQotCi0KIEBkYXRhY2xhc3MKIGNsYXNzIERlcGVuZGFudDoKICAgICBwYXRoX3BhcmFtczogTGlzdFtNb2RlbEZpZWxkXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQpAQCAtNDIsNyArMzYsNiBAQCBjbGFzcyBEZXBlbmRhbnQ6CiAgICAgY29va2llX3BhcmFtczogTGlzdFtNb2RlbEZpZWxkXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgIGJvZHlfcGFyYW1zOiBMaXN0W01vZGVsRmllbGRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICAgZGVwZW5kZW5jaWVzOiBMaXN0WyJEZXBlbmRhbnQiXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQotICAgIHNlY3VyaXR5X3JlcXVpcmVtZW50czogTGlzdFtTZWN1cml0eVJlcXVpcmVtZW50XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgIG5hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lCiAgICAgY2FsbDogT3B0aW9uYWxbQ2FsbGFibGVbLi4uLCBBbnldXSA9IE5vbmUKICAgICByZXF1ZXN0X3BhcmFtX25hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lCkBAIC04MywxMSArNzYsMzIgQEAgZGVmIF91c2VzX3Njb3BlcyhzZWxmKSAtPiBib29sOgogICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgaWYgc2VsZi5zZWN1cml0eV9zY29wZXNfcGFyYW1fbmFtZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgIHJldHVybiBUcnVlCisgICAgICAgIGlmIHNlbGYuX2lzX3NlY3VyaXR5X3NjaGVtZToKKyAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgIGZvciBzdWJfZGVwIGluIHNlbGYuZGVwZW5kZW5jaWVzOgogICAgICAgICAgICAgaWYgc3ViX2RlcC5fdXNlc19zY29wZXM6CiAgICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgcmV0dXJuIEZhbHNlCiAKKyAgICBAY2FjaGVkX3Byb3BlcnR5CisgICAgZGVmIF9pc19zZWN1cml0eV9zY2hlbWUoc2VsZikgLT4gYm9vbDoKKyAgICAgICAgaWYgc2VsZi5jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgorICAgICAgICB1bndyYXBwZWQgPSBfdW53cmFwcGVkX2NhbGwoc2VsZi5jYWxsKQorICAgICAgICByZXR1cm4gaXNpbnN0YW5jZSh1bndyYXBwZWQsIFNlY3VyaXR5QmFzZSkKKworICAgICMgTWFpbmx5IHRvIGdldCB0aGUgdHlwZSBvZiBTZWN1cml0eUJhc2UsIGJ1dCBpdCdzIHRoZSBzYW1lIHNlbGYuY2FsbAorICAgIEBjYWNoZWRfcHJvcGVydHkKKyAgICBkZWYgX3NlY3VyaXR5X3NjaGVtZShzZWxmKSAtPiBTZWN1cml0eUJhc2U6CisgICAgICAgIHVud3JhcHBlZCA9IF91bndyYXBwZWRfY2FsbChzZWxmLmNhbGwpCisgICAgICAgIGFzc2VydCBpc2luc3RhbmNlKHVud3JhcHBlZCwgU2VjdXJpdHlCYXNlKQorICAgICAgICByZXR1cm4gdW53cmFwcGVkCisKKyAgICBAY2FjaGVkX3Byb3BlcnR5CisgICAgZGVmIF9zZWN1cml0eV9kZXBlbmRlbmNpZXMoc2VsZikgLT4gTGlzdFsiRGVwZW5kYW50Il06CisgICAgICAgIHNlY3VyaXR5X2RlcHMgPSBbZGVwIGZvciBkZXAgaW4gc2VsZi5kZXBlbmRlbmNpZXMgaWYgZGVwLl9pc19zZWN1cml0eV9zY2hlbWVdCisgICAgICAgIHJldHVybiBzZWN1cml0eV9kZXBzCisKICAgICBAY2FjaGVkX3Byb3BlcnR5CiAgICAgZGVmIGlzX2dlbl9jYWxsYWJsZShzZWxmKSAtPiBib29sOgogICAgICAgICBpZiBzZWxmLmNhbGwgaXMgTm9uZToKLS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTU1LDEwICs1NSw5IEBACiAgICAgYXN5bmNjb250ZXh0bWFuYWdlciwKICAgICBjb250ZXh0bWFuYWdlcl9pbl90aHJlYWRwb29sLAogKQotZnJvbSBmYXN0YXBpLmRlcGVuZGVuY2llcy5tb2RlbHMgaW1wb3J0IERlcGVuZGFudCwgU2VjdXJpdHlSZXF1aXJlbWVudAorZnJvbSBmYXN0YXBpLmRlcGVuZGVuY2llcy5tb2RlbHMgaW1wb3J0IERlcGVuZGFudAogZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0IERlcGVuZGVuY3lTY29wZUVycm9yCiBmcm9tIGZhc3RhcGkubG9nZ2VyIGltcG9ydCBsb2dnZXIKLWZyb20gZmFzdGFwaS5zZWN1cml0eS5iYXNlIGltcG9ydCBTZWN1cml0eUJhc2UKIGZyb20gZmFzdGFwaS5zZWN1cml0eS5vYXV0aDIgaW1wb3J0IFNlY3VyaXR5U2NvcGVzCiBmcm9tIGZhc3RhcGkudHlwZXMgaW1wb3J0IERlcGVuZGVuY3lDYWNoZUtleQogZnJvbSBmYXN0YXBpLnV0aWxzIGltcG9ydCBjcmVhdGVfbW9kZWxfZmllbGQsIGdldF9wYXRoX3BhcmFtX25hbWVzCkBAIC0xNDIsMzMgKzE0MSw1MiBAQCBkZWYgZ2V0X2ZsYXRfZGVwZW5kYW50KAogICAgICosCiAgICAgc2tpcF9yZXBlYXRzOiBib29sID0gRmFsc2UsCiAgICAgdmlzaXRlZDogT3B0aW9uYWxbTGlzdFtEZXBlbmRlbmN5Q2FjaGVLZXldXSA9IE5vbmUsCisgICAgcGFyZW50X29hdXRoX3Njb3BlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUsCiApIC0+IERlcGVuZGFudDoKICAgICBpZiB2aXNpdGVkIGlzIE5vbmU6CiAgICAgICAgIHZpc2l0ZWQgPSBbXQogICAgIHZpc2l0ZWQuYXBwZW5kKGRlcGVuZGFudC5jYWNoZV9rZXkpCisgICAgdXNlX3BhcmVudF9vYXV0aF9zY29wZXMgPSAocGFyZW50X29hdXRoX3Njb3BlcyBvciBbXSkgKyAoCisgICAgICAgIGRlcGVuZGFudC5vYXV0aF9zY29wZXMgb3IgW10KKyAgICApCiAKICAgICBmbGF0X2RlcGVuZGFudCA9IERlcGVuZGFudCgKICAgICAgICAgcGF0aF9wYXJhbXM9ZGVwZW5kYW50LnBhdGhfcGFyYW1zLmNvcHkoKSwKICAgICAgICAgcXVlcnlfcGFyYW1zPWRlcGVuZGFudC5xdWVyeV9wYXJhbXMuY29weSgpLAogICAgICAgICBoZWFkZXJfcGFyYW1zPWRlcGVuZGFudC5oZWFkZXJfcGFyYW1zLmNvcHkoKSwKICAgICAgICAgY29va2llX3BhcmFtcz1kZXBlbmRhbnQuY29va2llX3BhcmFtcy5jb3B5KCksCiAgICAgICAgIGJvZHlfcGFyYW1zPWRlcGVuZGFudC5ib2R5X3BhcmFtcy5jb3B5KCksCi0gICAgICAgIHNlY3VyaXR5X3JlcXVpcmVtZW50cz1kZXBlbmRhbnQuc2VjdXJpdHlfcmVxdWlyZW1lbnRzLmNvcHkoKSwKKyAgICAgICAgbmFtZT1kZXBlbmRhbnQubmFtZSwKKyAgICAgICAgY2FsbD1kZXBlbmRhbnQuY2FsbCwKKyAgICAgICAgcmVxdWVzdF9wYXJhbV9uYW1lPWRlcGVuZGFudC5yZXF1ZXN0X3BhcmFtX25hbWUsCisgICAgICAgIHdlYnNvY2tldF9wYXJhbV9uYW1lPWRlcGVuZGFudC53ZWJzb2NrZXRfcGFyYW1fbmFtZSwKKyAgICAgICAgaHR0cF9jb25uZWN0aW9uX3BhcmFtX25hbWU9ZGVwZW5kYW50Lmh0dHBfY29ubmVjdGlvbl9wYXJhbV9uYW1lLAorICAgICAgICByZXNwb25zZV9wYXJhbV9uYW1lPWRlcGVuZGFudC5yZXNwb25zZV9wYXJhbV9uYW1lLAorICAgICAgICBiYWNrZ3JvdW5kX3Rhc2tzX3BhcmFtX25hbWU9ZGVwZW5kYW50LmJhY2tncm91bmRfdGFza3NfcGFyYW1fbmFtZSwKKyAgICAgICAgc2VjdXJpdHlfc2NvcGVzX3BhcmFtX25hbWU9ZGVwZW5kYW50LnNlY3VyaXR5X3Njb3Blc19wYXJhbV9uYW1lLAorICAgICAgICBvd25fb2F1dGhfc2NvcGVzPWRlcGVuZGFudC5vd25fb2F1dGhfc2NvcGVzLAorICAgICAgICBwYXJlbnRfb2F1dGhfc2NvcGVzPXVzZV9wYXJlbnRfb2F1dGhfc2NvcGVzLAogICAgICAgICB1c2VfY2FjaGU9ZGVwZW5kYW50LnVzZV9jYWNoZSwKICAgICAgICAgcGF0aD1kZXBlbmRhbnQucGF0aCwKKyAgICAgICAgc2NvcGU9ZGVwZW5kYW50LnNjb3BlLAogICAgICkKICAgICBmb3Igc3ViX2RlcGVuZGFudCBpbiBkZXBlbmRhbnQuZGVwZW5kZW5jaWVzOgogICAgICAgICBpZiBza2lwX3JlcGVhdHMgYW5kIHN1Yl9kZXBlbmRhbnQuY2FjaGVfa2V5IGluIHZpc2l0ZWQ6CiAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICBmbGF0X3N1YiA9IGdldF9mbGF0X2RlcGVuZGFudCgKLSAgICAgICAgICAgIHN1Yl9kZXBlbmRhbnQsIHNraXBfcmVwZWF0cz1za2lwX3JlcGVhdHMsIHZpc2l0ZWQ9dmlzaXRlZAorICAgICAgICAgICAgc3ViX2RlcGVuZGFudCwKKyAgICAgICAgICAgIHNraXBfcmVwZWF0cz1za2lwX3JlcGVhdHMsCisgICAgICAgICAgICB2aXNpdGVkPXZpc2l0ZWQsCisgICAgICAgICAgICBwYXJlbnRfb2F1dGhfc2NvcGVzPWZsYXRfZGVwZW5kYW50Lm9hdXRoX3Njb3BlcywKICAgICAgICAgKQorICAgICAgICBmbGF0X2RlcGVuZGFudC5kZXBlbmRlbmNpZXMuYXBwZW5kKGZsYXRfc3ViKQogICAgICAgICBmbGF0X2RlcGVuZGFudC5wYXRoX3BhcmFtcy5leHRlbmQoZmxhdF9zdWIucGF0aF9wYXJhbXMpCiAgICAgICAgIGZsYXRfZGVwZW5kYW50LnF1ZXJ5X3BhcmFtcy5leHRlbmQoZmxhdF9zdWIucXVlcnlfcGFyYW1zKQogICAgICAgICBmbGF0X2RlcGVuZGFudC5oZWFkZXJfcGFyYW1zLmV4dGVuZChmbGF0X3N1Yi5oZWFkZXJfcGFyYW1zKQogICAgICAgICBmbGF0X2RlcGVuZGFudC5jb29raWVfcGFyYW1zLmV4dGVuZChmbGF0X3N1Yi5jb29raWVfcGFyYW1zKQogICAgICAgICBmbGF0X2RlcGVuZGFudC5ib2R5X3BhcmFtcy5leHRlbmQoZmxhdF9zdWIuYm9keV9wYXJhbXMpCi0gICAgICAgIGZsYXRfZGVwZW5kYW50LnNlY3VyaXR5X3JlcXVpcmVtZW50cy5leHRlbmQoZmxhdF9zdWIuc2VjdXJpdHlfcmVxdWlyZW1lbnRzKQorICAgICAgICBmbGF0X2RlcGVuZGFudC5kZXBlbmRlbmNpZXMuZXh0ZW5kKGZsYXRfc3ViLmRlcGVuZGVuY2llcykKKwogICAgIHJldHVybiBmbGF0X2RlcGVuZGFudAogCiAKQEAgLTI1OCwxMSArMjc2LDYgQEAgZGVmIGdldF9kZXBlbmRhbnQoCiAgICAgcGF0aF9wYXJhbV9uYW1lcyA9IGdldF9wYXRoX3BhcmFtX25hbWVzKHBhdGgpCiAgICAgZW5kcG9pbnRfc2lnbmF0dXJlID0gZ2V0X3R5cGVkX3NpZ25hdHVyZShjYWxsKQogICAgIHNpZ25hdHVyZV9wYXJhbXMgPSBlbmRwb2ludF9zaWduYXR1cmUucGFyYW1ldGVycwotICAgIGlmIGlzaW5zdGFuY2UoY2FsbCwgU2VjdXJpdHlCYXNlKToKLSAgICAgICAgc2VjdXJpdHlfcmVxdWlyZW1lbnQgPSBTZWN1cml0eVJlcXVpcmVtZW50KAotICAgICAgICAgICAgc2VjdXJpdHlfc2NoZW1lPWNhbGwsIHNjb3Blcz1jdXJyZW50X3Njb3BlcwotICAgICAgICApCi0gICAgICAgIGRlcGVuZGFudC5zZWN1cml0eV9yZXF1aXJlbWVudHMuYXBwZW5kKHNlY3VyaXR5X3JlcXVpcmVtZW50KQogICAgIGZvciBwYXJhbV9uYW1lLCBwYXJhbSBpbiBzaWduYXR1cmVfcGFyYW1zLml0ZW1zKCk6CiAgICAgICAgIGlzX3BhdGhfcGFyYW0gPSBwYXJhbV9uYW1lIGluIHBhdGhfcGFyYW1fbmFtZXMKICAgICAgICAgcGFyYW1fZGV0YWlscyA9IGFuYWx5emVfcGFyYW0oCi0tLSBhL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQorKysgYi9mYXN0YXBpL29wZW5hcGkvdXRpbHMucHkKQEAgLTgxLDE4ICs4MSwxOCBAQCBkZWYgZ2V0X29wZW5hcGlfc2VjdXJpdHlfZGVmaW5pdGlvbnMoCiAgICAgc2VjdXJpdHlfZGVmaW5pdGlvbnMgPSB7fQogICAgICMgVXNlIGEgZGljdCB0byBtZXJnZSBzY29wZXMgZm9yIHNhbWUgc2VjdXJpdHkgc2NoZW1lCiAgICAgb3BlcmF0aW9uX3NlY3VyaXR5X2RpY3Q6IERpY3Rbc3RyLCBMaXN0W3N0cl1dID0ge30KLSAgICBmb3Igc2VjdXJpdHlfcmVxdWlyZW1lbnQgaW4gZmxhdF9kZXBlbmRhbnQuc2VjdXJpdHlfcmVxdWlyZW1lbnRzOgorICAgIGZvciBzZWN1cml0eV9kZXBlbmRlbmN5IGluIGZsYXRfZGVwZW5kYW50Ll9zZWN1cml0eV9kZXBlbmRlbmNpZXM6CiAgICAgICAgIHNlY3VyaXR5X2RlZmluaXRpb24gPSBqc29uYWJsZV9lbmNvZGVyKAotICAgICAgICAgICAgc2VjdXJpdHlfcmVxdWlyZW1lbnQuc2VjdXJpdHlfc2NoZW1lLm1vZGVsLAorICAgICAgICAgICAgc2VjdXJpdHlfZGVwZW5kZW5jeS5fc2VjdXJpdHlfc2NoZW1lLm1vZGVsLAogICAgICAgICAgICAgYnlfYWxpYXM9VHJ1ZSwKICAgICAgICAgICAgIGV4Y2x1ZGVfbm9uZT1UcnVlLAogICAgICAgICApCi0gICAgICAgIHNlY3VyaXR5X25hbWUgPSBzZWN1cml0eV9yZXF1aXJlbWVudC5zZWN1cml0eV9zY2hlbWUuc2NoZW1lX25hbWUKKyAgICAgICAgc2VjdXJpdHlfbmFtZSA9IHNlY3VyaXR5X2RlcGVuZGVuY3kuX3NlY3VyaXR5X3NjaGVtZS5zY2hlbWVfbmFtZQogICAgICAgICBzZWN1cml0eV9kZWZpbml0aW9uc1tzZWN1cml0eV9uYW1lXSA9IHNlY3VyaXR5X2RlZmluaXRpb24KICAgICAgICAgIyBNZXJnZSBzY29wZXMgZm9yIHRoZSBzYW1lIHNlY3VyaXR5IHNjaGVtZQogICAgICAgICBpZiBzZWN1cml0eV9uYW1lIG5vdCBpbiBvcGVyYXRpb25fc2VjdXJpdHlfZGljdDoKICAgICAgICAgICAgIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0W3NlY3VyaXR5X25hbWVdID0gW10KLSAgICAgICAgZm9yIHNjb3BlIGluIHNlY3VyaXR5X3JlcXVpcmVtZW50LnNjb3BlcyBvciBbXToKKyAgICAgICAgZm9yIHNjb3BlIGluIHNlY3VyaXR5X2RlcGVuZGVuY3kub2F1dGhfc2NvcGVzIG9yIFtdOgogICAgICAgICAgICAgaWYgc2NvcGUgbm90IGluIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0W3NlY3VyaXR5X25hbWVdOgogICAgICAgICAgICAgICAgIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0W3NlY3VyaXR5X25hbWVdLmFwcGVuZChzY29wZSkKICAgICBvcGVyYXRpb25fc2VjdXJpdHkgPSBbCg==", "fastapi_14448": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMTUsNiArMTUsMTkgQEAKICAgICBmcm9tIGFzeW5jaW8gaW1wb3J0IGlzY29yb3V0aW5lZnVuY3Rpb24KIAogCitkZWYgX3Vud3JhcHBlZF9jYWxsKGNhbGw6IE9wdGlvbmFsW0NhbGxhYmxlWy4uLiwgQW55XV0pIC0+IEFueToKKyAgICBpZiBjYWxsIGlzIE5vbmU6CisgICAgICAgIHJldHVybiBjYWxsICAjIHByYWdtYTogbm8gY292ZXIKKyAgICB1bndyYXBwZWQgPSBpbnNwZWN0LnVud3JhcChfaW1wYXJ0aWFsKGNhbGwpKQorICAgIHJldHVybiB1bndyYXBwZWQKKworCitkZWYgX2ltcGFydGlhbChmdW5jOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IENhbGxhYmxlWy4uLiwgQW55XToKKyAgICB3aGlsZSBpc2luc3RhbmNlKGZ1bmMsIHBhcnRpYWwpOgorICAgICAgICBmdW5jID0gZnVuYy5mdW5jCisgICAgcmV0dXJuIGZ1bmMKKworCiBAZGF0YWNsYXNzCiBjbGFzcyBTZWN1cml0eVJlcXVpcmVtZW50OgogICAgIHNlY3VyaXR5X3NjaGVtZTogU2VjdXJpdHlCYXNlCkBAIC03NSwzNyArODgsODIgQEAgZGVmIF91c2VzX3Njb3BlcyhzZWxmKSAtPiBib29sOgogICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgIHJldHVybiBGYWxzZQogCi0gICAgQGNhY2hlZF9wcm9wZXJ0eQotICAgIGRlZiBfdW53cmFwcGVkX2NhbGwoc2VsZikgLT4gQW55OgotICAgICAgICBpZiBzZWxmLmNhbGwgaXMgTm9uZToKLSAgICAgICAgICAgIHJldHVybiBzZWxmLmNhbGwgICMgcHJhZ21hOiBubyBjb3ZlcgotICAgICAgICB1bndyYXBwZWQgPSBpbnNwZWN0LnVud3JhcChzZWxmLmNhbGwpCi0gICAgICAgIGlmIGlzaW5zdGFuY2UodW53cmFwcGVkLCBwYXJ0aWFsKToKLSAgICAgICAgICAgIHVud3JhcHBlZCA9IHVud3JhcHBlZC5mdW5jCi0gICAgICAgIHJldHVybiB1bndyYXBwZWQKLQogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfZ2VuX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Ci0gICAgICAgIGlmIGluc3BlY3QuaXNnZW5lcmF0b3JmdW5jdGlvbihzZWxmLl91bndyYXBwZWRfY2FsbCk6CisgICAgICAgIGlmIHNlbGYuY2FsbCBpcyBOb25lOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlICAjIHByYWdtYTogbm8gY292ZXIKKyAgICAgICAgaWYgaW5zcGVjdC5pc2dlbmVyYXRvcmZ1bmN0aW9uKAorICAgICAgICAgICAgX2ltcGFydGlhbChzZWxmLmNhbGwpCisgICAgICAgICkgb3IgaW5zcGVjdC5pc2dlbmVyYXRvcmZ1bmN0aW9uKF91bndyYXBwZWRfY2FsbChzZWxmLmNhbGwpKToKKyAgICAgICAgICAgIHJldHVybiBUcnVlCisgICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihfaW1wYXJ0aWFsKHNlbGYuY2FsbCksICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKKyAgICAgICAgaWYgZHVuZGVyX2NhbGwgaXMgTm9uZToKKyAgICAgICAgICAgIHJldHVybiBGYWxzZSAgIyBwcmFnbWE6IG5vIGNvdmVyCisgICAgICAgIGlmIGluc3BlY3QuaXNnZW5lcmF0b3JmdW5jdGlvbigKKyAgICAgICAgICAgIF9pbXBhcnRpYWwoZHVuZGVyX2NhbGwpCisgICAgICAgICkgb3IgaW5zcGVjdC5pc2dlbmVyYXRvcmZ1bmN0aW9uKF91bndyYXBwZWRfY2FsbChkdW5kZXJfY2FsbCkpOgogICAgICAgICAgICAgcmV0dXJuIFRydWUKLSAgICAgICAgZHVuZGVyX2NhbGwgPSBnZXRhdHRyKHNlbGYuX3Vud3JhcHBlZF9jYWxsLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0Ci0gICAgICAgIHJldHVybiBpbnNwZWN0LmlzZ2VuZXJhdG9yZnVuY3Rpb24oZHVuZGVyX2NhbGwpCisgICAgICAgIGR1bmRlcl91bndyYXBwZWRfY2FsbCA9IGdldGF0dHIoX3Vud3JhcHBlZF9jYWxsKHNlbGYuY2FsbCksICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKKyAgICAgICAgaWYgZHVuZGVyX3Vud3JhcHBlZF9jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgorICAgICAgICBpZiBpbnNwZWN0LmlzZ2VuZXJhdG9yZnVuY3Rpb24oCisgICAgICAgICAgICBfaW1wYXJ0aWFsKGR1bmRlcl91bndyYXBwZWRfY2FsbCkKKyAgICAgICAgKSBvciBpbnNwZWN0LmlzZ2VuZXJhdG9yZnVuY3Rpb24oX3Vud3JhcHBlZF9jYWxsKGR1bmRlcl91bndyYXBwZWRfY2FsbCkpOgorICAgICAgICAgICAgcmV0dXJuIFRydWUKKyAgICAgICAgcmV0dXJuIEZhbHNlCiAKICAgICBAY2FjaGVkX3Byb3BlcnR5CiAgICAgZGVmIGlzX2FzeW5jX2dlbl9jYWxsYWJsZShzZWxmKSAtPiBib29sOgotICAgICAgICBpZiBpbnNwZWN0LmlzYXN5bmNnZW5mdW5jdGlvbihzZWxmLl91bndyYXBwZWRfY2FsbCk6CisgICAgICAgIGlmIHNlbGYuY2FsbCBpcyBOb25lOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlICAjIHByYWdtYTogbm8gY292ZXIKKyAgICAgICAgaWYgaW5zcGVjdC5pc2FzeW5jZ2VuZnVuY3Rpb24oCisgICAgICAgICAgICBfaW1wYXJ0aWFsKHNlbGYuY2FsbCkKKyAgICAgICAgKSBvciBpbnNwZWN0LmlzYXN5bmNnZW5mdW5jdGlvbihfdW53cmFwcGVkX2NhbGwoc2VsZi5jYWxsKSk6CisgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBkdW5kZXJfY2FsbCA9IGdldGF0dHIoX2ltcGFydGlhbChzZWxmLmNhbGwpLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0CisgICAgICAgIGlmIGR1bmRlcl9jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgorICAgICAgICBpZiBpbnNwZWN0LmlzYXN5bmNnZW5mdW5jdGlvbigKKyAgICAgICAgICAgIF9pbXBhcnRpYWwoZHVuZGVyX2NhbGwpCisgICAgICAgICkgb3IgaW5zcGVjdC5pc2FzeW5jZ2VuZnVuY3Rpb24oX3Vud3JhcHBlZF9jYWxsKGR1bmRlcl9jYWxsKSk6CisgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBkdW5kZXJfdW53cmFwcGVkX2NhbGwgPSBnZXRhdHRyKF91bndyYXBwZWRfY2FsbChzZWxmLmNhbGwpLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0CisgICAgICAgIGlmIGR1bmRlcl91bndyYXBwZWRfY2FsbCBpcyBOb25lOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlICAjIHByYWdtYTogbm8gY292ZXIKKyAgICAgICAgaWYgaW5zcGVjdC5pc2FzeW5jZ2VuZnVuY3Rpb24oCisgICAgICAgICAgICBfaW1wYXJ0aWFsKGR1bmRlcl91bndyYXBwZWRfY2FsbCkKKyAgICAgICAgKSBvciBpbnNwZWN0LmlzYXN5bmNnZW5mdW5jdGlvbihfdW53cmFwcGVkX2NhbGwoZHVuZGVyX3Vud3JhcHBlZF9jYWxsKSk6CiAgICAgICAgICAgICByZXR1cm4gVHJ1ZQotICAgICAgICBkdW5kZXJfY2FsbCA9IGdldGF0dHIoc2VsZi5fdW53cmFwcGVkX2NhbGwsICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKLSAgICAgICAgcmV0dXJuIGluc3BlY3QuaXNhc3luY2dlbmZ1bmN0aW9uKGR1bmRlcl9jYWxsKQorICAgICAgICByZXR1cm4gRmFsc2UKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfY29yb3V0aW5lX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Ci0gICAgICAgIGlmIGluc3BlY3QuaXNyb3V0aW5lKHNlbGYuX3Vud3JhcHBlZF9jYWxsKToKLSAgICAgICAgICAgIHJldHVybiBpc2Nvcm91dGluZWZ1bmN0aW9uKHNlbGYuX3Vud3JhcHBlZF9jYWxsKQotICAgICAgICBpZiBpbnNwZWN0LmlzY2xhc3Moc2VsZi5fdW53cmFwcGVkX2NhbGwpOgotICAgICAgICAgICAgcmV0dXJuIEZhbHNlCi0gICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihzZWxmLl91bndyYXBwZWRfY2FsbCwgIl9fY2FsbF9fIiwgTm9uZSkgICMgbm9xYTogQjAwNAotICAgICAgICByZXR1cm4gaXNjb3JvdXRpbmVmdW5jdGlvbihkdW5kZXJfY2FsbCkKKyAgICAgICAgaWYgc2VsZi5jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgorICAgICAgICBpZiBpbnNwZWN0Lmlzcm91dGluZShfaW1wYXJ0aWFsKHNlbGYuY2FsbCkpIGFuZCBpc2Nvcm91dGluZWZ1bmN0aW9uKAorICAgICAgICAgICAgX2ltcGFydGlhbChzZWxmLmNhbGwpCisgICAgICAgICk6CisgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBpZiBpbnNwZWN0Lmlzcm91dGluZShfdW53cmFwcGVkX2NhbGwoc2VsZi5jYWxsKSkgYW5kIGlzY29yb3V0aW5lZnVuY3Rpb24oCisgICAgICAgICAgICBfdW53cmFwcGVkX2NhbGwoc2VsZi5jYWxsKQorICAgICAgICApOgorICAgICAgICAgICAgcmV0dXJuIFRydWUKKyAgICAgICAgZHVuZGVyX2NhbGwgPSBnZXRhdHRyKF9pbXBhcnRpYWwoc2VsZi5jYWxsKSwgIl9fY2FsbF9fIiwgTm9uZSkgICMgbm9xYTogQjAwNAorICAgICAgICBpZiBkdW5kZXJfY2FsbCBpcyBOb25lOgorICAgICAgICAgICAgcmV0dXJuIEZhbHNlICAjIHByYWdtYTogbm8gY292ZXIKKyAgICAgICAgaWYgaXNjb3JvdXRpbmVmdW5jdGlvbihfaW1wYXJ0aWFsKGR1bmRlcl9jYWxsKSkgb3IgaXNjb3JvdXRpbmVmdW5jdGlvbigKKyAgICAgICAgICAgIF91bndyYXBwZWRfY2FsbChkdW5kZXJfY2FsbCkKKyAgICAgICAgKToKKyAgICAgICAgICAgIHJldHVybiBUcnVlCisgICAgICAgIGR1bmRlcl91bndyYXBwZWRfY2FsbCA9IGdldGF0dHIoX3Vud3JhcHBlZF9jYWxsKHNlbGYuY2FsbCksICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKKyAgICAgICAgaWYgZHVuZGVyX3Vud3JhcHBlZF9jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gRmFsc2UgICMgcHJhZ21hOiBubyBjb3ZlcgorICAgICAgICBpZiBpc2Nvcm91dGluZWZ1bmN0aW9uKAorICAgICAgICAgICAgX2ltcGFydGlhbChkdW5kZXJfdW53cmFwcGVkX2NhbGwpCisgICAgICAgICkgb3IgaXNjb3JvdXRpbmVmdW5jdGlvbihfdW53cmFwcGVkX2NhbGwoZHVuZGVyX3Vud3JhcHBlZF9jYWxsKSk6CisgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICAjIGlmIGluc3BlY3QuaXNjbGFzcyhzZWxmLmNhbGwpOiBGYWxzZSwgY292ZXJlZCBieSBkZWZhdWx0IHJldHVybgorICAgICAgICByZXR1cm4gRmFsc2UKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgY29tcHV0ZWRfc2NvcGUoc2VsZikgLT4gVW5pb25bc3RyLCBOb25lXToKLS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTU0OCwxMCArNTQ4LDEwIEBAIGFzeW5jIGRlZiBfc29sdmVfZ2VuZXJhdG9yKAogICAgICosIGRlcGVuZGFudDogRGVwZW5kYW50LCBzdGFjazogQXN5bmNFeGl0U3RhY2ssIHN1Yl92YWx1ZXM6IERpY3Rbc3RyLCBBbnldCiApIC0+IEFueToKICAgICBhc3NlcnQgZGVwZW5kYW50LmNhbGwKLSAgICBpZiBkZXBlbmRhbnQuaXNfZ2VuX2NhbGxhYmxlOgotICAgICAgICBjbSA9IGNvbnRleHRtYW5hZ2VyX2luX3RocmVhZHBvb2woY29udGV4dG1hbmFnZXIoZGVwZW5kYW50LmNhbGwpKCoqc3ViX3ZhbHVlcykpCi0gICAgZWxpZiBkZXBlbmRhbnQuaXNfYXN5bmNfZ2VuX2NhbGxhYmxlOgorICAgIGlmIGRlcGVuZGFudC5pc19hc3luY19nZW5fY2FsbGFibGU6CiAgICAgICAgIGNtID0gYXN5bmNjb250ZXh0bWFuYWdlcihkZXBlbmRhbnQuY2FsbCkoKipzdWJfdmFsdWVzKQorICAgIGVsaWYgZGVwZW5kYW50LmlzX2dlbl9jYWxsYWJsZToKKyAgICAgICAgY20gPSBjb250ZXh0bWFuYWdlcl9pbl90aHJlYWRwb29sKGNvbnRleHRtYW5hZ2VyKGRlcGVuZGFudC5jYWxsKSgqKnN1Yl92YWx1ZXMpKQogICAgIHJldHVybiBhd2FpdCBzdGFjay5lbnRlcl9hc3luY19jb250ZXh0KGNtKQogCiAK", "fastapi_14455": "LS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtNzksNyArNzksOCBAQCBkZWYgZ2V0X29wZW5hcGlfc2VjdXJpdHlfZGVmaW5pdGlvbnMoCiAgICAgZmxhdF9kZXBlbmRhbnQ6IERlcGVuZGFudCwKICkgLT4gVHVwbGVbRGljdFtzdHIsIEFueV0sIExpc3RbRGljdFtzdHIsIEFueV1dXToKICAgICBzZWN1cml0eV9kZWZpbml0aW9ucyA9IHt9Ci0gICAgb3BlcmF0aW9uX3NlY3VyaXR5ID0gW10KKyAgICAjIFVzZSBhIGRpY3QgdG8gbWVyZ2Ugc2NvcGVzIGZvciBzYW1lIHNlY3VyaXR5IHNjaGVtZQorICAgIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0OiBEaWN0W3N0ciwgTGlzdFtzdHJdXSA9IHt9CiAgICAgZm9yIHNlY3VyaXR5X3JlcXVpcmVtZW50IGluIGZsYXRfZGVwZW5kYW50LnNlY3VyaXR5X3JlcXVpcmVtZW50czoKICAgICAgICAgc2VjdXJpdHlfZGVmaW5pdGlvbiA9IGpzb25hYmxlX2VuY29kZXIoCiAgICAgICAgICAgICBzZWN1cml0eV9yZXF1aXJlbWVudC5zZWN1cml0eV9zY2hlbWUubW9kZWwsCkBAIC04OCw3ICs4OSwxNSBAQCBkZWYgZ2V0X29wZW5hcGlfc2VjdXJpdHlfZGVmaW5pdGlvbnMoCiAgICAgICAgICkKICAgICAgICAgc2VjdXJpdHlfbmFtZSA9IHNlY3VyaXR5X3JlcXVpcmVtZW50LnNlY3VyaXR5X3NjaGVtZS5zY2hlbWVfbmFtZQogICAgICAgICBzZWN1cml0eV9kZWZpbml0aW9uc1tzZWN1cml0eV9uYW1lXSA9IHNlY3VyaXR5X2RlZmluaXRpb24KLSAgICAgICAgb3BlcmF0aW9uX3NlY3VyaXR5LmFwcGVuZCh7c2VjdXJpdHlfbmFtZTogc2VjdXJpdHlfcmVxdWlyZW1lbnQuc2NvcGVzfSkKKyAgICAgICAgIyBNZXJnZSBzY29wZXMgZm9yIHRoZSBzYW1lIHNlY3VyaXR5IHNjaGVtZQorICAgICAgICBpZiBzZWN1cml0eV9uYW1lIG5vdCBpbiBvcGVyYXRpb25fc2VjdXJpdHlfZGljdDoKKyAgICAgICAgICAgIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0W3NlY3VyaXR5X25hbWVdID0gW10KKyAgICAgICAgZm9yIHNjb3BlIGluIHNlY3VyaXR5X3JlcXVpcmVtZW50LnNjb3BlcyBvciBbXToKKyAgICAgICAgICAgIGlmIHNjb3BlIG5vdCBpbiBvcGVyYXRpb25fc2VjdXJpdHlfZGljdFtzZWN1cml0eV9uYW1lXToKKyAgICAgICAgICAgICAgICBvcGVyYXRpb25fc2VjdXJpdHlfZGljdFtzZWN1cml0eV9uYW1lXS5hcHBlbmQoc2NvcGUpCisgICAgb3BlcmF0aW9uX3NlY3VyaXR5ID0gWworICAgICAgICB7bmFtZTogc2NvcGVzfSBmb3IgbmFtZSwgc2NvcGVzIGluIG9wZXJhdGlvbl9zZWN1cml0eV9kaWN0Lml0ZW1zKCkKKyAgICBdCiAgICAgcmV0dXJuIHNlY3VyaXR5X2RlZmluaXRpb25zLCBvcGVyYXRpb25fc2VjdXJpdHkKIAogCg==", "fastapi_9753": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMSw3ICsxLDcgQEAKIGltcG9ydCBpbnNwZWN0CiBpbXBvcnQgc3lzCiBmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCi1mcm9tIGZ1bmN0b29scyBpbXBvcnQgY2FjaGVkX3Byb3BlcnR5Citmcm9tIGZ1bmN0b29scyBpbXBvcnQgY2FjaGVkX3Byb3BlcnR5LCBwYXJ0aWFsCiBmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZSwgTGlzdCwgT3B0aW9uYWwsIFNlcXVlbmNlLCBVbmlvbgogCiBmcm9tIGZhc3RhcGkuX2NvbXBhdCBpbXBvcnQgTW9kZWxGaWVsZApAQCAtNzksNyArNzksMTAgQEAgZGVmIF91c2VzX3Njb3BlcyhzZWxmKSAtPiBib29sOgogICAgIGRlZiBfdW53cmFwcGVkX2NhbGwoc2VsZikgLT4gQW55OgogICAgICAgICBpZiBzZWxmLmNhbGwgaXMgTm9uZToKICAgICAgICAgICAgIHJldHVybiBzZWxmLmNhbGwgICMgcHJhZ21hOiBubyBjb3ZlcgotICAgICAgICByZXR1cm4gaW5zcGVjdC51bndyYXAoc2VsZi5jYWxsKQorICAgICAgICB1bndyYXBwZWQgPSBpbnNwZWN0LnVud3JhcChzZWxmLmNhbGwpCisgICAgICAgIGlmIGlzaW5zdGFuY2UodW53cmFwcGVkLCBwYXJ0aWFsKToKKyAgICAgICAgICAgIHVud3JhcHBlZCA9IHVud3JhcHBlZC5mdW5jCisgICAgICAgIHJldHVybiB1bndyYXBwZWQKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfZ2VuX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Cg==", "fastapi_14430": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMTcsNyArMTcsNyBAQAogCiBmcm9tIGZhc3RhcGkuX2NvbXBhdCBpbXBvcnQgbWF5X3YxLCBzaGFyZWQKIGZyb20gZmFzdGFwaS5vcGVuYXBpLmNvbnN0YW50cyBpbXBvcnQgUkVGX1RFTVBMQVRFCi1mcm9tIGZhc3RhcGkudHlwZXMgaW1wb3J0IEluY0V4LCBNb2RlbE5hbWVNYXAKK2Zyb20gZmFzdGFwaS50eXBlcyBpbXBvcnQgSW5jRXgsIE1vZGVsTmFtZU1hcCwgVW5pb25UeXBlCiBmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwsIFR5cGVBZGFwdGVyLCBjcmVhdGVfbW9kZWwKIGZyb20gcHlkYW50aWMgaW1wb3J0IFB5ZGFudGljU2NoZW1hR2VuZXJhdGlvbkVycm9yIGFzIFB5ZGFudGljU2NoZW1hR2VuZXJhdGlvbkVycm9yCiBmcm9tIHB5ZGFudGljIGltcG9ydCBQeWRhbnRpY1VuZGVmaW5lZEFubm90YXRpb24gYXMgUHlkYW50aWNVbmRlZmluZWRBbm5vdGF0aW9uCkBAIC0zODEsNyArMzgxLDcgQEAgZGVmIGNvcHlfZmllbGRfaW5mbygqLCBmaWVsZF9pbmZvOiBGaWVsZEluZm8sIGFubm90YXRpb246IEFueSkgLT4gRmllbGRJbmZvOgogCiBkZWYgc2VyaWFsaXplX3NlcXVlbmNlX3ZhbHVlKCosIGZpZWxkOiBNb2RlbEZpZWxkLCB2YWx1ZTogQW55KSAtPiBTZXF1ZW5jZVtBbnldOgogICAgIG9yaWdpbl90eXBlID0gZ2V0X29yaWdpbihmaWVsZC5maWVsZF9pbmZvLmFubm90YXRpb24pIG9yIGZpZWxkLmZpZWxkX2luZm8uYW5ub3RhdGlvbgotICAgIGlmIG9yaWdpbl90eXBlIGlzIFVuaW9uOiAgIyBIYW5kbGUgb3B0aW9uYWwgc2VxdWVuY2VzCisgICAgaWYgb3JpZ2luX3R5cGUgaXMgVW5pb24gb3Igb3JpZ2luX3R5cGUgaXMgVW5pb25UeXBlOiAgIyBIYW5kbGUgb3B0aW9uYWwgc2VxdWVuY2VzCiAgICAgICAgIHVuaW9uX2FyZ3MgPSBnZXRfYXJncyhmaWVsZC5maWVsZF9pbmZvLmFubm90YXRpb24pCiAgICAgICAgIGZvciB1bmlvbl9hcmcgaW4gdW5pb25fYXJnczoKICAgICAgICAgICAgIGlmIHVuaW9uX2FyZyBpcyB0eXBlKE5vbmUpOgo=", "fastapi_5077": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTE5Miw3ICsxOTIsOCBAQCBkZWYgZ2V0X2ZsYXRfcGFyYW1zKGRlcGVuZGFudDogRGVwZW5kYW50KSAtPiBMaXN0W01vZGVsRmllbGRdOgogCiBkZWYgZ2V0X3R5cGVkX3NpZ25hdHVyZShjYWxsOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IGluc3BlY3QuU2lnbmF0dXJlOgogICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKGNhbGwpCi0gICAgZ2xvYmFsbnMgPSBnZXRhdHRyKGNhbGwsICJfX2dsb2JhbHNfXyIsIHt9KQorICAgIHVud3JhcHBlZCA9IGluc3BlY3QudW53cmFwKGNhbGwpCisgICAgZ2xvYmFsbnMgPSBnZXRhdHRyKHVud3JhcHBlZCwgIl9fZ2xvYmFsc19fIiwge30pCiAgICAgdHlwZWRfcGFyYW1zID0gWwogICAgICAgICBpbnNwZWN0LlBhcmFtZXRlcigKICAgICAgICAgICAgIG5hbWU9cGFyYW0ubmFtZSwKQEAgLTIxNywxMiArMjE4LDEzIEBAIGRlZiBnZXRfdHlwZWRfYW5ub3RhdGlvbihhbm5vdGF0aW9uOiBBbnksIGdsb2JhbG5zOiBEaWN0W3N0ciwgQW55XSkgLT4gQW55OgogCiBkZWYgZ2V0X3R5cGVkX3JldHVybl9hbm5vdGF0aW9uKGNhbGw6IENhbGxhYmxlWy4uLiwgQW55XSkgLT4gQW55OgogICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKGNhbGwpCisgICAgdW53cmFwcGVkID0gaW5zcGVjdC51bndyYXAoY2FsbCkKICAgICBhbm5vdGF0aW9uID0gc2lnbmF0dXJlLnJldHVybl9hbm5vdGF0aW9uCiAKICAgICBpZiBhbm5vdGF0aW9uIGlzIGluc3BlY3QuU2lnbmF0dXJlLmVtcHR5OgogICAgICAgICByZXR1cm4gTm9uZQogCi0gICAgZ2xvYmFsbnMgPSBnZXRhdHRyKGNhbGwsICJfX2dsb2JhbHNfXyIsIHt9KQorICAgIGdsb2JhbG5zID0gZ2V0YXR0cih1bndyYXBwZWQsICJfX2dsb2JhbHNfXyIsIHt9KQogICAgIHJldHVybiBnZXRfdHlwZWRfYW5ub3RhdGlvbihhbm5vdGF0aW9uLCBnbG9iYWxucykKIAogCg==", "fastapi_9555": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtNzUsMjcgKzc1LDMzIEBAIGRlZiBfdXNlc19zY29wZXMoc2VsZikgLT4gYm9vbDoKICAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgICByZXR1cm4gRmFsc2UKIAorICAgIEBjYWNoZWRfcHJvcGVydHkKKyAgICBkZWYgX3Vud3JhcHBlZF9jYWxsKHNlbGYpIC0+IEFueToKKyAgICAgICAgaWYgc2VsZi5jYWxsIGlzIE5vbmU6CisgICAgICAgICAgICByZXR1cm4gc2VsZi5jYWxsICAjIHByYWdtYTogbm8gY292ZXIKKyAgICAgICAgcmV0dXJuIGluc3BlY3QudW53cmFwKHNlbGYuY2FsbCkKKwogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfZ2VuX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Ci0gICAgICAgIGlmIGluc3BlY3QuaXNnZW5lcmF0b3JmdW5jdGlvbihzZWxmLmNhbGwpOgorICAgICAgICBpZiBpbnNwZWN0LmlzZ2VuZXJhdG9yZnVuY3Rpb24oc2VsZi5fdW53cmFwcGVkX2NhbGwpOgogICAgICAgICAgICAgcmV0dXJuIFRydWUKLSAgICAgICAgZHVuZGVyX2NhbGwgPSBnZXRhdHRyKHNlbGYuY2FsbCwgIl9fY2FsbF9fIiwgTm9uZSkgICMgbm9xYTogQjAwNAorICAgICAgICBkdW5kZXJfY2FsbCA9IGdldGF0dHIoc2VsZi5fdW53cmFwcGVkX2NhbGwsICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKICAgICAgICAgcmV0dXJuIGluc3BlY3QuaXNnZW5lcmF0b3JmdW5jdGlvbihkdW5kZXJfY2FsbCkKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfYXN5bmNfZ2VuX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Ci0gICAgICAgIGlmIGluc3BlY3QuaXNhc3luY2dlbmZ1bmN0aW9uKHNlbGYuY2FsbCk6CisgICAgICAgIGlmIGluc3BlY3QuaXNhc3luY2dlbmZ1bmN0aW9uKHNlbGYuX3Vud3JhcHBlZF9jYWxsKToKICAgICAgICAgICAgIHJldHVybiBUcnVlCi0gICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihzZWxmLmNhbGwsICJfX2NhbGxfXyIsIE5vbmUpICAjIG5vcWE6IEIwMDQKKyAgICAgICAgZHVuZGVyX2NhbGwgPSBnZXRhdHRyKHNlbGYuX3Vud3JhcHBlZF9jYWxsLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0CiAgICAgICAgIHJldHVybiBpbnNwZWN0LmlzYXN5bmNnZW5mdW5jdGlvbihkdW5kZXJfY2FsbCkKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfY29yb3V0aW5lX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Ci0gICAgICAgIGlmIGluc3BlY3QuaXNyb3V0aW5lKHNlbGYuY2FsbCk6Ci0gICAgICAgICAgICByZXR1cm4gaXNjb3JvdXRpbmVmdW5jdGlvbihzZWxmLmNhbGwpCi0gICAgICAgIGlmIGluc3BlY3QuaXNjbGFzcyhzZWxmLmNhbGwpOgorICAgICAgICBpZiBpbnNwZWN0Lmlzcm91dGluZShzZWxmLl91bndyYXBwZWRfY2FsbCk6CisgICAgICAgICAgICByZXR1cm4gaXNjb3JvdXRpbmVmdW5jdGlvbihzZWxmLl91bndyYXBwZWRfY2FsbCkKKyAgICAgICAgaWYgaW5zcGVjdC5pc2NsYXNzKHNlbGYuX3Vud3JhcHBlZF9jYWxsKToKICAgICAgICAgICAgIHJldHVybiBGYWxzZQotICAgICAgICBkdW5kZXJfY2FsbCA9IGdldGF0dHIoc2VsZi5jYWxsLCAiX19jYWxsX18iLCBOb25lKSAgIyBub3FhOiBCMDA0CisgICAgICAgIGR1bmRlcl9jYWxsID0gZ2V0YXR0cihzZWxmLl91bndyYXBwZWRfY2FsbCwgIl9fY2FsbF9fIiwgTm9uZSkgICMgbm9xYTogQjAwNAogICAgICAgICByZXR1cm4gaXNjb3JvdXRpbmVmdW5jdGlvbihkdW5kZXJfY2FsbCkKIAogICAgIEBjYWNoZWRfcHJvcGVydHkK", "fastapi_14361": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMzA0LDcgKzMwNCw3IEBAIGRlZiBfcmVtYXBfZGVmaW5pdGlvbnNfYW5kX2ZpZWxkX21hcHBpbmdzKAogICAgIG9sZF9uYW1lX3RvX25ld19uYW1lX21hcCA9IHt9CiAgICAgZm9yIGZpZWxkX2tleSwgc2NoZW1hIGluIGZpZWxkX21hcHBpbmcuaXRlbXMoKToKICAgICAgICAgbW9kZWwgPSBmaWVsZF9rZXlbMF0udHlwZV8KLSAgICAgICAgaWYgbW9kZWwgbm90IGluIG1vZGVsX25hbWVfbWFwOgorICAgICAgICBpZiBtb2RlbCBub3QgaW4gbW9kZWxfbmFtZV9tYXAgb3IgIiRyZWYiIG5vdCBpbiBzY2hlbWE6CiAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICBuZXdfbmFtZSA9IG1vZGVsX25hbWVfbWFwW21vZGVsXQogICAgICAgICBvbGRfbmFtZSA9IHNjaGVtYVsiJHJlZiJdLnNwbGl0KCIvIilbLTFdCg==", "fastapi_13207": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMTgwLDggKzE4MCwxMyBAQCBkZWYgZ2V0X3NjaGVtYV9mcm9tX21vZGVsX2ZpZWxkKAogICAgIF0sCiAgICAgc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXM6IGJvb2wgPSBUcnVlLAogKSAtPiBEaWN0W3N0ciwgQW55XToKKyAgICBjb21wdXRlZF9maWVsZHMgPSBmaWVsZC5fdHlwZV9hZGFwdGVyLmNvcmVfc2NoZW1hLmdldCgic2NoZW1hIiwge30pLmdldCgKKyAgICAgICAgImNvbXB1dGVkX2ZpZWxkcyIsIFtdCisgICAgKQogICAgIG92ZXJyaWRlX21vZGU6IFVuaW9uW0xpdGVyYWxbInZhbGlkYXRpb24iXSwgTm9uZV0gPSAoCi0gICAgICAgIE5vbmUgaWYgc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXMgZWxzZSAidmFsaWRhdGlvbiIKKyAgICAgICAgTm9uZQorICAgICAgICBpZiAoc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXMgb3IgbGVuKGNvbXB1dGVkX2ZpZWxkcykgPiAwKQorICAgICAgICBlbHNlICJ2YWxpZGF0aW9uIgogICAgICkKICAgICAjIFRoaXMgZXhwZWN0cyB0aGF0IEdlbmVyYXRlSnNvblNjaGVtYSB3YXMgYWxyZWFkeSB1c2VkIHRvIGdlbmVyYXRlIHRoZSBkZWZpbml0aW9ucwogICAgIGpzb25fc2NoZW1hID0gZmllbGRfbWFwcGluZ1soZmllbGQsIG92ZXJyaWRlX21vZGUgb3IgZmllbGQubW9kZSldCkBAIC0yMDMsOSArMjA4LDE0IEBAIGRlZiBnZXRfZGVmaW5pdGlvbnMoCiAgICAgRGljdFtUdXBsZVtNb2RlbEZpZWxkLCBMaXRlcmFsWyJ2YWxpZGF0aW9uIiwgInNlcmlhbGl6YXRpb24iXV0sIEpzb25TY2hlbWFWYWx1ZV0sCiAgICAgRGljdFtzdHIsIERpY3Rbc3RyLCBBbnldXSwKIF06CisgICAgaGFzX2NvbXB1dGVkX2ZpZWxkczogYm9vbCA9IGFueSgKKyAgICAgICAgZmllbGQuX3R5cGVfYWRhcHRlci5jb3JlX3NjaGVtYS5nZXQoInNjaGVtYSIsIHt9KS5nZXQoImNvbXB1dGVkX2ZpZWxkcyIsIFtdKQorICAgICAgICBmb3IgZmllbGQgaW4gZmllbGRzCisgICAgKQorCiAgICAgc2NoZW1hX2dlbmVyYXRvciA9IEdlbmVyYXRlSnNvblNjaGVtYShyZWZfdGVtcGxhdGU9UkVGX1RFTVBMQVRFKQogICAgIG92ZXJyaWRlX21vZGU6IFVuaW9uW0xpdGVyYWxbInZhbGlkYXRpb24iXSwgTm9uZV0gPSAoCi0gICAgICAgIE5vbmUgaWYgc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXMgZWxzZSAidmFsaWRhdGlvbiIKKyAgICAgICAgTm9uZSBpZiAoc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXMgb3IgaGFzX2NvbXB1dGVkX2ZpZWxkcykgZWxzZSAidmFsaWRhdGlvbiIKICAgICApCiAgICAgdmFsaWRhdGlvbl9maWVsZHMgPSBbZmllbGQgZm9yIGZpZWxkIGluIGZpZWxkcyBpZiBmaWVsZC5tb2RlID09ICJ2YWxpZGF0aW9uIl0KICAgICBzZXJpYWxpemF0aW9uX2ZpZWxkcyA9IFtmaWVsZCBmb3IgZmllbGQgaW4gZmllbGRzIGlmIGZpZWxkLm1vZGUgPT0gInNlcmlhbGl6YXRpb24iXQo=", "fastapi_14360": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTc4Nyw5ICs3ODcsOCBAQCBkZWYgcmVxdWVzdF9wYXJhbXNfdG9fYXJncygKICAgICAgICAgICAgICAgICApCiAgICAgICAgIHZhbHVlID0gX2dldF9tdWx0aWRpY3RfdmFsdWUoZmllbGQsIHJlY2VpdmVkX3BhcmFtcywgYWxpYXM9YWxpYXMpCiAgICAgICAgIGlmIHZhbHVlIGlzIG5vdCBOb25lOgotICAgICAgICAgICAgcGFyYW1zX3RvX3Byb2Nlc3NbZmllbGQubmFtZV0gPSB2YWx1ZQorICAgICAgICAgICAgcGFyYW1zX3RvX3Byb2Nlc3NbZmllbGQuYWxpYXNdID0gdmFsdWUKICAgICAgICAgcHJvY2Vzc2VkX2tleXMuYWRkKGFsaWFzIG9yIGZpZWxkLmFsaWFzKQotICAgICAgICBwcm9jZXNzZWRfa2V5cy5hZGQoZmllbGQubmFtZSkKIAogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfcGFyYW1zLmtleXMoKToKICAgICAgICAgaWYga2V5IG5vdCBpbiBwcm9jZXNzZWRfa2V5czoK", "fastapi_14356": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTc5MSw5ICs3OTEsMTYgQEAgZGVmIHJlcXVlc3RfcGFyYW1zX3RvX2FyZ3MoCiAgICAgICAgIHByb2Nlc3NlZF9rZXlzLmFkZChhbGlhcyBvciBmaWVsZC5hbGlhcykKICAgICAgICAgcHJvY2Vzc2VkX2tleXMuYWRkKGZpZWxkLm5hbWUpCiAKLSAgICBmb3Iga2V5LCB2YWx1ZSBpbiByZWNlaXZlZF9wYXJhbXMuaXRlbXMoKToKKyAgICBmb3Iga2V5IGluIHJlY2VpdmVkX3BhcmFtcy5rZXlzKCk6CiAgICAgICAgIGlmIGtleSBub3QgaW4gcHJvY2Vzc2VkX2tleXM6Ci0gICAgICAgICAgICBwYXJhbXNfdG9fcHJvY2Vzc1trZXldID0gdmFsdWUKKyAgICAgICAgICAgIGlmIGhhc2F0dHIocmVjZWl2ZWRfcGFyYW1zLCAiZ2V0bGlzdCIpOgorICAgICAgICAgICAgICAgIHZhbHVlID0gcmVjZWl2ZWRfcGFyYW1zLmdldGxpc3Qoa2V5KQorICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIGxpc3QpIGFuZCAobGVuKHZhbHVlKSA9PSAxKToKKyAgICAgICAgICAgICAgICAgICAgcGFyYW1zX3RvX3Byb2Nlc3Nba2V5XSA9IHZhbHVlWzBdCisgICAgICAgICAgICAgICAgZWxzZToKKyAgICAgICAgICAgICAgICAgICAgcGFyYW1zX3RvX3Byb2Nlc3Nba2V5XSA9IHZhbHVlCisgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgIHBhcmFtc190b19wcm9jZXNzW2tleV0gPSByZWNlaXZlZF9wYXJhbXMuZ2V0KGtleSkKIAogICAgIGlmIHNpbmdsZV9ub3RfZW1iZWRkZWRfZmllbGQ6CiAgICAgICAgIGZpZWxkX2luZm8gPSBmaXJzdF9maWVsZC5maWVsZF9pbmZvCg==", "fastapi_5624": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTI3OCw3ICsyNzgsOSBAQCBkZWYgZ2V0X2RlcGVuZGFudCgKICAgICAgICAgICAgIHVzZV9zZWN1cml0eV9zY29wZXMgPSBzZWN1cml0eV9zY29wZXMgb3IgW10KICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UocGFyYW1fZGV0YWlscy5kZXBlbmRzLCBwYXJhbXMuU2VjdXJpdHkpOgogICAgICAgICAgICAgICAgIGlmIHBhcmFtX2RldGFpbHMuZGVwZW5kcy5zY29wZXM6Ci0gICAgICAgICAgICAgICAgICAgIHVzZV9zZWN1cml0eV9zY29wZXMuZXh0ZW5kKHBhcmFtX2RldGFpbHMuZGVwZW5kcy5zY29wZXMpCisgICAgICAgICAgICAgICAgICAgIHVzZV9zZWN1cml0eV9zY29wZXMgPSB1c2Vfc2VjdXJpdHlfc2NvcGVzICsgbGlzdCgKKyAgICAgICAgICAgICAgICAgICAgICAgIHBhcmFtX2RldGFpbHMuZGVwZW5kcy5zY29wZXMKKyAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgc3ViX2RlcGVuZGFudCA9IGdldF9kZXBlbmRhbnQoCiAgICAgICAgICAgICAgICAgcGF0aD1wYXRoLAogICAgICAgICAgICAgICAgIGNhbGw9cGFyYW1fZGV0YWlscy5kZXBlbmRzLmRlcGVuZGVuY3ksCg==", "fastapi_14419": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMzgsMTkgKzM4LDQzIEBAIGNsYXNzIERlcGVuZGFudDoKICAgICByZXNwb25zZV9wYXJhbV9uYW1lOiBPcHRpb25hbFtzdHJdID0gTm9uZQogICAgIGJhY2tncm91bmRfdGFza3NfcGFyYW1fbmFtZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgICBzZWN1cml0eV9zY29wZXNfcGFyYW1fbmFtZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUKLSAgICBzZWN1cml0eV9zY29wZXM6IE9wdGlvbmFsW0xpc3Rbc3RyXV0gPSBOb25lCisgICAgb3duX29hdXRoX3Njb3BlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUKKyAgICBwYXJlbnRfb2F1dGhfc2NvcGVzOiBPcHRpb25hbFtMaXN0W3N0cl1dID0gTm9uZQogICAgIHVzZV9jYWNoZTogYm9vbCA9IFRydWUKICAgICBwYXRoOiBPcHRpb25hbFtzdHJdID0gTm9uZQogICAgIHNjb3BlOiBVbmlvbltMaXRlcmFsWyJmdW5jdGlvbiIsICJyZXF1ZXN0Il0sIE5vbmVdID0gTm9uZQogCisgICAgQGNhY2hlZF9wcm9wZXJ0eQorICAgIGRlZiBvYXV0aF9zY29wZXMoc2VsZikgLT4gTGlzdFtzdHJdOgorICAgICAgICBzY29wZXMgPSBzZWxmLnBhcmVudF9vYXV0aF9zY29wZXMuY29weSgpIGlmIHNlbGYucGFyZW50X29hdXRoX3Njb3BlcyBlbHNlIFtdCisgICAgICAgICMgVGhpcyBkb2Vzbid0IHVzZSBhIHNldCB0byBwcmVzZXJ2ZSBvcmRlciwganVzdCBpbiBjYXNlCisgICAgICAgIGZvciBzY29wZSBpbiBzZWxmLm93bl9vYXV0aF9zY29wZXMgb3IgW106CisgICAgICAgICAgICBpZiBzY29wZSBub3QgaW4gc2NvcGVzOgorICAgICAgICAgICAgICAgIHNjb3Blcy5hcHBlbmQoc2NvcGUpCisgICAgICAgIHJldHVybiBzY29wZXMKKwogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgY2FjaGVfa2V5KHNlbGYpIC0+IERlcGVuZGVuY3lDYWNoZUtleToKKyAgICAgICAgc2NvcGVzX2Zvcl9jYWNoZSA9ICgKKyAgICAgICAgICAgIHR1cGxlKHNvcnRlZChzZXQoc2VsZi5vYXV0aF9zY29wZXMgb3IgW10pKSkgaWYgc2VsZi5fdXNlc19zY29wZXMgZWxzZSAoKQorICAgICAgICApCiAgICAgICAgIHJldHVybiAoCiAgICAgICAgICAgICBzZWxmLmNhbGwsCi0gICAgICAgICAgICB0dXBsZShzb3J0ZWQoc2V0KHNlbGYuc2VjdXJpdHlfc2NvcGVzIG9yIFtdKSkpLAorICAgICAgICAgICAgc2NvcGVzX2Zvcl9jYWNoZSwKICAgICAgICAgICAgIHNlbGYuY29tcHV0ZWRfc2NvcGUgb3IgIiIsCiAgICAgICAgICkKIAorICAgIEBjYWNoZWRfcHJvcGVydHkKKyAgICBkZWYgX3VzZXNfc2NvcGVzKHNlbGYpIC0+IGJvb2w6CisgICAgICAgIGlmIHNlbGYub3duX29hdXRoX3Njb3BlczoKKyAgICAgICAgICAgIHJldHVybiBUcnVlCisgICAgICAgIGlmIHNlbGYuc2VjdXJpdHlfc2NvcGVzX3BhcmFtX25hbWUgaXMgbm90IE5vbmU6CisgICAgICAgICAgICByZXR1cm4gVHJ1ZQorICAgICAgICBmb3Igc3ViX2RlcCBpbiBzZWxmLmRlcGVuZGVuY2llczoKKyAgICAgICAgICAgIGlmIHN1Yl9kZXAuX3VzZXNfc2NvcGVzOgorICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCisgICAgICAgIHJldHVybiBGYWxzZQorCiAgICAgQGNhY2hlZF9wcm9wZXJ0eQogICAgIGRlZiBpc19nZW5fY2FsbGFibGUoc2VsZikgLT4gYm9vbDoKICAgICAgICAgaWYgaW5zcGVjdC5pc2dlbmVyYXRvcmZ1bmN0aW9uKHNlbGYuY2FsbCk6Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC01OCw4ICs1OCw3IEBACiBmcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgRGVwZW5kZW5jeVNjb3BlRXJyb3IKIGZyb20gZmFzdGFwaS5sb2dnZXIgaW1wb3J0IGxvZ2dlcgogZnJvbSBmYXN0YXBpLnNlY3VyaXR5LmJhc2UgaW1wb3J0IFNlY3VyaXR5QmFzZQotZnJvbSBmYXN0YXBpLnNlY3VyaXR5Lm9hdXRoMiBpbXBvcnQgT0F1dGgyLCBTZWN1cml0eVNjb3BlcwotZnJvbSBmYXN0YXBpLnNlY3VyaXR5Lm9wZW5faWRfY29ubmVjdF91cmwgaW1wb3J0IE9wZW5JZENvbm5lY3QKK2Zyb20gZmFzdGFwaS5zZWN1cml0eS5vYXV0aDIgaW1wb3J0IFNlY3VyaXR5U2NvcGVzCiBmcm9tIGZhc3RhcGkudHlwZXMgaW1wb3J0IERlcGVuZGVuY3lDYWNoZUtleQogZnJvbSBmYXN0YXBpLnV0aWxzIGltcG9ydCBjcmVhdGVfbW9kZWxfZmllbGQsIGdldF9wYXRoX3BhcmFtX25hbWVzCiBmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwKQEAgLTEyNiwxNCArMTI1LDE0IEBAIGRlZiBnZXRfcGFyYW1ldGVybGVzc19zdWJfZGVwZW5kYW50KCosIGRlcGVuZHM6IHBhcmFtcy5EZXBlbmRzLCBwYXRoOiBzdHIpIC0+IERlCiAgICAgYXNzZXJ0IGNhbGxhYmxlKGRlcGVuZHMuZGVwZW5kZW5jeSksICgKICAgICAgICAgIkEgcGFyYW1ldGVyLWxlc3MgZGVwZW5kZW5jeSBtdXN0IGhhdmUgYSBjYWxsYWJsZSBkZXBlbmRlbmN5IgogICAgICkKLSAgICB1c2Vfc2VjdXJpdHlfc2NvcGVzOiBMaXN0W3N0cl0gPSBbXQorICAgIG93bl9vYXV0aF9zY29wZXM6IExpc3Rbc3RyXSA9IFtdCiAgICAgaWYgaXNpbnN0YW5jZShkZXBlbmRzLCBwYXJhbXMuU2VjdXJpdHkpIGFuZCBkZXBlbmRzLnNjb3BlczoKLSAgICAgICAgdXNlX3NlY3VyaXR5X3Njb3Blcy5leHRlbmQoZGVwZW5kcy5zY29wZXMpCisgICAgICAgIG93bl9vYXV0aF9zY29wZXMuZXh0ZW5kKGRlcGVuZHMuc2NvcGVzKQogICAgIHJldHVybiBnZXRfZGVwZW5kYW50KAogICAgICAgICBwYXRoPXBhdGgsCiAgICAgICAgIGNhbGw9ZGVwZW5kcy5kZXBlbmRlbmN5LAogICAgICAgICBzY29wZT1kZXBlbmRzLnNjb3BlLAotICAgICAgICBzZWN1cml0eV9zY29wZXM9dXNlX3NlY3VyaXR5X3Njb3BlcywKKyAgICAgICAgb3duX29hdXRoX3Njb3Blcz1vd25fb2F1dGhfc2NvcGVzLAogICAgICkKIAogCkBAIC0yMzIsMjcgKzIzMSwyNyBAQCBkZWYgZ2V0X2RlcGVuZGFudCgKICAgICBwYXRoOiBzdHIsCiAgICAgY2FsbDogQ2FsbGFibGVbLi4uLCBBbnldLAogICAgIG5hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAotICAgIHNlY3VyaXR5X3Njb3BlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUsCisgICAgb3duX29hdXRoX3Njb3BlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUsCisgICAgcGFyZW50X29hdXRoX3Njb3BlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUsCiAgICAgdXNlX2NhY2hlOiBib29sID0gVHJ1ZSwKICAgICBzY29wZTogVW5pb25bTGl0ZXJhbFsiZnVuY3Rpb24iLCAicmVxdWVzdCJdLCBOb25lXSA9IE5vbmUsCiApIC0+IERlcGVuZGFudDoKICAgICBkZXBlbmRhbnQgPSBEZXBlbmRhbnQoCiAgICAgICAgIGNhbGw9Y2FsbCwKICAgICAgICAgbmFtZT1uYW1lLAogICAgICAgICBwYXRoPXBhdGgsCi0gICAgICAgIHNlY3VyaXR5X3Njb3Blcz1zZWN1cml0eV9zY29wZXMsCiAgICAgICAgIHVzZV9jYWNoZT11c2VfY2FjaGUsCiAgICAgICAgIHNjb3BlPXNjb3BlLAorICAgICAgICBvd25fb2F1dGhfc2NvcGVzPW93bl9vYXV0aF9zY29wZXMsCisgICAgICAgIHBhcmVudF9vYXV0aF9zY29wZXM9cGFyZW50X29hdXRoX3Njb3BlcywKICAgICApCisgICAgY3VycmVudF9zY29wZXMgPSAocGFyZW50X29hdXRoX3Njb3BlcyBvciBbXSkgKyAob3duX29hdXRoX3Njb3BlcyBvciBbXSkKICAgICBwYXRoX3BhcmFtX25hbWVzID0gZ2V0X3BhdGhfcGFyYW1fbmFtZXMocGF0aCkKICAgICBlbmRwb2ludF9zaWduYXR1cmUgPSBnZXRfdHlwZWRfc2lnbmF0dXJlKGNhbGwpCiAgICAgc2lnbmF0dXJlX3BhcmFtcyA9IGVuZHBvaW50X3NpZ25hdHVyZS5wYXJhbWV0ZXJzCiAgICAgaWYgaXNpbnN0YW5jZShjYWxsLCBTZWN1cml0eUJhc2UpOgotICAgICAgICB1c2Vfc2NvcGVzOiBMaXN0W3N0cl0gPSBbXQotICAgICAgICBpZiBpc2luc3RhbmNlKGNhbGwsIChPQXV0aDIsIE9wZW5JZENvbm5lY3QpKToKLSAgICAgICAgICAgIHVzZV9zY29wZXMgPSBzZWN1cml0eV9zY29wZXMgb3IgdXNlX3Njb3BlcwogICAgICAgICBzZWN1cml0eV9yZXF1aXJlbWVudCA9IFNlY3VyaXR5UmVxdWlyZW1lbnQoCi0gICAgICAgICAgICBzZWN1cml0eV9zY2hlbWU9Y2FsbCwgc2NvcGVzPXVzZV9zY29wZXMKKyAgICAgICAgICAgIHNlY3VyaXR5X3NjaGVtZT1jYWxsLCBzY29wZXM9Y3VycmVudF9zY29wZXMKICAgICAgICAgKQogICAgICAgICBkZXBlbmRhbnQuc2VjdXJpdHlfcmVxdWlyZW1lbnRzLmFwcGVuZChzZWN1cml0eV9yZXF1aXJlbWVudCkKICAgICBmb3IgcGFyYW1fbmFtZSwgcGFyYW0gaW4gc2lnbmF0dXJlX3BhcmFtcy5pdGVtcygpOgpAQCAtMjc1LDE3ICsyNzQsMTYgQEAgZGVmIGdldF9kZXBlbmRhbnQoCiAgICAgICAgICAgICAgICAgICAgIGYnVGhlIGRlcGVuZGVuY3kgIntkZXBlbmRhbnQuY2FsbC5fX25hbWVfX30iIGhhcyBhIHNjb3BlIG9mICcKICAgICAgICAgICAgICAgICAgICAgJyJyZXF1ZXN0IiwgaXQgY2Fubm90IGRlcGVuZCBvbiBkZXBlbmRlbmNpZXMgd2l0aCBzY29wZSAiZnVuY3Rpb24iLicKICAgICAgICAgICAgICAgICApCi0gICAgICAgICAgICB1c2Vfc2VjdXJpdHlfc2NvcGVzID0gc2VjdXJpdHlfc2NvcGVzIG9yIFtdCisgICAgICAgICAgICBzdWJfb3duX29hdXRoX3Njb3BlczogTGlzdFtzdHJdID0gW10KICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UocGFyYW1fZGV0YWlscy5kZXBlbmRzLCBwYXJhbXMuU2VjdXJpdHkpOgogICAgICAgICAgICAgICAgIGlmIHBhcmFtX2RldGFpbHMuZGVwZW5kcy5zY29wZXM6Ci0gICAgICAgICAgICAgICAgICAgIHVzZV9zZWN1cml0eV9zY29wZXMgPSB1c2Vfc2VjdXJpdHlfc2NvcGVzICsgbGlzdCgKLSAgICAgICAgICAgICAgICAgICAgICAgIHBhcmFtX2RldGFpbHMuZGVwZW5kcy5zY29wZXMKLSAgICAgICAgICAgICAgICAgICAgKQorICAgICAgICAgICAgICAgICAgICBzdWJfb3duX29hdXRoX3Njb3BlcyA9IGxpc3QocGFyYW1fZGV0YWlscy5kZXBlbmRzLnNjb3BlcykKICAgICAgICAgICAgIHN1Yl9kZXBlbmRhbnQgPSBnZXRfZGVwZW5kYW50KAogICAgICAgICAgICAgICAgIHBhdGg9cGF0aCwKICAgICAgICAgICAgICAgICBjYWxsPXBhcmFtX2RldGFpbHMuZGVwZW5kcy5kZXBlbmRlbmN5LAogICAgICAgICAgICAgICAgIG5hbWU9cGFyYW1fbmFtZSwKLSAgICAgICAgICAgICAgICBzZWN1cml0eV9zY29wZXM9dXNlX3NlY3VyaXR5X3Njb3BlcywKKyAgICAgICAgICAgICAgICBvd25fb2F1dGhfc2NvcGVzPXN1Yl9vd25fb2F1dGhfc2NvcGVzLAorICAgICAgICAgICAgICAgIHBhcmVudF9vYXV0aF9zY29wZXM9Y3VycmVudF9zY29wZXMsCiAgICAgICAgICAgICAgICAgdXNlX2NhY2hlPXBhcmFtX2RldGFpbHMuZGVwZW5kcy51c2VfY2FjaGUsCiAgICAgICAgICAgICAgICAgc2NvcGU9cGFyYW1fZGV0YWlscy5kZXBlbmRzLnNjb3BlLAogICAgICAgICAgICAgKQpAQCAtNjExLDcgKzYwOSw3IEBAIGFzeW5jIGRlZiBzb2x2ZV9kZXBlbmRlbmNpZXMoCiAgICAgICAgICAgICAgICAgcGF0aD11c2VfcGF0aCwKICAgICAgICAgICAgICAgICBjYWxsPWNhbGwsCiAgICAgICAgICAgICAgICAgbmFtZT1zdWJfZGVwZW5kYW50Lm5hbWUsCi0gICAgICAgICAgICAgICAgc2VjdXJpdHlfc2NvcGVzPXN1Yl9kZXBlbmRhbnQuc2VjdXJpdHlfc2NvcGVzLAorICAgICAgICAgICAgICAgIHBhcmVudF9vYXV0aF9zY29wZXM9c3ViX2RlcGVuZGFudC5vYXV0aF9zY29wZXMsCiAgICAgICAgICAgICAgICAgc2NvcGU9c3ViX2RlcGVuZGFudC5zY29wZSwKICAgICAgICAgICAgICkKIApAQCAtNjkzLDcgKzY5MSw3IEBAIGFzeW5jIGRlZiBzb2x2ZV9kZXBlbmRlbmNpZXMoCiAgICAgICAgIHZhbHVlc1tkZXBlbmRhbnQucmVzcG9uc2VfcGFyYW1fbmFtZV0gPSByZXNwb25zZQogICAgIGlmIGRlcGVuZGFudC5zZWN1cml0eV9zY29wZXNfcGFyYW1fbmFtZToKICAgICAgICAgdmFsdWVzW2RlcGVuZGFudC5zZWN1cml0eV9zY29wZXNfcGFyYW1fbmFtZV0gPSBTZWN1cml0eVNjb3BlcygKLSAgICAgICAgICAgIHNjb3Blcz1kZXBlbmRhbnQuc2VjdXJpdHlfc2NvcGVzCisgICAgICAgICAgICBzY29wZXM9ZGVwZW5kYW50Lm9hdXRoX3Njb3BlcwogICAgICAgICApCiAgICAgcmV0dXJuIFNvbHZlZERlcGVuZGVuY3koCiAgICAgICAgIHZhbHVlcz12YWx1ZXMsCg==", "fastapi_14301": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTEyOSw3ICsxMjksMTAgQEAgZGVmIGdldF9wYXJhbWV0ZXJsZXNzX3N1Yl9kZXBlbmRhbnQoKiwgZGVwZW5kczogcGFyYW1zLkRlcGVuZHMsIHBhdGg6IHN0cikgLT4gRGUKICAgICBpZiBpc2luc3RhbmNlKGRlcGVuZHMsIHBhcmFtcy5TZWN1cml0eSkgYW5kIGRlcGVuZHMuc2NvcGVzOgogICAgICAgICB1c2Vfc2VjdXJpdHlfc2NvcGVzLmV4dGVuZChkZXBlbmRzLnNjb3BlcykKICAgICByZXR1cm4gZ2V0X2RlcGVuZGFudCgKLSAgICAgICAgcGF0aD1wYXRoLCBjYWxsPWRlcGVuZHMuZGVwZW5kZW5jeSwgc2VjdXJpdHlfc2NvcGVzPXVzZV9zZWN1cml0eV9zY29wZXMKKyAgICAgICAgcGF0aD1wYXRoLAorICAgICAgICBjYWxsPWRlcGVuZHMuZGVwZW5kZW5jeSwKKyAgICAgICAgc2NvcGU9ZGVwZW5kcy5zY29wZSwKKyAgICAgICAgc2VjdXJpdHlfc2NvcGVzPXVzZV9zZWN1cml0eV9zY29wZXMsCiAgICAgKQogCiAK", "fastapi_14262": "...", "fastapi_14266": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTI0OCw2ICsyNDgsMTQgQEAgZGVmIGdldF9kZXBlbmRhbnQoCiAgICAgcGF0aF9wYXJhbV9uYW1lcyA9IGdldF9wYXRoX3BhcmFtX25hbWVzKHBhdGgpCiAgICAgZW5kcG9pbnRfc2lnbmF0dXJlID0gZ2V0X3R5cGVkX3NpZ25hdHVyZShjYWxsKQogICAgIHNpZ25hdHVyZV9wYXJhbXMgPSBlbmRwb2ludF9zaWduYXR1cmUucGFyYW1ldGVycworICAgIGlmIGlzaW5zdGFuY2UoY2FsbCwgU2VjdXJpdHlCYXNlKToKKyAgICAgICAgdXNlX3Njb3BlczogTGlzdFtzdHJdID0gW10KKyAgICAgICAgaWYgaXNpbnN0YW5jZShjYWxsLCAoT0F1dGgyLCBPcGVuSWRDb25uZWN0KSk6CisgICAgICAgICAgICB1c2Vfc2NvcGVzID0gc2VjdXJpdHlfc2NvcGVzCisgICAgICAgIHNlY3VyaXR5X3JlcXVpcmVtZW50ID0gU2VjdXJpdHlSZXF1aXJlbWVudCgKKyAgICAgICAgICAgIHNlY3VyaXR5X3NjaGVtZT1jYWxsLCBzY29wZXM9dXNlX3Njb3BlcworICAgICAgICApCisgICAgICAgIGRlcGVuZGFudC5zZWN1cml0eV9yZXF1aXJlbWVudHMuYXBwZW5kKHNlY3VyaXR5X3JlcXVpcmVtZW50KQogICAgIGZvciBwYXJhbV9uYW1lLCBwYXJhbSBpbiBzaWduYXR1cmVfcGFyYW1zLml0ZW1zKCk6CiAgICAgICAgIGlzX3BhdGhfcGFyYW0gPSBwYXJhbV9uYW1lIGluIHBhdGhfcGFyYW1fbmFtZXMKICAgICAgICAgcGFyYW1fZGV0YWlscyA9IGFuYWx5emVfcGFyYW0oCkBAIC0yNjksMTYgKzI3Nyw2IEBAIGRlZiBnZXRfZGVwZW5kYW50KAogICAgICAgICAgICAgICAgIHNlY3VyaXR5X3Njb3Blcz11c2Vfc2VjdXJpdHlfc2NvcGVzLAogICAgICAgICAgICAgICAgIHVzZV9jYWNoZT1wYXJhbV9kZXRhaWxzLmRlcGVuZHMudXNlX2NhY2hlLAogICAgICAgICAgICAgKQotICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwYXJhbV9kZXRhaWxzLmRlcGVuZHMuZGVwZW5kZW5jeSwgU2VjdXJpdHlCYXNlKToKLSAgICAgICAgICAgICAgICB1c2Vfc2NvcGVzOiBMaXN0W3N0cl0gPSBbXQotICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UoCi0gICAgICAgICAgICAgICAgICAgIHBhcmFtX2RldGFpbHMuZGVwZW5kcy5kZXBlbmRlbmN5LCAoT0F1dGgyLCBPcGVuSWRDb25uZWN0KQotICAgICAgICAgICAgICAgICk6Ci0gICAgICAgICAgICAgICAgICAgIHVzZV9zY29wZXMgPSB1c2Vfc2VjdXJpdHlfc2NvcGVzCi0gICAgICAgICAgICAgICAgc2VjdXJpdHlfcmVxdWlyZW1lbnQgPSBTZWN1cml0eVJlcXVpcmVtZW50KAotICAgICAgICAgICAgICAgICAgICBzZWN1cml0eV9zY2hlbWU9cGFyYW1fZGV0YWlscy5kZXBlbmRzLmRlcGVuZGVuY3ksIHNjb3Blcz11c2Vfc2NvcGVzCi0gICAgICAgICAgICAgICAgKQotICAgICAgICAgICAgICAgIHN1Yl9kZXBlbmRhbnQuc2VjdXJpdHlfcmVxdWlyZW1lbnRzLmFwcGVuZChzZWN1cml0eV9yZXF1aXJlbWVudCkKICAgICAgICAgICAgIGRlcGVuZGFudC5kZXBlbmRlbmNpZXMuYXBwZW5kKHN1Yl9kZXBlbmRhbnQpCiAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICBpZiBhZGRfbm9uX2ZpZWxkX3BhcmFtX3RvX2RlcGVuZGVuY3koCg==", "fastapi_14246": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMjA3LDExICsyMDcsMzEgQEAgZGVmIGdldF9kZWZpbml0aW9ucygKICAgICBvdmVycmlkZV9tb2RlOiBVbmlvbltMaXRlcmFsWyJ2YWxpZGF0aW9uIl0sIE5vbmVdID0gKAogICAgICAgICBOb25lIGlmIHNlcGFyYXRlX2lucHV0X291dHB1dF9zY2hlbWFzIGVsc2UgInZhbGlkYXRpb24iCiAgICAgKQotICAgIGZsYXRfbW9kZWxzID0gZ2V0X2ZsYXRfbW9kZWxzX2Zyb21fZmllbGRzKGZpZWxkcywga25vd25fbW9kZWxzPXNldCgpKQotICAgIGZsYXRfbW9kZWxfZmllbGRzID0gWwotICAgICAgICBNb2RlbEZpZWxkKGZpZWxkX2luZm89RmllbGRJbmZvKGFubm90YXRpb249bW9kZWwpLCBuYW1lPW1vZGVsLl9fbmFtZV9fKQotICAgICAgICBmb3IgbW9kZWwgaW4gZmxhdF9tb2RlbHMKKyAgICB2YWxpZGF0aW9uX2ZpZWxkcyA9IFtmaWVsZCBmb3IgZmllbGQgaW4gZmllbGRzIGlmIGZpZWxkLm1vZGUgPT0gInZhbGlkYXRpb24iXQorICAgIHNlcmlhbGl6YXRpb25fZmllbGRzID0gW2ZpZWxkIGZvciBmaWVsZCBpbiBmaWVsZHMgaWYgZmllbGQubW9kZSA9PSAic2VyaWFsaXphdGlvbiJdCisgICAgZmxhdF92YWxpZGF0aW9uX21vZGVscyA9IGdldF9mbGF0X21vZGVsc19mcm9tX2ZpZWxkcygKKyAgICAgICAgdmFsaWRhdGlvbl9maWVsZHMsIGtub3duX21vZGVscz1zZXQoKQorICAgICkKKyAgICBmbGF0X3NlcmlhbGl6YXRpb25fbW9kZWxzID0gZ2V0X2ZsYXRfbW9kZWxzX2Zyb21fZmllbGRzKAorICAgICAgICBzZXJpYWxpemF0aW9uX2ZpZWxkcywga25vd25fbW9kZWxzPXNldCgpCisgICAgKQorICAgIGZsYXRfdmFsaWRhdGlvbl9tb2RlbF9maWVsZHMgPSBbCisgICAgICAgIE1vZGVsRmllbGQoCisgICAgICAgICAgICBmaWVsZF9pbmZvPUZpZWxkSW5mbyhhbm5vdGF0aW9uPW1vZGVsKSwKKyAgICAgICAgICAgIG5hbWU9bW9kZWwuX19uYW1lX18sCisgICAgICAgICAgICBtb2RlPSJ2YWxpZGF0aW9uIiwKKyAgICAgICAgKQorICAgICAgICBmb3IgbW9kZWwgaW4gZmxhdF92YWxpZGF0aW9uX21vZGVscworICAgIF0KKyAgICBmbGF0X3NlcmlhbGl6YXRpb25fbW9kZWxfZmllbGRzID0gWworICAgICAgICBNb2RlbEZpZWxkKAorICAgICAgICAgICAgZmllbGRfaW5mbz1GaWVsZEluZm8oYW5ub3RhdGlvbj1tb2RlbCksCisgICAgICAgICAgICBuYW1lPW1vZGVsLl9fbmFtZV9fLAorICAgICAgICAgICAgbW9kZT0ic2VyaWFsaXphdGlvbiIsCisgICAgICAgICkKKyAgICAgICAgZm9yIG1vZGVsIGluIGZsYXRfc2VyaWFsaXphdGlvbl9tb2RlbHMKICAgICBdCisgICAgZmxhdF9tb2RlbF9maWVsZHMgPSBmbGF0X3ZhbGlkYXRpb25fbW9kZWxfZmllbGRzICsgZmxhdF9zZXJpYWxpemF0aW9uX21vZGVsX2ZpZWxkcwogICAgIGlucHV0X3R5cGVzID0ge2YudHlwZV8gZm9yIGYgaW4gZmllbGRzfQogICAgIHVuaXF1ZV9mbGF0X21vZGVsX2ZpZWxkcyA9IHsKICAgICAgICAgZiBmb3IgZiBpbiBmbGF0X21vZGVsX2ZpZWxkcyBpZiBmLnR5cGVfIG5vdCBpbiBpbnB1dF90eXBlcwo=", "fastapi_14186": "...", "fastapi_9425": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTI1NCw2ICsyNTQsOCBAQCBkZWYgZ2V0X3R5cGVkX2Fubm90YXRpb24oYW5ub3RhdGlvbjogQW55LCBnbG9iYWxuczogRGljdFtzdHIsIEFueV0pIC0+IEFueToKICAgICBpZiBpc2luc3RhbmNlKGFubm90YXRpb24sIHN0cik6CiAgICAgICAgIGFubm90YXRpb24gPSBGb3J3YXJkUmVmKGFubm90YXRpb24pCiAgICAgICAgIGFubm90YXRpb24gPSBldmFsdWF0ZV9mb3J3YXJkcmVmKGFubm90YXRpb24sIGdsb2JhbG5zLCBnbG9iYWxucykKKyAgICAgICAgaWYgYW5ub3RhdGlvbiBpcyB0eXBlKE5vbmUpOgorICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICByZXR1cm4gYW5ub3RhdGlvbgogCiAK", "fastapi_13713": "LS0tIGEvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKKysrIGIvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKQEAgLTgxMCw2ICs4MTAsMzIgQEAgY2xhc3MgSXRlbShCYXNlTW9kZWwpOgogICAgICAgICAgICAgICAgICIiIgogICAgICAgICAgICAgKSwKICAgICAgICAgXSA9IFRydWUsCisgICAgICAgIG9wZW5hcGlfZXh0ZXJuYWxfZG9jczogQW5ub3RhdGVkWworICAgICAgICAgICAgT3B0aW9uYWxbRGljdFtzdHIsIEFueV1dLAorICAgICAgICAgICAgRG9jKAorICAgICAgICAgICAgICAgICIiIgorICAgICAgICAgICAgICAgIFRoaXMgZmllbGQgYWxsb3dzIHlvdSB0byBwcm92aWRlIGFkZGl0aW9uYWwgZXh0ZXJuYWwgZG9jdW1lbnRhdGlvbiBsaW5rcy4KKyAgICAgICAgICAgICAgICBJZiBwcm92aWRlZCwgaXQgbXVzdCBiZSBhIGRpY3Rpb25hcnkgY29udGFpbmluZzoKKworICAgICAgICAgICAgICAgICogYGRlc2NyaXB0aW9uYDogQSBicmllZiBkZXNjcmlwdGlvbiBvZiB0aGUgZXh0ZXJuYWwgZG9jdW1lbnRhdGlvbi4KKyAgICAgICAgICAgICAgICAqIGB1cmxgOiBUaGUgVVJMIHBvaW50aW5nIHRvIHRoZSBleHRlcm5hbCBkb2N1bWVudGF0aW9uLiBUaGUgdmFsdWUgKipNVVNUKioKKyAgICAgICAgICAgICAgICBiZSBhIHZhbGlkIFVSTCBmb3JtYXQuCisKKyAgICAgICAgICAgICAgICAqKkV4YW1wbGUqKjoKKworICAgICAgICAgICAgICAgIGBgYHB5dGhvbgorICAgICAgICAgICAgICAgIGZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSQorCisgICAgICAgICAgICAgICAgZXh0ZXJuYWxfZG9jcyA9IHsKKyAgICAgICAgICAgICAgICAgICAgImRlc2NyaXB0aW9uIjogIkRldGFpbGVkIEFQSSBSZWZlcmVuY2UiLAorICAgICAgICAgICAgICAgICAgICAidXJsIjogImh0dHBzOi8vZXhhbXBsZS5jb20vYXBpLWRvY3MiLAorICAgICAgICAgICAgICAgIH0KKworICAgICAgICAgICAgICAgIGFwcCA9IEZhc3RBUEkob3BlbmFwaV9leHRlcm5hbF9kb2NzPWV4dGVybmFsX2RvY3MpCisgICAgICAgICAgICAgICAgYGBgCisgICAgICAgICAgICAgICAgIiIiCisgICAgICAgICAgICApLAorICAgICAgICBdID0gTm9uZSwKICAgICAgICAgKipleHRyYTogQW5ub3RhdGVkWwogICAgICAgICAgICAgQW55LAogICAgICAgICAgICAgRG9jKApAQCAtODM4LDYgKzg2NCw3IEBAIGNsYXNzIEl0ZW0oQmFzZU1vZGVsKToKICAgICAgICAgc2VsZi5zd2FnZ2VyX3VpX3BhcmFtZXRlcnMgPSBzd2FnZ2VyX3VpX3BhcmFtZXRlcnMKICAgICAgICAgc2VsZi5zZXJ2ZXJzID0gc2VydmVycyBvciBbXQogICAgICAgICBzZWxmLnNlcGFyYXRlX2lucHV0X291dHB1dF9zY2hlbWFzID0gc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXMKKyAgICAgICAgc2VsZi5vcGVuYXBpX2V4dGVybmFsX2RvY3MgPSBvcGVuYXBpX2V4dGVybmFsX2RvY3MKICAgICAgICAgc2VsZi5leHRyYSA9IGV4dHJhCiAgICAgICAgIHNlbGYub3BlbmFwaV92ZXJzaW9uOiBBbm5vdGF0ZWRbCiAgICAgICAgICAgICBzdHIsCkBAIC05OTIsNiArMTAxOSw3IEBAIGRlZiBvcGVuYXBpKHNlbGYpIC0+IERpY3Rbc3RyLCBBbnldOgogICAgICAgICAgICAgICAgIHRhZ3M9c2VsZi5vcGVuYXBpX3RhZ3MsCiAgICAgICAgICAgICAgICAgc2VydmVycz1zZWxmLnNlcnZlcnMsCiAgICAgICAgICAgICAgICAgc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXM9c2VsZi5zZXBhcmF0ZV9pbnB1dF9vdXRwdXRfc2NoZW1hcywKKyAgICAgICAgICAgICAgICBleHRlcm5hbF9kb2NzPXNlbGYub3BlbmFwaV9leHRlcm5hbF9kb2NzLAogICAgICAgICAgICAgKQogICAgICAgICByZXR1cm4gc2VsZi5vcGVuYXBpX3NjaGVtYQogCi0tLSBhL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQorKysgYi9mYXN0YXBpL29wZW5hcGkvdXRpbHMucHkKQEAgLTQ4OCw2ICs0ODgsNyBAQCBkZWYgZ2V0X29wZW5hcGkoCiAgICAgY29udGFjdDogT3B0aW9uYWxbRGljdFtzdHIsIFVuaW9uW3N0ciwgQW55XV1dID0gTm9uZSwKICAgICBsaWNlbnNlX2luZm86IE9wdGlvbmFsW0RpY3Rbc3RyLCBVbmlvbltzdHIsIEFueV1dXSA9IE5vbmUsCiAgICAgc2VwYXJhdGVfaW5wdXRfb3V0cHV0X3NjaGVtYXM6IGJvb2wgPSBUcnVlLAorICAgIGV4dGVybmFsX2RvY3M6IE9wdGlvbmFsW0RpY3Rbc3RyLCBBbnldXSA9IE5vbmUsCiApIC0+IERpY3Rbc3RyLCBBbnldOgogICAgIGluZm86IERpY3Rbc3RyLCBBbnldID0geyJ0aXRsZSI6IHRpdGxlLCAidmVyc2lvbiI6IHZlcnNpb259CiAgICAgaWYgc3VtbWFyeToKQEAgLTU2NSw0ICs1NjYsNiBAQCBkZWYgZ2V0X29wZW5hcGkoCiAgICAgICAgIG91dHB1dFsid2ViaG9va3MiXSA9IHdlYmhvb2tfcGF0aHMKICAgICBpZiB0YWdzOgogICAgICAgICBvdXRwdXRbInRhZ3MiXSA9IHRhZ3MKKyAgICBpZiBleHRlcm5hbF9kb2NzOgorICAgICAgICBvdXRwdXRbImV4dGVybmFsRG9jcyJdID0gZXh0ZXJuYWxfZG9jcwogICAgIHJldHVybiBqc29uYWJsZV9lbmNvZGVyKE9wZW5BUEkoKipvdXRwdXQpLCBieV9hbGlhcz1UcnVlLCBleGNsdWRlX25vbmU9VHJ1ZSkgICMgdHlwZTogaWdub3JlCg==", "fastapi_14077": "LS0tIGEvZG9jc19zcmMvaGFuZGxpbmdfZXJyb3JzL3R1dG9yaWFsMDA1LnB5CisrKyBiL2RvY3Nfc3JjL2hhbmRsaW5nX2Vycm9ycy90dXRvcmlhbDAwNS5weQpAQCAtMSw0ICsxLDQgQEAKLWZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSSwgUmVxdWVzdCwgc3RhdHVzCitmcm9tIGZhc3RhcGkgaW1wb3J0IEZhc3RBUEksIFJlcXVlc3QKIGZyb20gZmFzdGFwaS5lbmNvZGVycyBpbXBvcnQganNvbmFibGVfZW5jb2RlcgogZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0IFJlcXVlc3RWYWxpZGF0aW9uRXJyb3IKIGZyb20gZmFzdGFwaS5yZXNwb25zZXMgaW1wb3J0IEpTT05SZXNwb25zZQpAQCAtMTAsNyArMTAsNyBAQAogQGFwcC5leGNlcHRpb25faGFuZGxlcihSZXF1ZXN0VmFsaWRhdGlvbkVycm9yKQogYXN5bmMgZGVmIHZhbGlkYXRpb25fZXhjZXB0aW9uX2hhbmRsZXIocmVxdWVzdDogUmVxdWVzdCwgZXhjOiBSZXF1ZXN0VmFsaWRhdGlvbkVycm9yKToKICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKAotICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXMuSFRUUF80MjJfVU5QUk9DRVNTQUJMRV9FTlRJVFksCisgICAgICAgIHN0YXR1c19jb2RlPTQyMiwKICAgICAgICAgY29udGVudD1qc29uYWJsZV9lbmNvZGVyKHsiZGV0YWlsIjogZXhjLmVycm9ycygpLCAiYm9keSI6IGV4Yy5ib2R5fSksCiAgICAgKQogCi0tLSBhL2Zhc3RhcGkvZXhjZXB0aW9uX2hhbmRsZXJzLnB5CisrKyBiL2Zhc3RhcGkvZXhjZXB0aW9uX2hhbmRsZXJzLnB5CkBAIC01LDcgKzUsNyBAQAogZnJvbSBzdGFybGV0dGUuZXhjZXB0aW9ucyBpbXBvcnQgSFRUUEV4Y2VwdGlvbgogZnJvbSBzdGFybGV0dGUucmVxdWVzdHMgaW1wb3J0IFJlcXVlc3QKIGZyb20gc3RhcmxldHRlLnJlc3BvbnNlcyBpbXBvcnQgSlNPTlJlc3BvbnNlLCBSZXNwb25zZQotZnJvbSBzdGFybGV0dGUuc3RhdHVzIGltcG9ydCBIVFRQXzQyMl9VTlBST0NFU1NBQkxFX0VOVElUWSwgV1NfMTAwOF9QT0xJQ1lfVklPTEFUSU9OCitmcm9tIHN0YXJsZXR0ZS5zdGF0dXMgaW1wb3J0IFdTXzEwMDhfUE9MSUNZX1ZJT0xBVElPTgogCiAKIGFzeW5jIGRlZiBodHRwX2V4Y2VwdGlvbl9oYW5kbGVyKHJlcXVlc3Q6IFJlcXVlc3QsIGV4YzogSFRUUEV4Y2VwdGlvbikgLT4gUmVzcG9uc2U6CkBAIC0yMSw3ICsyMSw3IEBAIGFzeW5jIGRlZiByZXF1ZXN0X3ZhbGlkYXRpb25fZXhjZXB0aW9uX2hhbmRsZXIoCiAgICAgcmVxdWVzdDogUmVxdWVzdCwgZXhjOiBSZXF1ZXN0VmFsaWRhdGlvbkVycm9yCiApIC0+IEpTT05SZXNwb25zZToKICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKAotICAgICAgICBzdGF0dXNfY29kZT1IVFRQXzQyMl9VTlBST0NFU1NBQkxFX0VOVElUWSwKKyAgICAgICAgc3RhdHVzX2NvZGU9NDIyLAogICAgICAgICBjb250ZW50PXsiZGV0YWlsIjoganNvbmFibGVfZW5jb2RlcihleGMuZXJyb3JzKCkpfSwKICAgICApCiAKLS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtMzUsNyArMzUsNiBAQAogZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsCiBmcm9tIHN0YXJsZXR0ZS5yZXNwb25zZXMgaW1wb3J0IEpTT05SZXNwb25zZQogZnJvbSBzdGFybGV0dGUucm91dGluZyBpbXBvcnQgQmFzZVJvdXRlCi1mcm9tIHN0YXJsZXR0ZS5zdGF0dXMgaW1wb3J0IEhUVFBfNDIyX1VOUFJPQ0VTU0FCTEVfRU5USVRZCiBmcm9tIHR5cGluZ19leHRlbnNpb25zIGltcG9ydCBMaXRlcmFsCiAKIHZhbGlkYXRpb25fZXJyb3JfZGVmaW5pdGlvbiA9IHsKQEAgLTQxNiw3ICs0MTUsNyBAQCBkZWYgZ2V0X29wZW5hcGlfcGF0aCgKICAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgICBkZWVwX2RpY3RfdXBkYXRlKG9wZW5hcGlfcmVzcG9uc2UsIHByb2Nlc3NfcmVzcG9uc2UpCiAgICAgICAgICAgICAgICAgICAgIG9wZW5hcGlfcmVzcG9uc2VbImRlc2NyaXB0aW9uIl0gPSBkZXNjcmlwdGlvbgotICAgICAgICAgICAgaHR0cDQyMiA9IHN0cihIVFRQXzQyMl9VTlBST0NFU1NBQkxFX0VOVElUWSkKKyAgICAgICAgICAgIGh0dHA0MjIgPSAiNDIyIgogICAgICAgICAgICAgYWxsX3JvdXRlX3BhcmFtcyA9IGdldF9mbGF0X3BhcmFtcyhyb3V0ZS5kZXBlbmRhbnQpCiAgICAgICAgICAgICBpZiAoYWxsX3JvdXRlX3BhcmFtcyBvciByb3V0ZS5ib2R5X2ZpZWxkKSBhbmQgbm90IGFueSgKICAgICAgICAgICAgICAgICBzdGF0dXMgaW4gb3BlcmF0aW9uWyJyZXNwb25zZXMiXQo=", "fastapi_14791": "LS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtNDksNiArNDksOCBAQAogICAgICAgICB9LAogICAgICAgICAibXNnIjogeyJ0aXRsZSI6ICJNZXNzYWdlIiwgInR5cGUiOiAic3RyaW5nIn0sCiAgICAgICAgICJ0eXBlIjogeyJ0aXRsZSI6ICJFcnJvciBUeXBlIiwgInR5cGUiOiAic3RyaW5nIn0sCisgICAgICAgICJpbnB1dCI6IHsidGl0bGUiOiAiSW5wdXQifSwKKyAgICAgICAgImN0eCI6IHsidGl0bGUiOiAiQ29udGV4dCIsICJ0eXBlIjogIm9iamVjdCJ9LAogICAgIH0sCiAgICAgInJlcXVpcmVkIjogWyJsb2MiLCAibXNnIiwgInR5cGUiXSwKIH0K", "fastapi_15800": "...", "fastapi_15785": "LS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtNDc5LDI2ICs0NzksMjIgQEAgZGVmIGdldF9vcGVuYXBpX3BhdGgoCiAKIAogZGVmIF9nZXRfYXBpX3JvdXRlX2Zvcl9vcGVuYXBpKAotICAgIHJvdXRlOiBCYXNlUm91dGUsIHJvdXRlX2NvbnRleHQ6IHJvdXRpbmcuX0VmZmVjdGl2ZVJvdXRlQ29udGV4dCB8IE5vbmUKKyAgICByb3V0ZV9jb250ZXh0OiByb3V0aW5nLlJvdXRlQ29udGV4dCwKICkgLT4gcm91dGluZy5fQVBJUm91dGVMaWtlIHwgTm9uZToKLSAgICBpZiByb3V0ZV9jb250ZXh0IGlzIG5vdCBOb25lIGFuZCBpc2luc3RhbmNlKAotICAgICAgICByb3V0ZV9jb250ZXh0Lm9yaWdpbmFsX3JvdXRlLCByb3V0aW5nLkFQSVJvdXRlCi0gICAgKToKKyAgICBpZiBpc2luc3RhbmNlKHJvdXRlX2NvbnRleHQub3JpZ2luYWxfcm91dGUsIHJvdXRpbmcuQVBJUm91dGUpOgogICAgICAgICByZXR1cm4gY2FzdChyb3V0aW5nLl9BUElSb3V0ZUxpa2UsIHJvdXRlX2NvbnRleHQpCi0gICAgaWYgaXNpbnN0YW5jZShyb3V0ZSwgcm91dGluZy5BUElSb3V0ZSk6Ci0gICAgICAgIHJldHVybiBjYXN0KHJvdXRpbmcuX0FQSVJvdXRlTGlrZSwgcm91dGUpCiAgICAgcmV0dXJuIE5vbmUKIAogCiBkZWYgZ2V0X2ZpZWxkc19mcm9tX3JvdXRlcygKLSAgICByb3V0ZXM6IFNlcXVlbmNlW0Jhc2VSb3V0ZV0sCisgICAgcm91dGVzOiBTZXF1ZW5jZVtCYXNlUm91dGUgfCByb3V0aW5nLlJvdXRlQ29udGV4dF0sCiApIC0+IGxpc3RbTW9kZWxGaWVsZF06CiAgICAgYm9keV9maWVsZHNfZnJvbV9yb3V0ZXM6IGxpc3RbTW9kZWxGaWVsZF0gPSBbXQogICAgIHJlc3BvbnNlc19mcm9tX3JvdXRlczogbGlzdFtNb2RlbEZpZWxkXSA9IFtdCiAgICAgcmVxdWVzdF9maWVsZHNfZnJvbV9yb3V0ZXM6IGxpc3RbTW9kZWxGaWVsZF0gPSBbXQogICAgIGNhbGxiYWNrX2ZsYXRfbW9kZWxzOiBsaXN0W01vZGVsRmllbGRdID0gW10KLSAgICBmb3Igcm91dGUsIHJvdXRlX2NvbnRleHQgaW4gcm91dGluZy5faXRlcl9yb3V0ZXNfd2l0aF9jb250ZXh0KHJvdXRlcyk6Ci0gICAgICAgIGFwaV9yb3V0ZSA9IF9nZXRfYXBpX3JvdXRlX2Zvcl9vcGVuYXBpKHJvdXRlLCByb3V0ZV9jb250ZXh0KQorICAgIGZvciByb3V0ZV9jb250ZXh0IGluIHJvdXRpbmcuaXRlcl9yb3V0ZV9jb250ZXh0cyhyb3V0ZXMpOgorICAgICAgICBhcGlfcm91dGUgPSBfZ2V0X2FwaV9yb3V0ZV9mb3Jfb3BlbmFwaShyb3V0ZV9jb250ZXh0KQogICAgICAgICBpZiBhcGlfcm91dGUgaXMgTm9uZToKICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIGlmIGFwaV9yb3V0ZS5pbmNsdWRlX2luX3NjaGVtYToKQEAgLTUzMSw4ICs1MjcsOCBAQCBkZWYgZ2V0X29wZW5hcGkoCiAgICAgb3BlbmFwaV92ZXJzaW9uOiBzdHIgPSAiMy4xLjAiLAogICAgIHN1bW1hcnk6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgIGRlc2NyaXB0aW9uOiBzdHIgfCBOb25lID0gTm9uZSwKLSAgICByb3V0ZXM6IFNlcXVlbmNlW0Jhc2VSb3V0ZV0sCi0gICAgd2ViaG9va3M6IFNlcXVlbmNlW0Jhc2VSb3V0ZV0gfCBOb25lID0gTm9uZSwKKyAgICByb3V0ZXM6IFNlcXVlbmNlW0Jhc2VSb3V0ZSB8IHJvdXRpbmcuUm91dGVDb250ZXh0XSwKKyAgICB3ZWJob29rczogU2VxdWVuY2VbQmFzZVJvdXRlIHwgcm91dGluZy5Sb3V0ZUNvbnRleHRdIHwgTm9uZSA9IE5vbmUsCiAgICAgdGFnczogbGlzdFtkaWN0W3N0ciwgQW55XV0gfCBOb25lID0gTm9uZSwKICAgICBzZXJ2ZXJzOiBsaXN0W2RpY3Rbc3RyLCBzdHIgfCBBbnldXSB8IE5vbmUgPSBOb25lLAogICAgIHRlcm1zX29mX3NlcnZpY2U6IHN0ciB8IE5vbmUgPSBOb25lLApAQCAtNTY3LDggKzU2Myw4IEBAIGRlZiBnZXRfb3BlbmFwaSgKICAgICAgICAgbW9kZWxfbmFtZV9tYXA9bW9kZWxfbmFtZV9tYXAsCiAgICAgICAgIHNlcGFyYXRlX2lucHV0X291dHB1dF9zY2hlbWFzPXNlcGFyYXRlX2lucHV0X291dHB1dF9zY2hlbWFzLAogICAgICkKLSAgICBmb3Igcm91dGUsIHJvdXRlX2NvbnRleHQgaW4gcm91dGluZy5faXRlcl9yb3V0ZXNfd2l0aF9jb250ZXh0KHJvdXRlcyk6Ci0gICAgICAgIGFwaV9yb3V0ZSA9IF9nZXRfYXBpX3JvdXRlX2Zvcl9vcGVuYXBpKHJvdXRlLCByb3V0ZV9jb250ZXh0KQorICAgIGZvciByb3V0ZV9jb250ZXh0IGluIHJvdXRpbmcuaXRlcl9yb3V0ZV9jb250ZXh0cyhyb3V0ZXMpOgorICAgICAgICBhcGlfcm91dGUgPSBfZ2V0X2FwaV9yb3V0ZV9mb3Jfb3BlbmFwaShyb3V0ZV9jb250ZXh0KQogICAgICAgICBpZiBhcGlfcm91dGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICByZXN1bHQgPSBnZXRfb3BlbmFwaV9wYXRoKAogICAgICAgICAgICAgICAgIHJvdXRlPWFwaV9yb3V0ZSwKQEAgLTU4Nyw4ICs1ODMsOCBAQCBkZWYgZ2V0X29wZW5hcGkoCiAgICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICBpZiBwYXRoX2RlZmluaXRpb25zOgogICAgICAgICAgICAgICAgICAgICBkZWZpbml0aW9ucy51cGRhdGUocGF0aF9kZWZpbml0aW9ucykKLSAgICBmb3Igd2ViaG9vaywgd2ViaG9va19jb250ZXh0IGluIHJvdXRpbmcuX2l0ZXJfcm91dGVzX3dpdGhfY29udGV4dCh3ZWJob29rcyBvciBbXSk6Ci0gICAgICAgIGFwaV93ZWJob29rID0gX2dldF9hcGlfcm91dGVfZm9yX29wZW5hcGkod2ViaG9vaywgd2ViaG9va19jb250ZXh0KQorICAgIGZvciB3ZWJob29rX2NvbnRleHQgaW4gcm91dGluZy5pdGVyX3JvdXRlX2NvbnRleHRzKHdlYmhvb2tzIG9yIFtdKToKKyAgICAgICAgYXBpX3dlYmhvb2sgPSBfZ2V0X2FwaV9yb3V0ZV9mb3Jfb3BlbmFwaSh3ZWJob29rX2NvbnRleHQpCiAgICAgICAgIGlmIGFwaV93ZWJob29rIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgcmVzdWx0ID0gZ2V0X29wZW5hcGlfcGF0aCgKICAgICAgICAgICAgICAgICByb3V0ZT1hcGlfd2ViaG9vaywKLS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtMTQ1NCw2ICsxNDU0LDQ3IEBAIGRlZiB1cmxfcGF0aF9mb3Ioc2VsZiwgbmFtZTogc3RyLCAvLCAqKnBhdGhfcGFyYW1zOiBBbnkpIC0+IEFueToKICAgICAgICAgcmV0dXJuIFVSTFBhdGgocGF0aD1wYXRoLCBwcm90b2NvbD0iaHR0cCIpCiAKIAorQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKK2NsYXNzIFJvdXRlQ29udGV4dDoKKyAgICByb3V0ZTogQmFzZVJvdXRlCisgICAgX3JvdXRlX2NvbnRleHQ6IF9FZmZlY3RpdmVSb3V0ZUNvbnRleHQgfCBOb25lID0gZmllbGQoZGVmYXVsdD1Ob25lLCByZXByPUZhbHNlKQorCisgICAgQHByb3BlcnR5CisgICAgZGVmIG9yaWdpbmFsX3JvdXRlKHNlbGYpIC0+IEJhc2VSb3V0ZToKKyAgICAgICAgaWYgc2VsZi5fcm91dGVfY29udGV4dCBpcyBub3QgTm9uZToKKyAgICAgICAgICAgIHJldHVybiBzZWxmLl9yb3V0ZV9jb250ZXh0Lm9yaWdpbmFsX3JvdXRlCisgICAgICAgIHJldHVybiBzZWxmLnJvdXRlCisKKyAgICBAcHJvcGVydHkKKyAgICBkZWYgX2VmZmVjdGl2ZV9yb3V0ZShzZWxmKSAtPiBCYXNlUm91dGUgfCBfRWZmZWN0aXZlUm91dGVDb250ZXh0OgorICAgICAgICBpZiBzZWxmLl9yb3V0ZV9jb250ZXh0IGlzIG5vdCBOb25lOgorICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3JvdXRlX2NvbnRleHQKKyAgICAgICAgcmV0dXJuIHNlbGYucm91dGUKKworICAgIEBwcm9wZXJ0eQorICAgIGRlZiBwYXRoKHNlbGYpIC0+IHN0ciB8IE5vbmU6CisgICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuX2VmZmVjdGl2ZV9yb3V0ZSwgInBhdGgiLCBOb25lKQorCisgICAgQHByb3BlcnR5CisgICAgZGVmIHBhdGhfZm9ybWF0KHNlbGYpIC0+IHN0ciB8IE5vbmU6CisgICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuX2VmZmVjdGl2ZV9yb3V0ZSwgInBhdGhfZm9ybWF0IiwgTm9uZSkKKworICAgIEBwcm9wZXJ0eQorICAgIGRlZiBuYW1lKHNlbGYpIC0+IHN0ciB8IE5vbmU6CisgICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuX2VmZmVjdGl2ZV9yb3V0ZSwgIm5hbWUiLCBOb25lKQorCisgICAgQHByb3BlcnR5CisgICAgZGVmIG1ldGhvZHMoc2VsZikgLT4gc2V0W3N0cl0gfCBOb25lOgorICAgICAgICByZXR1cm4gZ2V0YXR0cihzZWxmLl9lZmZlY3RpdmVfcm91dGUsICJtZXRob2RzIiwgTm9uZSkKKworICAgIEBwcm9wZXJ0eQorICAgIGRlZiBlbmRwb2ludChzZWxmKSAtPiBDYWxsYWJsZVsuLi4sIEFueV0gfCBOb25lOgorICAgICAgICByZXR1cm4gZ2V0YXR0cihzZWxmLl9lZmZlY3RpdmVfcm91dGUsICJlbmRwb2ludCIsIE5vbmUpCisKKyAgICBkZWYgX19nZXRhdHRyX18oc2VsZiwgbmFtZTogc3RyKSAtPiBBbnk6CisgICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuX2VmZmVjdGl2ZV9yb3V0ZSwgbmFtZSkKKworCiBAZGF0YWNsYXNzCiBjbGFzcyBfSW5jbHVkZWRSb3V0ZXIoQmFzZVJvdXRlKToKICAgICBvcmlnaW5hbF9yb3V0ZXI6ICJBUElSb3V0ZXIiCkBAIC0xNjU0LDYgKzE2OTUsMjAgQEAgZGVmIF9pdGVyX2luY2x1ZGVkX3JvdXRlX2NhbmRpZGF0ZXMocm91dGVzOiBTZXF1ZW5jZVtCYXNlUm91dGVdKSAtPiBJdGVyYXRvcltCYXMKICAgICAgICAgICAgIHlpZWxkIHJvdXRlCiAKIAorZGVmIGl0ZXJfcm91dGVfY29udGV4dHMoCisgICAgcm91dGVzOiBTZXF1ZW5jZVtCYXNlUm91dGUgfCBSb3V0ZUNvbnRleHRdLAorKSAtPiBJdGVyYXRvcltSb3V0ZUNvbnRleHRdOgorICAgIGZvciByb3V0ZSBpbiByb3V0ZXM6CisgICAgICAgIGlmIGlzaW5zdGFuY2Uocm91dGUsIFJvdXRlQ29udGV4dCk6CisgICAgICAgICAgICB5aWVsZCByb3V0ZQorICAgICAgICAgICAgY29udGludWUKKyAgICAgICAgZm9yIG9yaWdpbmFsX3JvdXRlLCByb3V0ZV9jb250ZXh0IGluIF9pdGVyX3JvdXRlc193aXRoX2NvbnRleHQoW3JvdXRlXSk6CisgICAgICAgICAgICBpZiByb3V0ZV9jb250ZXh0IGlzIE5vbmU6CisgICAgICAgICAgICAgICAgeWllbGQgUm91dGVDb250ZXh0KG9yaWdpbmFsX3JvdXRlKQorICAgICAgICAgICAgZWxzZToKKyAgICAgICAgICAgICAgICB5aWVsZCBSb3V0ZUNvbnRleHQob3JpZ2luYWxfcm91dGUsIHJvdXRlX2NvbnRleHQpCisKKwogZGVmIF9pdGVyX3JvdXRlc193aXRoX2NvbnRleHQoCiAgICAgcm91dGVzOiBTZXF1ZW5jZVtCYXNlUm91dGVdLAogKSAtPiBJdGVyYXRvclt0dXBsZVtCYXNlUm91dGUsIF9FZmZlY3RpdmVSb3V0ZUNvbnRleHQgfCBOb25lXV06Cg==", "fastapi_14371": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMTEwLDYgKzExMCwxOCBAQCBkZWYgYWxpYXMoc2VsZikgLT4gc3RyOgogICAgICAgICBhID0gc2VsZi5maWVsZF9pbmZvLmFsaWFzCiAgICAgICAgIHJldHVybiBhIGlmIGEgaXMgbm90IE5vbmUgZWxzZSBzZWxmLm5hbWUKIAorICAgIEBwcm9wZXJ0eQorICAgIGRlZiB2YWxpZGF0aW9uX2FsaWFzKHNlbGYpIC0+IFVuaW9uW3N0ciwgTm9uZV06CisgICAgICAgIHZhID0gc2VsZi5maWVsZF9pbmZvLnZhbGlkYXRpb25fYWxpYXMKKyAgICAgICAgaWYgaXNpbnN0YW5jZSh2YSwgc3RyKSBhbmQgdmE6CisgICAgICAgICAgICByZXR1cm4gdmEKKyAgICAgICAgcmV0dXJuIE5vbmUKKworICAgIEBwcm9wZXJ0eQorICAgIGRlZiBzZXJpYWxpemF0aW9uX2FsaWFzKHNlbGYpIC0+IFVuaW9uW3N0ciwgTm9uZV06CisgICAgICAgIHNhID0gc2VsZi5maWVsZF9pbmZvLnNlcmlhbGl6YXRpb25fYWxpYXMKKyAgICAgICAgcmV0dXJuIHNhIG9yIE5vbmUKKwogICAgIEBwcm9wZXJ0eQogICAgIGRlZiByZXF1aXJlZChzZWxmKSAtPiBib29sOgogICAgICAgICByZXR1cm4gc2VsZi5maWVsZF9pbmZvLmlzX3JlcXVpcmVkKCkKQEAgLTI0MywxMiArMjU1LDE4IEBAIGRlZiBnZXRfc2NoZW1hX2Zyb21fbW9kZWxfZmllbGQoCiAgICAgICAgIGlmIChzZXBhcmF0ZV9pbnB1dF9vdXRwdXRfc2NoZW1hcyBvciBfaGFzX2NvbXB1dGVkX2ZpZWxkcyhmaWVsZCkpCiAgICAgICAgIGVsc2UgInZhbGlkYXRpb24iCiAgICAgKQorICAgIGZpZWxkX2FsaWFzID0gKAorICAgICAgICAoZmllbGQudmFsaWRhdGlvbl9hbGlhcyBvciBmaWVsZC5hbGlhcykKKyAgICAgICAgaWYgZmllbGQubW9kZSA9PSAidmFsaWRhdGlvbiIKKyAgICAgICAgZWxzZSAoZmllbGQuc2VyaWFsaXphdGlvbl9hbGlhcyBvciBmaWVsZC5hbGlhcykKKyAgICApCisKICAgICAjIFRoaXMgZXhwZWN0cyB0aGF0IEdlbmVyYXRlSnNvblNjaGVtYSB3YXMgYWxyZWFkeSB1c2VkIHRvIGdlbmVyYXRlIHRoZSBkZWZpbml0aW9ucwogICAgIGpzb25fc2NoZW1hID0gZmllbGRfbWFwcGluZ1soZmllbGQsIG92ZXJyaWRlX21vZGUgb3IgZmllbGQubW9kZSldCiAgICAgaWYgIiRyZWYiIG5vdCBpbiBqc29uX3NjaGVtYToKICAgICAgICAgIyBUT0RPIHJlbW92ZSB3aGVuIGRlcHJlY2F0aW5nIFB5ZGFudGljIHYxCiAgICAgICAgICMgUmVmOiBodHRwczovL2dpdGh1Yi5jb20vcHlkYW50aWMvcHlkYW50aWMvYmxvYi9kNjE3OTJjYzQyYzgwYjEzYjIzZTNmZmE3NGJjMzdlYzdjNzdmN2QxL3B5ZGFudGljL3NjaGVtYS5weSNMMjA3Ci0gICAgICAgIGpzb25fc2NoZW1hWyJ0aXRsZSJdID0gZmllbGQuZmllbGRfaW5mby50aXRsZSBvciBmaWVsZC5hbGlhcy50aXRsZSgpLnJlcGxhY2UoCisgICAgICAgIGpzb25fc2NoZW1hWyJ0aXRsZSJdID0gZmllbGQuZmllbGRfaW5mby50aXRsZSBvciBmaWVsZF9hbGlhcy50aXRsZSgpLnJlcGxhY2UoCiAgICAgICAgICAgICAiXyIsICIgIgogICAgICAgICApCiAgICAgcmV0dXJuIGpzb25fc2NoZW1hCi0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC03NTIsNyArNzUyLDcgQEAgZGVmIF92YWxpZGF0ZV92YWx1ZV93aXRoX21vZGVsX2ZpZWxkKAogZGVmIF9nZXRfbXVsdGlkaWN0X3ZhbHVlKAogICAgIGZpZWxkOiBNb2RlbEZpZWxkLCB2YWx1ZXM6IE1hcHBpbmdbc3RyLCBBbnldLCBhbGlhczogVW5pb25bc3RyLCBOb25lXSA9IE5vbmUKICkgLT4gQW55OgotICAgIGFsaWFzID0gYWxpYXMgb3IgZmllbGQuYWxpYXMKKyAgICBhbGlhcyA9IGFsaWFzIG9yIGdldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKQogICAgIGlmIGlzX3NlcXVlbmNlX2ZpZWxkKGZpZWxkKSBhbmQgaXNpbnN0YW5jZSh2YWx1ZXMsIChJbW11dGFibGVNdWx0aURpY3QsIEhlYWRlcnMpKToKICAgICAgICAgdmFsdWUgPSB2YWx1ZXMuZ2V0bGlzdChhbGlhcykKICAgICBlbHNlOgpAQCAtODA5LDE1ICs4MDksMTMgQEAgZGVmIHJlcXVlc3RfcGFyYW1zX3RvX2FyZ3MoCiAgICAgICAgICAgICAgICAgZmllbGQuZmllbGRfaW5mbywgImNvbnZlcnRfdW5kZXJzY29yZXMiLCBkZWZhdWx0X2NvbnZlcnRfdW5kZXJzY29yZXMKICAgICAgICAgICAgICkKICAgICAgICAgICAgIGlmIGNvbnZlcnRfdW5kZXJzY29yZXM6Ci0gICAgICAgICAgICAgICAgYWxpYXMgPSAoCi0gICAgICAgICAgICAgICAgICAgIGZpZWxkLmFsaWFzCi0gICAgICAgICAgICAgICAgICAgIGlmIGZpZWxkLmFsaWFzICE9IGZpZWxkLm5hbWUKLSAgICAgICAgICAgICAgICAgICAgZWxzZSBmaWVsZC5uYW1lLnJlcGxhY2UoIl8iLCAiLSIpCi0gICAgICAgICAgICAgICAgKQorICAgICAgICAgICAgICAgIGFsaWFzID0gZ2V0X3ZhbGlkYXRpb25fYWxpYXMoZmllbGQpCisgICAgICAgICAgICAgICAgaWYgYWxpYXMgPT0gZmllbGQubmFtZToKKyAgICAgICAgICAgICAgICAgICAgYWxpYXMgPSBhbGlhcy5yZXBsYWNlKCJfIiwgIi0iKQogICAgICAgICB2YWx1ZSA9IF9nZXRfbXVsdGlkaWN0X3ZhbHVlKGZpZWxkLCByZWNlaXZlZF9wYXJhbXMsIGFsaWFzPWFsaWFzKQogICAgICAgICBpZiB2YWx1ZSBpcyBub3QgTm9uZToKLSAgICAgICAgICAgIHBhcmFtc190b19wcm9jZXNzW2ZpZWxkLmFsaWFzXSA9IHZhbHVlCi0gICAgICAgIHByb2Nlc3NlZF9rZXlzLmFkZChhbGlhcyBvciBmaWVsZC5hbGlhcykKKyAgICAgICAgICAgIHBhcmFtc190b19wcm9jZXNzW2dldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKV0gPSB2YWx1ZQorICAgICAgICBwcm9jZXNzZWRfa2V5cy5hZGQoYWxpYXMgb3IgZ2V0X3ZhbGlkYXRpb25fYWxpYXMoZmllbGQpKQogCiAgICAgZm9yIGtleSBpbiByZWNlaXZlZF9wYXJhbXMua2V5cygpOgogICAgICAgICBpZiBrZXkgbm90IGluIHByb2Nlc3NlZF9rZXlzOgpAQCAtODQ3LDcgKzg0NSw3IEBAIGRlZiByZXF1ZXN0X3BhcmFtc190b19hcmdzKAogICAgICAgICBhc3NlcnQgaXNpbnN0YW5jZShmaWVsZF9pbmZvLCAocGFyYW1zLlBhcmFtLCB0ZW1wX3B5ZGFudGljX3YxX3BhcmFtcy5QYXJhbSkpLCAoCiAgICAgICAgICAgICAiUGFyYW1zIG11c3QgYmUgc3ViY2xhc3NlcyBvZiBQYXJhbSIKICAgICAgICAgKQotICAgICAgICBsb2MgPSAoZmllbGRfaW5mby5pbl8udmFsdWUsIGZpZWxkLmFsaWFzKQorICAgICAgICBsb2MgPSAoZmllbGRfaW5mby5pbl8udmFsdWUsIGdldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKSkKICAgICAgICAgdl8sIGVycm9yc18gPSBfdmFsaWRhdGVfdmFsdWVfd2l0aF9tb2RlbF9maWVsZCgKICAgICAgICAgICAgIGZpZWxkPWZpZWxkLCB2YWx1ZT12YWx1ZSwgdmFsdWVzPXZhbHVlcywgbG9jPWxvYwogICAgICAgICApCkBAIC05MzYsOCArOTM0LDggQEAgYXN5bmMgZGVmIHByb2Nlc3NfZm4oCiAgICAgICAgICAgICAgICAgICAgIHRnLnN0YXJ0X3Nvb24ocHJvY2Vzc19mbiwgc3ViX3ZhbHVlLnJlYWQpCiAgICAgICAgICAgICB2YWx1ZSA9IHNlcmlhbGl6ZV9zZXF1ZW5jZV92YWx1ZShmaWVsZD1maWVsZCwgdmFsdWU9cmVzdWx0cykKICAgICAgICAgaWYgdmFsdWUgaXMgbm90IE5vbmU6Ci0gICAgICAgICAgICB2YWx1ZXNbZmllbGQuYWxpYXNdID0gdmFsdWUKLSAgICBmaWVsZF9hbGlhc2VzID0ge2ZpZWxkLmFsaWFzIGZvciBmaWVsZCBpbiBib2R5X2ZpZWxkc30KKyAgICAgICAgICAgIHZhbHVlc1tnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCldID0gdmFsdWUKKyAgICBmaWVsZF9hbGlhc2VzID0ge2dldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKSBmb3IgZmllbGQgaW4gYm9keV9maWVsZHN9CiAgICAgZm9yIGtleSBpbiByZWNlaXZlZF9ib2R5LmtleXMoKToKICAgICAgICAgaWYga2V5IG5vdCBpbiBmaWVsZF9hbGlhc2VzOgogICAgICAgICAgICAgcGFyYW1fdmFsdWVzID0gcmVjZWl2ZWRfYm9keS5nZXRsaXN0KGtleSkKQEAgLTk3OSwxMSArOTc3LDExIEBAIGFzeW5jIGRlZiByZXF1ZXN0X2JvZHlfdG9fYXJncygKICAgICAgICAgKQogICAgICAgICByZXR1cm4ge2ZpcnN0X2ZpZWxkLm5hbWU6IHZffSwgZXJyb3JzXwogICAgIGZvciBmaWVsZCBpbiBib2R5X2ZpZWxkczoKLSAgICAgICAgbG9jID0gKCJib2R5IiwgZmllbGQuYWxpYXMpCisgICAgICAgIGxvYyA9ICgiYm9keSIsIGdldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKSkKICAgICAgICAgdmFsdWU6IE9wdGlvbmFsW0FueV0gPSBOb25lCiAgICAgICAgIGlmIGJvZHlfdG9fcHJvY2VzcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgIHRyeToKLSAgICAgICAgICAgICAgICB2YWx1ZSA9IGJvZHlfdG9fcHJvY2Vzcy5nZXQoZmllbGQuYWxpYXMpCisgICAgICAgICAgICAgICAgdmFsdWUgPSBib2R5X3RvX3Byb2Nlc3MuZ2V0KGdldF92YWxpZGF0aW9uX2FsaWFzKGZpZWxkKSkKICAgICAgICAgICAgICMgSWYgdGhlIHJlY2VpdmVkIGJvZHkgaXMgYSBsaXN0LCBub3QgYSBkaWN0CiAgICAgICAgICAgICBleGNlcHQgQXR0cmlidXRlRXJyb3I6CiAgICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChnZXRfbWlzc2luZ19maWVsZF9lcnJvcihsb2MpKQpAQCAtMTA2MiwzICsxMDYwLDggQEAgZGVmIGdldF9ib2R5X2ZpZWxkKAogICAgICAgICBmaWVsZF9pbmZvPUJvZHlGaWVsZEluZm8oKipCb2R5RmllbGRJbmZvX2t3YXJncyksCiAgICAgKQogICAgIHJldHVybiBmaW5hbF9maWVsZAorCisKK2RlZiBnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZDogTW9kZWxGaWVsZCkgLT4gc3RyOgorICAgIHZhID0gZ2V0YXR0cihmaWVsZCwgInZhbGlkYXRpb25fYWxpYXMiLCBOb25lKQorICAgIHJldHVybiB2YSBvciBmaWVsZC5hbGlhcwotLS0gYS9mYXN0YXBpL29wZW5hcGkvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CkBAIC0xOSw2ICsxOSw3IEBACiAgICAgX2dldF9mbGF0X2ZpZWxkc19mcm9tX3BhcmFtcywKICAgICBnZXRfZmxhdF9kZXBlbmRhbnQsCiAgICAgZ2V0X2ZsYXRfcGFyYW1zLAorICAgIGdldF92YWxpZGF0aW9uX2FsaWFzLAogKQogZnJvbSBmYXN0YXBpLmVuY29kZXJzIGltcG9ydCBqc29uYWJsZV9lbmNvZGVyCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5jb25zdGFudHMgaW1wb3J0IE1FVEhPRFNfV0lUSF9CT0RZLCBSRUZfUFJFRklYCkBAIC0xNDEsMTUgKzE0MiwxNSBAQCBkZWYgX2dldF9vcGVuYXBpX29wZXJhdGlvbl9wYXJhbWV0ZXJzKAogICAgICAgICAgICAgICAgIGZpZWxkX21hcHBpbmc9ZmllbGRfbWFwcGluZywKICAgICAgICAgICAgICAgICBzZXBhcmF0ZV9pbnB1dF9vdXRwdXRfc2NoZW1hcz1zZXBhcmF0ZV9pbnB1dF9vdXRwdXRfc2NoZW1hcywKICAgICAgICAgICAgICkKLSAgICAgICAgICAgIG5hbWUgPSBwYXJhbS5hbGlhcworICAgICAgICAgICAgbmFtZSA9IGdldF92YWxpZGF0aW9uX2FsaWFzKHBhcmFtKQogICAgICAgICAgICAgY29udmVydF91bmRlcnNjb3JlcyA9IGdldGF0dHIoCiAgICAgICAgICAgICAgICAgcGFyYW0uZmllbGRfaW5mbywKICAgICAgICAgICAgICAgICAiY29udmVydF91bmRlcnNjb3JlcyIsCiAgICAgICAgICAgICAgICAgZGVmYXVsdF9jb252ZXJ0X3VuZGVyc2NvcmVzLAogICAgICAgICAgICAgKQogICAgICAgICAgICAgaWYgKAogICAgICAgICAgICAgICAgIHBhcmFtX3R5cGUgPT0gUGFyYW1UeXBlcy5oZWFkZXIKLSAgICAgICAgICAgICAgICBhbmQgcGFyYW0uYWxpYXMgPT0gcGFyYW0ubmFtZQorICAgICAgICAgICAgICAgIGFuZCBuYW1lID09IHBhcmFtLm5hbWUKICAgICAgICAgICAgICAgICBhbmQgY29udmVydF91bmRlcnNjb3JlcwogICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgICBuYW1lID0gcGFyYW0ubmFtZS5yZXBsYWNlKCJfIiwgIi0iKQotLS0gYS9mYXN0YXBpL3BhcmFtcy5weQorKysgYi9mYXN0YXBpL3BhcmFtcy5weQpAQCAtMTE1LDYgKzExNSwxMCBAQCBkZWYgX19pbml0X18oCiAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICBrd2FyZ3NbImRlcHJlY2F0ZWQiXSA9IGRlcHJlY2F0ZWQKICAgICAgICAgaWYgUFlEQU5USUNfVjI6CisgICAgICAgICAgICBpZiBzZXJpYWxpemF0aW9uX2FsaWFzIGluIChfVW5zZXQsIE5vbmUpIGFuZCBpc2luc3RhbmNlKGFsaWFzLCBzdHIpOgorICAgICAgICAgICAgICAgIHNlcmlhbGl6YXRpb25fYWxpYXMgPSBhbGlhcworICAgICAgICAgICAgaWYgdmFsaWRhdGlvbl9hbGlhcyBpbiAoX1Vuc2V0LCBOb25lKToKKyAgICAgICAgICAgICAgICB2YWxpZGF0aW9uX2FsaWFzID0gYWxpYXMKICAgICAgICAgICAgIGt3YXJncy51cGRhdGUoCiAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAiYW5ub3RhdGlvbiI6IGFubm90YXRpb24sCkBAIC01NzEsNiArNTc1LDEwIEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgZWxzZToKICAgICAgICAgICAgIGt3YXJnc1siZGVwcmVjYXRlZCJdID0gZGVwcmVjYXRlZAogICAgICAgICBpZiBQWURBTlRJQ19WMjoKKyAgICAgICAgICAgIGlmIHNlcmlhbGl6YXRpb25fYWxpYXMgaW4gKF9VbnNldCwgTm9uZSkgYW5kIGlzaW5zdGFuY2UoYWxpYXMsIHN0cik6CisgICAgICAgICAgICAgICAgc2VyaWFsaXphdGlvbl9hbGlhcyA9IGFsaWFzCisgICAgICAgICAgICBpZiB2YWxpZGF0aW9uX2FsaWFzIGluIChfVW5zZXQsIE5vbmUpOgorICAgICAgICAgICAgICAgIHZhbGlkYXRpb25fYWxpYXMgPSBhbGlhcwogICAgICAgICAgICAga3dhcmdzLnVwZGF0ZSgKICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICJhbm5vdGF0aW9uIjogYW5ub3RhdGlvbiwK", "fastapi_14605": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTUxLDcgKzUxLDcgQEAKICAgICBjb250ZXh0bWFuYWdlcl9pbl90aHJlYWRwb29sLAogKQogZnJvbSBmYXN0YXBpLmRlcGVuZGVuY2llcy5tb2RlbHMgaW1wb3J0IERlcGVuZGFudAotZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0IERlcGVuZGVuY3lTY29wZUVycm9yCitmcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgRGVwZW5kZW5jeVNjb3BlRXJyb3IsIEZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcKIGZyb20gZmFzdGFwaS5sb2dnZXIgaW1wb3J0IGxvZ2dlcgogZnJvbSBmYXN0YXBpLnNlY3VyaXR5Lm9hdXRoMiBpbXBvcnQgU2VjdXJpdHlTY29wZXMKIGZyb20gZmFzdGFwaS50eXBlcyBpbXBvcnQgRGVwZW5kZW5jeUNhY2hlS2V5CkBAIC0zMjcsNyArMzI3LDcgQEAgZGVmIGdldF9kZXBlbmRhbnQoCiAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgICJweWRhbnRpYy52MSBpcyBkZXByZWNhdGVkIGFuZCB3aWxsIHNvb24gc3RvcCBiZWluZyBzdXBwb3J0ZWQgYnkgRmFzdEFQSS4iCiAgICAgICAgICAgICAgICAgZiIgUGxlYXNlIHVwZGF0ZSB0aGUgcGFyYW0ge3BhcmFtX25hbWV9OiB7cGFyYW1fZGV0YWlscy50eXBlX2Fubm90YXRpb24hcn0uIiwKLSAgICAgICAgICAgICAgICBjYXRlZ29yeT1EZXByZWNhdGlvbldhcm5pbmcsCisgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RmFzdEFQSURlcHJlY2F0aW9uV2FybmluZywKICAgICAgICAgICAgICAgICBzdGFja2xldmVsPTUsCiAgICAgICAgICAgICApCiAgICAgICAgIGlmIGlzaW5zdGFuY2UoCi0tLSBhL2Zhc3RhcGkvZXhjZXB0aW9ucy5weQorKysgYi9mYXN0YXBpL2V4Y2VwdGlvbnMucHkKQEAgLTIzMSwzICsyMzEsMTAgQEAgZGVmIF9faW5pdF9fKAogICAgICkgLT4gTm9uZToKICAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhlcnJvcnMsIGVuZHBvaW50X2N0eD1lbmRwb2ludF9jdHgpCiAgICAgICAgIHNlbGYuYm9keSA9IGJvZHkKKworCitjbGFzcyBGYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nKFVzZXJXYXJuaW5nKToKKyAgICAiIiIKKyAgICBBIGN1c3RvbSBkZXByZWNhdGlvbiB3YXJuaW5nIGFzIERlcHJlY2F0aW9uV2FybmluZyBpcyBpZ25vcmVkCisgICAgUmVmOiBodHRwczovL3NldGhtbGFyc29uLmRldi9kZXByZWNhdGlvbnMtdmlhLXdhcm5pbmdzLWRvbnQtd29yay1mb3ItcHl0aG9uLWxpYnJhcmllcworICAgICIiIgotLS0gYS9mYXN0YXBpL29wZW5hcGkvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CkBAIC0yMyw2ICsyMyw3IEBACiAgICAgZ2V0X3ZhbGlkYXRpb25fYWxpYXMsCiApCiBmcm9tIGZhc3RhcGkuZW5jb2RlcnMgaW1wb3J0IGpzb25hYmxlX2VuY29kZXIKK2Zyb20gZmFzdGFwaS5leGNlcHRpb25zIGltcG9ydCBGYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5jb25zdGFudHMgaW1wb3J0IE1FVEhPRFNfV0lUSF9CT0RZLCBSRUZfUFJFRklYCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5tb2RlbHMgaW1wb3J0IE9wZW5BUEkKIGZyb20gZmFzdGFwaS5wYXJhbXMgaW1wb3J0IEJvZHksIFBhcmFtVHlwZXMKQEAgLTIxNSw5ICsyMTYsOSBAQCBkZWYgZ2VuZXJhdGVfb3BlcmF0aW9uX2lkKAogICAgICosIHJvdXRlOiByb3V0aW5nLkFQSVJvdXRlLCBtZXRob2Q6IHN0cgogKSAtPiBzdHI6ICAjIHByYWdtYTogbm9jb3ZlcgogICAgIHdhcm5pbmdzLndhcm4oCi0gICAgICAgICJmYXN0YXBpLm9wZW5hcGkudXRpbHMuZ2VuZXJhdGVfb3BlcmF0aW9uX2lkKCkgd2FzIGRlcHJlY2F0ZWQsICIKKyAgICAgICAgbWVzc2FnZT0iZmFzdGFwaS5vcGVuYXBpLnV0aWxzLmdlbmVyYXRlX29wZXJhdGlvbl9pZCgpIHdhcyBkZXByZWNhdGVkLCAiCiAgICAgICAgICJpdCBpcyBub3QgdXNlZCBpbnRlcm5hbGx5LCBhbmQgd2lsbCBiZSByZW1vdmVkIHNvb24iLAotICAgICAgICBEZXByZWNhdGlvbldhcm5pbmcsCisgICAgICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCiAgICAgICAgIHN0YWNrbGV2ZWw9MiwKICAgICApCiAgICAgaWYgcm91dGUub3BlcmF0aW9uX2lkOgotLS0gYS9mYXN0YXBpL3BhcmFtcy5weQorKysgYi9mYXN0YXBpL3BhcmFtcy5weQpAQCAtNCw2ICs0LDcgQEAKIGZyb20gZW51bSBpbXBvcnQgRW51bQogZnJvbSB0eXBpbmcgaW1wb3J0IEFubm90YXRlZCwgQW55LCBDYWxsYWJsZSwgT3B0aW9uYWwsIFVuaW9uCiAKK2Zyb20gZmFzdGFwaS5leGNlcHRpb25zIGltcG9ydCBGYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nCiBmcm9tIGZhc3RhcGkub3BlbmFwaS5tb2RlbHMgaW1wb3J0IEV4YW1wbGUKIGZyb20gcHlkYW50aWMuZmllbGRzIGltcG9ydCBGaWVsZEluZm8KIGZyb20gdHlwaW5nX2V4dGVuc2lvbnMgaW1wb3J0IExpdGVyYWwsIGRlcHJlY2F0ZWQKQEAgLTc1LDcgKzc2LDcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICBpZiBleGFtcGxlIGlzIG5vdCBfVW5zZXQ6CiAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgICJgZXhhbXBsZWAgaGFzIGJlZW4gZGVwcmVjYXRlZCwgcGxlYXNlIHVzZSBgZXhhbXBsZXNgIGluc3RlYWQiLAotICAgICAgICAgICAgICAgIGNhdGVnb3J5PURlcHJlY2F0aW9uV2FybmluZywKKyAgICAgICAgICAgICAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAogICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NCwKICAgICAgICAgICAgICkKICAgICAgICAgc2VsZi5leGFtcGxlID0gZXhhbXBsZQpAQCAtMTA1LDcgKzEwNiw3IEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgaWYgcmVnZXggaXMgbm90IE5vbmU6CiAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgICJgcmVnZXhgIGhhcyBiZWVuIGRlcHJlY2F0ZWQsIHBsZWFzZSB1c2UgYHBhdHRlcm5gIGluc3RlYWQiLAotICAgICAgICAgICAgICAgIGNhdGVnb3J5PURlcHJlY2F0aW9uV2FybmluZywKKyAgICAgICAgICAgICAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAogICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NCwKICAgICAgICAgICAgICkKICAgICAgICAgY3VycmVudF9qc29uX3NjaGVtYV9leHRyYSA9IGpzb25fc2NoZW1hX2V4dHJhIG9yIGV4dHJhCkBAIC01MzAsNyArNTMxLDcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICBpZiBleGFtcGxlIGlzIG5vdCBfVW5zZXQ6CiAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgICJgZXhhbXBsZWAgaGFzIGJlZW4gZGVwcmVjYXRlZCwgcGxlYXNlIHVzZSBgZXhhbXBsZXNgIGluc3RlYWQiLAotICAgICAgICAgICAgICAgIGNhdGVnb3J5PURlcHJlY2F0aW9uV2FybmluZywKKyAgICAgICAgICAgICAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAogICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NCwKICAgICAgICAgICAgICkKICAgICAgICAgc2VsZi5leGFtcGxlID0gZXhhbXBsZQpAQCAtNTYwLDcgKzU2MSw3IEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgaWYgcmVnZXggaXMgbm90IE5vbmU6CiAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgICJgcmVnZXhgIGhhcyBiZWVuIGRlcHJlY2F0ZWQsIHBsZWFzZSB1c2UgYHBhdHRlcm5gIGluc3RlYWQiLAotICAgICAgICAgICAgICAgIGNhdGVnb3J5PURlcHJlY2F0aW9uV2FybmluZywKKyAgICAgICAgICAgICAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAogICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NCwKICAgICAgICAgICAgICkKICAgICAgICAgY3VycmVudF9qc29uX3NjaGVtYV9leHRyYSA9IGpzb25fc2NoZW1hX2V4dHJhIG9yIGV4dHJhCi0tLSBhL2Zhc3RhcGkvcm91dGluZy5weQorKysgYi9mYXN0YXBpL3JvdXRpbmcucHkKQEAgLTQ3LDYgKzQ3LDcgQEAKIGZyb20gZmFzdGFwaS5lbmNvZGVycyBpbXBvcnQganNvbmFibGVfZW5jb2RlcgogZnJvbSBmYXN0YXBpLmV4Y2VwdGlvbnMgaW1wb3J0ICgKICAgICBFbmRwb2ludENvbnRleHQsCisgICAgRmFzdEFQSURlcHJlY2F0aW9uV2FybmluZywKICAgICBGYXN0QVBJRXJyb3IsCiAgICAgUmVxdWVzdFZhbGlkYXRpb25FcnJvciwKICAgICBSZXNwb25zZVZhbGlkYXRpb25FcnJvciwKQEAgLTY0MCw3ICs2NDEsNyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICAgICAgICAgd2FybmluZ3Mud2FybigKICAgICAgICAgICAgICAgICAgICAgInB5ZGFudGljLnYxIGlzIGRlcHJlY2F0ZWQgYW5kIHdpbGwgc29vbiBzdG9wIGJlaW5nIHN1cHBvcnRlZCBieSBGYXN0QVBJLiIKICAgICAgICAgICAgICAgICAgICAgZiIgUGxlYXNlIHVwZGF0ZSB0aGUgcmVzcG9uc2UgbW9kZWwge3NlbGYucmVzcG9uc2VfbW9kZWwhcn0uIiwKLSAgICAgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgICAgICBjYXRlZ29yeT1GYXN0QVBJRGVwcmVjYXRpb25XYXJuaW5nLAogICAgICAgICAgICAgICAgICAgICBzdGFja2xldmVsPTQsCiAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgc2VsZi5yZXNwb25zZV9maWVsZCA9IGNyZWF0ZV9tb2RlbF9maWVsZCgKQEAgLTY4MCw3ICs2ODEsNyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICAgICAgICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgICAgICAgICAgICAgICAicHlkYW50aWMudjEgaXMgZGVwcmVjYXRlZCBhbmQgd2lsbCBzb29uIHN0b3AgYmVpbmcgc3VwcG9ydGVkIGJ5IEZhc3RBUEkuIgogICAgICAgICAgICAgICAgICAgICAgICAgZiIgSW4gcmVzcG9uc2VzPXt7fX0sIHBsZWFzZSB1cGRhdGUge21vZGVsfS4iLAotICAgICAgICAgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RmFzdEFQSURlcHJlY2F0aW9uV2FybmluZywKICAgICAgICAgICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NCwKICAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgIHJlc3BvbnNlX2ZpZWxkID0gY3JlYXRlX21vZGVsX2ZpZWxkKAotLS0gYS9mYXN0YXBpL3RlbXBfcHlkYW50aWNfdjFfcGFyYW1zLnB5CisrKyBiL2Zhc3RhcGkvdGVtcF9weWRhbnRpY192MV9wYXJhbXMucHkKQEAgLTEsNiArMSw3IEBACiBpbXBvcnQgd2FybmluZ3MKIGZyb20gdHlwaW5nIGltcG9ydCBBbm5vdGF0ZWQsIEFueSwgQ2FsbGFibGUsIE9wdGlvbmFsLCBVbmlvbgogCitmcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgRmFzdEFQSURlcHJlY2F0aW9uV2FybmluZwogZnJvbSBmYXN0YXBpLm9wZW5hcGkubW9kZWxzIGltcG9ydCBFeGFtcGxlCiBmcm9tIGZhc3RhcGkucGFyYW1zIGltcG9ydCBQYXJhbVR5cGVzCiBmcm9tIHR5cGluZ19leHRlbnNpb25zIGltcG9ydCBkZXByZWNhdGVkCkBAIC02Myw3ICs2NCw3IEBAIGRlZiBfX2luaXRfXygKICAgICAgICAgaWYgZXhhbXBsZSBpcyBub3QgX1Vuc2V0OgogICAgICAgICAgICAgd2FybmluZ3Mud2FybigKICAgICAgICAgICAgICAgICAiYGV4YW1wbGVgIGhhcyBiZWVuIGRlcHJlY2F0ZWQsIHBsZWFzZSB1c2UgYGV4YW1wbGVzYCBpbnN0ZWFkIiwKLSAgICAgICAgICAgICAgICBjYXRlZ29yeT1EZXByZWNhdGlvbldhcm5pbmcsCisgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RmFzdEFQSURlcHJlY2F0aW9uV2FybmluZywKICAgICAgICAgICAgICAgICBzdGFja2xldmVsPTQsCiAgICAgICAgICAgICApCiAgICAgICAgIHNlbGYuZXhhbXBsZSA9IGV4YW1wbGUKQEAgLTkzLDcgKzk0LDcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICBpZiByZWdleCBpcyBub3QgTm9uZToKICAgICAgICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgICAgICAgImByZWdleGAgaGFzIGJlZW4gZGVwcmVjYXRlZCwgcGxlYXNlIHVzZSBgcGF0dGVybmAgaW5zdGVhZCIsCi0gICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCiAgICAgICAgICAgICAgICAgc3RhY2tsZXZlbD00LAogICAgICAgICAgICAgKQogICAgICAgICBjdXJyZW50X2pzb25fc2NoZW1hX2V4dHJhID0ganNvbl9zY2hlbWFfZXh0cmEgb3IgZXh0cmEKQEAgLTUwMyw3ICs1MDQsNyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgIGlmIGV4YW1wbGUgaXMgbm90IF9VbnNldDoKICAgICAgICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgICAgICAgImBleGFtcGxlYCBoYXMgYmVlbiBkZXByZWNhdGVkLCBwbGVhc2UgdXNlIGBleGFtcGxlc2AgaW5zdGVhZCIsCi0gICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCiAgICAgICAgICAgICAgICAgc3RhY2tsZXZlbD00LAogICAgICAgICAgICAgKQogICAgICAgICBzZWxmLmV4YW1wbGUgPSBleGFtcGxlCkBAIC01MzMsNyArNTM0LDcgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICBpZiByZWdleCBpcyBub3QgTm9uZToKICAgICAgICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgICAgICAgImByZWdleGAgaGFzIGJlZW4gZGVwcmVjYXRlZCwgcGxlYXNlIHVzZSBgcGF0dGVybmAgaW5zdGVhZCIsCi0gICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCiAgICAgICAgICAgICAgICAgc3RhY2tsZXZlbD00LAogICAgICAgICAgICAgKQogICAgICAgICBjdXJyZW50X2pzb25fc2NoZW1hX2V4dHJhID0ganNvbl9zY2hlbWFfZXh0cmEgb3IgZXh0cmEKLS0tIGEvZmFzdGFwaS91dGlscy5weQorKysgYi9mYXN0YXBpL3V0aWxzLnB5CkBAIC0yMyw2ICsyMyw3IEBACiAgICAgbWF5X3YxLAogKQogZnJvbSBmYXN0YXBpLmRhdGFzdHJ1Y3R1cmVzIGltcG9ydCBEZWZhdWx0UGxhY2Vob2xkZXIsIERlZmF1bHRUeXBlCitmcm9tIGZhc3RhcGkuZXhjZXB0aW9ucyBpbXBvcnQgRmFzdEFQSURlcHJlY2F0aW9uV2FybmluZwogZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsCiBmcm9tIHB5ZGFudGljLmZpZWxkcyBpbXBvcnQgRmllbGRJbmZvCiBmcm9tIHR5cGluZ19leHRlbnNpb25zIGltcG9ydCBMaXRlcmFsCkBAIC0xOTUsOSArMTk2LDkgQEAgZGVmIGdlbmVyYXRlX29wZXJhdGlvbl9pZF9mb3JfcGF0aCgKICAgICAqLCBuYW1lOiBzdHIsIHBhdGg6IHN0ciwgbWV0aG9kOiBzdHIKICkgLT4gc3RyOiAgIyBwcmFnbWE6IG5vY292ZXIKICAgICB3YXJuaW5ncy53YXJuKAotICAgICAgICAiZmFzdGFwaS51dGlscy5nZW5lcmF0ZV9vcGVyYXRpb25faWRfZm9yX3BhdGgoKSB3YXMgZGVwcmVjYXRlZCwgIgorICAgICAgICBtZXNzYWdlPSJmYXN0YXBpLnV0aWxzLmdlbmVyYXRlX29wZXJhdGlvbl9pZF9mb3JfcGF0aCgpIHdhcyBkZXByZWNhdGVkLCAiCiAgICAgICAgICJpdCBpcyBub3QgdXNlZCBpbnRlcm5hbGx5LCBhbmQgd2lsbCBiZSByZW1vdmVkIHNvb24iLAotICAgICAgICBEZXByZWNhdGlvbldhcm5pbmcsCisgICAgICAgIGNhdGVnb3J5PUZhc3RBUElEZXByZWNhdGlvbldhcm5pbmcsCiAgICAgICAgIHN0YWNrbGV2ZWw9MiwKICAgICApCiAgICAgb3BlcmF0aW9uX2lkID0gZiJ7bmFtZX17cGF0aH0iCg==", "fastapi_15763": "LS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtMjQzNSw5ICsyNDM1LDE2IEBAIGRlZiByZWFkX3VzZXJzKCk6CiAgICAgICAgICAgICAgICAgIkEgcGF0aCBwcmVmaXggbXVzdCBub3QgZW5kIHdpdGggJy8nLCBhcyB0aGUgcm91dGVzIHdpbGwgc3RhcnQgd2l0aCAnLyciCiAgICAgICAgICAgICApCiAgICAgICAgIGVsc2U6Ci0gICAgICAgICAgICBmb3IgciBpbiBfaXRlcl9pbmNsdWRlZF9yb3V0ZV9jYW5kaWRhdGVzKHJvdXRlci5yb3V0ZXMpOgotICAgICAgICAgICAgICAgIHBhdGggPSBnZXRhdHRyKHIsICJwYXRoIiwgTm9uZSkKLSAgICAgICAgICAgICAgICBuYW1lID0gZ2V0YXR0cihyLCAibmFtZSIsICJ1bmtub3duIikKKyAgICAgICAgICAgIGZvciByb3V0ZSwgcm91dGVfY29udGV4dCBpbiBfaXRlcl9yb3V0ZXNfd2l0aF9jb250ZXh0KHJvdXRlci5yb3V0ZXMpOgorICAgICAgICAgICAgICAgIGlmIHJvdXRlX2NvbnRleHQgaXMgTm9uZToKKyAgICAgICAgICAgICAgICAgICAgcGF0aCA9IGdldGF0dHIocm91dGUsICJwYXRoIiwgTm9uZSkKKyAgICAgICAgICAgICAgICAgICAgbmFtZSA9IGdldGF0dHIocm91dGUsICJuYW1lIiwgInVua25vd24iKQorICAgICAgICAgICAgICAgIGVsaWYgcm91dGVfY29udGV4dC5zdGFybGV0dGVfcm91dGUgaXMgbm90IE5vbmU6CisgICAgICAgICAgICAgICAgICAgIHBhdGggPSBnZXRhdHRyKHJvdXRlX2NvbnRleHQuc3RhcmxldHRlX3JvdXRlLCAicGF0aCIsIE5vbmUpCisgICAgICAgICAgICAgICAgICAgIG5hbWUgPSBnZXRhdHRyKHJvdXRlX2NvbnRleHQuc3RhcmxldHRlX3JvdXRlLCAibmFtZSIsICJ1bmtub3duIikKKyAgICAgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgICAgICBwYXRoID0gcm91dGVfY29udGV4dC5wYXRoCisgICAgICAgICAgICAgICAgICAgIG5hbWUgPSByb3V0ZV9jb250ZXh0Lm5hbWUKICAgICAgICAgICAgICAgICBpZiBwYXRoIGlzIG5vdCBOb25lIGFuZCBub3QgcGF0aDoKICAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmFzdEFQSUVycm9yKAogICAgICAgICAgICAgICAgICAgICAgICAgZiJQcmVmaXggYW5kIHBhdGggY2Fubm90IGJlIGJvdGggZW1wdHkgKHBhdGggb3BlcmF0aW9uOiB7bmFtZX0pIgo=", "fastapi_14583": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTEsNiArMSw3IEBACiBpbXBvcnQgZGF0YWNsYXNzZXMKIGltcG9ydCBpbnNwZWN0CiBpbXBvcnQgc3lzCitpbXBvcnQgd2FybmluZ3MKIGZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBDb3JvdXRpbmUsIE1hcHBpbmcsIFNlcXVlbmNlCiBmcm9tIGNvbnRleHRsaWIgaW1wb3J0IEFzeW5jRXhpdFN0YWNrLCBjb250ZXh0bWFuYWdlcgogZnJvbSBjb3B5IGltcG9ydCBjb3B5LCBkZWVwY29weQpAQCAtMzIyLDYgKzMyMywxMyBAQCBkZWYgZ2V0X2RlcGVuZGFudCgKICAgICAgICAgICAgICkKICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIGFzc2VydCBwYXJhbV9kZXRhaWxzLmZpZWxkIGlzIG5vdCBOb25lCisgICAgICAgIGlmIGlzaW5zdGFuY2UocGFyYW1fZGV0YWlscy5maWVsZCwgbWF5X3YxLk1vZGVsRmllbGQpOgorICAgICAgICAgICAgd2FybmluZ3Mud2FybigKKyAgICAgICAgICAgICAgICAicHlkYW50aWMudjEgaXMgZGVwcmVjYXRlZCBhbmQgd2lsbCBzb29uIHN0b3AgYmVpbmcgc3VwcG9ydGVkIGJ5IEZhc3RBUEkuIgorICAgICAgICAgICAgICAgIGYiIFBsZWFzZSB1cGRhdGUgdGhlIHBhcmFtIHtwYXJhbV9uYW1lfToge3BhcmFtX2RldGFpbHMudHlwZV9hbm5vdGF0aW9uIXJ9LiIsCisgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgIHN0YWNrbGV2ZWw9NSwKKyAgICAgICAgICAgICkKICAgICAgICAgaWYgaXNpbnN0YW5jZSgKICAgICAgICAgICAgIHBhcmFtX2RldGFpbHMuZmllbGQuZmllbGRfaW5mbywgKHBhcmFtcy5Cb2R5LCB0ZW1wX3B5ZGFudGljX3YxX3BhcmFtcy5Cb2R5KQogICAgICAgICApOgotLS0gYS9mYXN0YXBpL3JvdXRpbmcucHkKKysrIGIvZmFzdGFwaS9yb3V0aW5nLnB5CkBAIC0yLDYgKzIsNyBAQAogaW1wb3J0IGZ1bmN0b29scwogaW1wb3J0IGluc3BlY3QKIGltcG9ydCBqc29uCitpbXBvcnQgd2FybmluZ3MKIGZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCAoCiAgICAgQXN5bmNJdGVyYXRvciwKICAgICBBd2FpdGFibGUsCkBAIC0yOCw2ICsyOSw3IEBACiAgICAgX2dldF9tb2RlbF9jb25maWcsCiAgICAgX21vZGVsX2R1bXAsCiAgICAgX25vcm1hbGl6ZV9lcnJvcnMsCisgICAgYW5ub3RhdGlvbl9pc19weWRhbnRpY192MSwKICAgICBsZW5pZW50X2lzc3ViY2xhc3MsCiAgICAgbWF5X3YxLAogKQpAQCAtNjM0LDYgKzYzNiwxMyBAQCBkZWYgX19pbml0X18oCiAgICAgICAgICAgICAgICAgZiJTdGF0dXMgY29kZSB7c3RhdHVzX2NvZGV9IG11c3Qgbm90IGhhdmUgYSByZXNwb25zZSBib2R5IgogICAgICAgICAgICAgKQogICAgICAgICAgICAgcmVzcG9uc2VfbmFtZSA9ICJSZXNwb25zZV8iICsgc2VsZi51bmlxdWVfaWQKKyAgICAgICAgICAgIGlmIGFubm90YXRpb25faXNfcHlkYW50aWNfdjEoc2VsZi5yZXNwb25zZV9tb2RlbCk6CisgICAgICAgICAgICAgICAgd2FybmluZ3Mud2FybigKKyAgICAgICAgICAgICAgICAgICAgInB5ZGFudGljLnYxIGlzIGRlcHJlY2F0ZWQgYW5kIHdpbGwgc29vbiBzdG9wIGJlaW5nIHN1cHBvcnRlZCBieSBGYXN0QVBJLiIKKyAgICAgICAgICAgICAgICAgICAgZiIgUGxlYXNlIHVwZGF0ZSB0aGUgcmVzcG9uc2UgbW9kZWwge3NlbGYucmVzcG9uc2VfbW9kZWwhcn0uIiwKKyAgICAgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgICAgICBzdGFja2xldmVsPTQsCisgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgc2VsZi5yZXNwb25zZV9maWVsZCA9IGNyZWF0ZV9tb2RlbF9maWVsZCgKICAgICAgICAgICAgICAgICBuYW1lPXJlc3BvbnNlX25hbWUsCiAgICAgICAgICAgICAgICAgdHlwZV89c2VsZi5yZXNwb25zZV9tb2RlbCwKQEAgLTY2Nyw2ICs2NzYsMTMgQEAgZGVmIF9faW5pdF9fKAogICAgICAgICAgICAgICAgICAgICBmIlN0YXR1cyBjb2RlIHthZGRpdGlvbmFsX3N0YXR1c19jb2RlfSBtdXN0IG5vdCBoYXZlIGEgcmVzcG9uc2UgYm9keSIKICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgcmVzcG9uc2VfbmFtZSA9IGYiUmVzcG9uc2Vfe2FkZGl0aW9uYWxfc3RhdHVzX2NvZGV9X3tzZWxmLnVuaXF1ZV9pZH0iCisgICAgICAgICAgICAgICAgaWYgYW5ub3RhdGlvbl9pc19weWRhbnRpY192MShtb2RlbCk6CisgICAgICAgICAgICAgICAgICAgIHdhcm5pbmdzLndhcm4oCisgICAgICAgICAgICAgICAgICAgICAgICAicHlkYW50aWMudjEgaXMgZGVwcmVjYXRlZCBhbmQgd2lsbCBzb29uIHN0b3AgYmVpbmcgc3VwcG9ydGVkIGJ5IEZhc3RBUEkuIgorICAgICAgICAgICAgICAgICAgICAgICAgZiIgSW4gcmVzcG9uc2VzPXt7fX0sIHBsZWFzZSB1cGRhdGUge21vZGVsfS4iLAorICAgICAgICAgICAgICAgICAgICAgICAgY2F0ZWdvcnk9RGVwcmVjYXRpb25XYXJuaW5nLAorICAgICAgICAgICAgICAgICAgICAgICAgc3RhY2tsZXZlbD00LAorICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgcmVzcG9uc2VfZmllbGQgPSBjcmVhdGVfbW9kZWxfZmllbGQoCiAgICAgICAgICAgICAgICAgICAgIG5hbWU9cmVzcG9uc2VfbmFtZSwgdHlwZV89bW9kZWwsIG1vZGU9InNlcmlhbGl6YXRpb24iCiAgICAgICAgICAgICAgICAgKQo=", "fastapi_15745": "...", "fastapi_14297": "LS0tIGEvZmFzdGFwaS9fY29tcGF0L3YyLnB5CisrKyBiL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQpAQCAtMzcxLDYgKzM3MSwxMyBAQCBkZWYgY29weV9maWVsZF9pbmZvKCosIGZpZWxkX2luZm86IEZpZWxkSW5mbywgYW5ub3RhdGlvbjogQW55KSAtPiBGaWVsZEluZm86CiAKIGRlZiBzZXJpYWxpemVfc2VxdWVuY2VfdmFsdWUoKiwgZmllbGQ6IE1vZGVsRmllbGQsIHZhbHVlOiBBbnkpIC0+IFNlcXVlbmNlW0FueV06CiAgICAgb3JpZ2luX3R5cGUgPSBnZXRfb3JpZ2luKGZpZWxkLmZpZWxkX2luZm8uYW5ub3RhdGlvbikgb3IgZmllbGQuZmllbGRfaW5mby5hbm5vdGF0aW9uCisgICAgaWYgb3JpZ2luX3R5cGUgaXMgVW5pb246ICAjIEhhbmRsZSBvcHRpb25hbCBzZXF1ZW5jZXMKKyAgICAgICAgdW5pb25fYXJncyA9IGdldF9hcmdzKGZpZWxkLmZpZWxkX2luZm8uYW5ub3RhdGlvbikKKyAgICAgICAgZm9yIHVuaW9uX2FyZyBpbiB1bmlvbl9hcmdzOgorICAgICAgICAgICAgaWYgdW5pb25fYXJnIGlzIHR5cGUoTm9uZSk6CisgICAgICAgICAgICAgICAgY29udGludWUKKyAgICAgICAgICAgIG9yaWdpbl90eXBlID0gZ2V0X29yaWdpbih1bmlvbl9hcmcpIG9yIHVuaW9uX2FyZworICAgICAgICAgICAgYnJlYWsKICAgICBhc3NlcnQgaXNzdWJjbGFzcyhvcmlnaW5fdHlwZSwgc2hhcmVkLnNlcXVlbmNlX3R5cGVzKSAgIyB0eXBlOiBpZ25vcmVbYXJnLXR5cGVdCiAgICAgcmV0dXJuIHNoYXJlZC5zZXF1ZW5jZV9hbm5vdGF0aW9uX3RvX3R5cGVbb3JpZ2luX3R5cGVdKHZhbHVlKSAgIyB0eXBlOiBpZ25vcmVbbm8tYW55LXJldHVybl0KIAo=", "fastapi_14303": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTkwMyw5ICs5MDMsMTMgQEAgYXN5bmMgZGVmIHByb2Nlc3NfZm4oCiAgICAgICAgIGlmIHZhbHVlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgdmFsdWVzW2ZpZWxkLmFsaWFzXSA9IHZhbHVlCiAgICAgZmllbGRfYWxpYXNlcyA9IHtmaWVsZC5hbGlhcyBmb3IgZmllbGQgaW4gYm9keV9maWVsZHN9Ci0gICAgZm9yIGtleSwgdmFsdWUgaW4gcmVjZWl2ZWRfYm9keS5pdGVtcygpOgorICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfYm9keS5rZXlzKCk6CiAgICAgICAgIGlmIGtleSBub3QgaW4gZmllbGRfYWxpYXNlczoKLSAgICAgICAgICAgIHZhbHVlc1trZXldID0gdmFsdWUKKyAgICAgICAgICAgIHBhcmFtX3ZhbHVlcyA9IHJlY2VpdmVkX2JvZHkuZ2V0bGlzdChrZXkpCisgICAgICAgICAgICBpZiBsZW4ocGFyYW1fdmFsdWVzKSA9PSAxOgorICAgICAgICAgICAgICAgIHZhbHVlc1trZXldID0gcGFyYW1fdmFsdWVzWzBdCisgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgIHZhbHVlc1trZXldID0gcGFyYW1fdmFsdWVzCiAgICAgcmV0dXJuIHZhbHVlcwogCiAK", "fastapi_13537": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTkwMiw4ICs5MDIsOSBAQCBhc3luYyBkZWYgcHJvY2Vzc19mbigKICAgICAgICAgICAgIHZhbHVlID0gc2VyaWFsaXplX3NlcXVlbmNlX3ZhbHVlKGZpZWxkPWZpZWxkLCB2YWx1ZT1yZXN1bHRzKQogICAgICAgICBpZiB2YWx1ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgIHZhbHVlc1tmaWVsZC5hbGlhc10gPSB2YWx1ZQorICAgIGZpZWxkX2FsaWFzZXMgPSB7ZmllbGQuYWxpYXMgZm9yIGZpZWxkIGluIGJvZHlfZmllbGRzfQogICAgIGZvciBrZXksIHZhbHVlIGluIHJlY2VpdmVkX2JvZHkuaXRlbXMoKToKLSAgICAgICAgaWYga2V5IG5vdCBpbiB2YWx1ZXM6CisgICAgICAgIGlmIGtleSBub3QgaW4gZmllbGRfYWxpYXNlczoKICAgICAgICAgICAgIHZhbHVlc1trZXldID0gdmFsdWUKICAgICByZXR1cm4gdmFsdWVzCiAK", "fastapi_14953": "LS0tIGEvZG9jc19zcmMvanNvbl9iYXNlNjRfYnl0ZXMvdHV0b3JpYWwwMDFfcHkzMTAucHkKKysrIGIvZG9jc19zcmMvanNvbl9iYXNlNjRfYnl0ZXMvdHV0b3JpYWwwMDFfcHkzMTAucHkKQEAgLTAsMCArMSw0NiBAQAorZnJvbSBmYXN0YXBpIGltcG9ydCBGYXN0QVBJCitmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwKKworCitjbGFzcyBEYXRhSW5wdXQoQmFzZU1vZGVsKToKKyAgICBkZXNjcmlwdGlvbjogc3RyCisgICAgZGF0YTogYnl0ZXMKKworICAgIG1vZGVsX2NvbmZpZyA9IHsidmFsX2pzb25fYnl0ZXMiOiAiYmFzZTY0In0KKworCitjbGFzcyBEYXRhT3V0cHV0KEJhc2VNb2RlbCk6CisgICAgZGVzY3JpcHRpb246IHN0cgorICAgIGRhdGE6IGJ5dGVzCisKKyAgICBtb2RlbF9jb25maWcgPSB7InNlcl9qc29uX2J5dGVzIjogImJhc2U2NCJ9CisKKworY2xhc3MgRGF0YUlucHV0T3V0cHV0KEJhc2VNb2RlbCk6CisgICAgZGVzY3JpcHRpb246IHN0cgorICAgIGRhdGE6IGJ5dGVzCisKKyAgICBtb2RlbF9jb25maWcgPSB7CisgICAgICAgICJ2YWxfanNvbl9ieXRlcyI6ICJiYXNlNjQiLAorICAgICAgICAic2VyX2pzb25fYnl0ZXMiOiAiYmFzZTY0IiwKKyAgICB9CisKKworYXBwID0gRmFzdEFQSSgpCisKKworQGFwcC5wb3N0KCIvZGF0YSIpCitkZWYgcG9zdF9kYXRhKGJvZHk6IERhdGFJbnB1dCk6CisgICAgY29udGVudCA9IGJvZHkuZGF0YS5kZWNvZGUoInV0Zi04IikKKyAgICByZXR1cm4geyJkZXNjcmlwdGlvbiI6IGJvZHkuZGVzY3JpcHRpb24sICJjb250ZW50IjogY29udGVudH0KKworCitAYXBwLmdldCgiL2RhdGEiKQorZGVmIGdldF9kYXRhKCkgLT4gRGF0YU91dHB1dDoKKyAgICBkYXRhID0gImhlbGxvIi5lbmNvZGUoInV0Zi04IikKKyAgICByZXR1cm4gRGF0YU91dHB1dChkZXNjcmlwdGlvbj0iQSBwbHVtYnVzIiwgZGF0YT1kYXRhKQorCisKK0BhcHAucG9zdCgiL2RhdGEtaW4tb3V0IikKK2RlZiBwb3N0X2RhdGFfaW5fb3V0KGJvZHk6IERhdGFJbnB1dE91dHB1dCkgLT4gRGF0YUlucHV0T3V0cHV0OgorICAgIHJldHVybiBib2R5Ci0tLSBhL2Zhc3RhcGkvX2NvbXBhdC92Mi5weQorKysgYi9mYXN0YXBpL19jb21wYXQvdjIucHkKQEAgLTI3LDcgKzI3LDcgQEAKICkKIGZyb20gcHlkYW50aWMuX2ludGVybmFsLl90eXBpbmdfZXh0cmEgaW1wb3J0IGV2YWxfdHlwZV9sZW5pZW50CiBmcm9tIHB5ZGFudGljLmZpZWxkcyBpbXBvcnQgRmllbGRJbmZvIGFzIEZpZWxkSW5mbwotZnJvbSBweWRhbnRpYy5qc29uX3NjaGVtYSBpbXBvcnQgR2VuZXJhdGVKc29uU2NoZW1hIGFzIEdlbmVyYXRlSnNvblNjaGVtYQorZnJvbSBweWRhbnRpYy5qc29uX3NjaGVtYSBpbXBvcnQgR2VuZXJhdGVKc29uU2NoZW1hIGFzIF9HZW5lcmF0ZUpzb25TY2hlbWEKIGZyb20gcHlkYW50aWMuanNvbl9zY2hlbWEgaW1wb3J0IEpzb25TY2hlbWFWYWx1ZSBhcyBKc29uU2NoZW1hVmFsdWUKIGZyb20gcHlkYW50aWNfY29yZSBpbXBvcnQgQ29yZVNjaGVtYSBhcyBDb3JlU2NoZW1hCiBmcm9tIHB5ZGFudGljX2NvcmUgaW1wb3J0IFB5ZGFudGljVW5kZWZpbmVkCkBAIC00MCw2ICs0MCwyMyBAQAogVW5kZWZpbmVkID0gUHlkYW50aWNVbmRlZmluZWQKIGV2YWx1YXRlX2ZvcndhcmRyZWYgPSBldmFsX3R5cGVfbGVuaWVudAogCisKK2NsYXNzIEdlbmVyYXRlSnNvblNjaGVtYShfR2VuZXJhdGVKc29uU2NoZW1hKToKKyAgICAjIFRPRE86IHJlbW92ZSB3aGVuIHRoaXMgaXMgbWVyZ2VkIChvciBlcXVpdmFsZW50KTogaHR0cHM6Ly9naXRodWIuY29tL3B5ZGFudGljL3B5ZGFudGljL3B1bGwvMTI4NDEKKyAgICAjIGFuZCBkcm9wcGluZyBzdXBwb3J0IGZvciBhbnkgdmVyc2lvbiBvZiBQeWRhbnRpYyBiZWZvcmUgdGhhdCBvbmUgKHNvLCBpbiBhIHZlcnkgbG9uZyB0aW1lKQorICAgIGRlZiBieXRlc19zY2hlbWEoc2VsZiwgc2NoZW1hOiBDb3JlU2NoZW1hKSAtPiBKc29uU2NoZW1hVmFsdWU6CisgICAgICAgIGpzb25fc2NoZW1hID0geyJ0eXBlIjogInN0cmluZyIsICJjb250ZW50TWVkaWFUeXBlIjogImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbSJ9CisgICAgICAgIGJ5dGVzX21vZGUgPSAoCisgICAgICAgICAgICBzZWxmLl9jb25maWcuc2VyX2pzb25fYnl0ZXMKKyAgICAgICAgICAgIGlmIHNlbGYubW9kZSA9PSAic2VyaWFsaXphdGlvbiIKKyAgICAgICAgICAgIGVsc2Ugc2VsZi5fY29uZmlnLnZhbF9qc29uX2J5dGVzCisgICAgICAgICkKKyAgICAgICAgaWYgYnl0ZXNfbW9kZSA9PSAiYmFzZTY0IjoKKyAgICAgICAgICAgIGpzb25fc2NoZW1hWyJjb250ZW50RW5jb2RpbmciXSA9ICJiYXNlNjQiCisgICAgICAgIHNlbGYudXBkYXRlX3dpdGhfdmFsaWRhdGlvbnMoanNvbl9zY2hlbWEsIHNjaGVtYSwgc2VsZi5WYWxpZGF0aW9uc01hcHBpbmcuYnl0ZXMpCisgICAgICAgIHJldHVybiBqc29uX3NjaGVtYQorCisKICMgVE9ETzogcmVtb3ZlIHdoZW4gZHJvcHBpbmcgc3VwcG9ydCBmb3IgUHlkYW50aWMgPCB2Mi4xMi4zCiBfQXR0cnMgPSB7CiAgICAgImRlZmF1bHQiOiAuLi4sCi0tLSBhL2Zhc3RhcGkvZGF0YXN0cnVjdHVyZXMucHkKKysrIGIvZmFzdGFwaS9kYXRhc3RydWN0dXJlcy5weQpAQCAtMTM5LDcgKzEzOSw3IEBAIGRlZiBfdmFsaWRhdGUoY2xzLCBfX2lucHV0X3ZhbHVlOiBBbnksIF86IEFueSkgLT4gIlVwbG9hZEZpbGUiOgogICAgIGRlZiBfX2dldF9weWRhbnRpY19qc29uX3NjaGVtYV9fKAogICAgICAgICBjbHMsIGNvcmVfc2NoZW1hOiBNYXBwaW5nW3N0ciwgQW55XSwgaGFuZGxlcjogR2V0SnNvblNjaGVtYUhhbmRsZXIKICAgICApIC0+IGRpY3Rbc3RyLCBBbnldOgotICAgICAgICByZXR1cm4geyJ0eXBlIjogInN0cmluZyIsICJmb3JtYXQiOiAiYmluYXJ5In0KKyAgICAgICAgcmV0dXJuIHsidHlwZSI6ICJzdHJpbmciLCAiY29udGVudE1lZGlhVHlwZSI6ICJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0ifQogCiAgICAgQGNsYXNzbWV0aG9kCiAgICAgZGVmIF9fZ2V0X3B5ZGFudGljX2NvcmVfc2NoZW1hX18oCi0tLSBhL3NjcmlwdHMvcGxheXdyaWdodC9qc29uX2Jhc2U2NF9ieXRlcy9pbWFnZTAxLnB5CisrKyBiL3NjcmlwdHMvcGxheXdyaWdodC9qc29uX2Jhc2U2NF9ieXRlcy9pbWFnZTAxLnB5CkBAIC0wLDAgKzEsMzcgQEAKK2ltcG9ydCBzdWJwcm9jZXNzCitpbXBvcnQgdGltZQorCitpbXBvcnQgaHR0cHgKK2Zyb20gcGxheXdyaWdodC5zeW5jX2FwaSBpbXBvcnQgUGxheXdyaWdodCwgc3luY19wbGF5d3JpZ2h0CisKKworIyBSdW4gcGxheXdyaWdodCBjb2RlZ2VuIHRvIGdlbmVyYXRlIHRoZSBjb2RlIGJlbG93LCBjb3B5IHBhc3RlIHRoZSBzZWN0aW9ucyBpbiBydW4oKQorZGVmIHJ1bihwbGF5d3JpZ2h0OiBQbGF5d3JpZ2h0KSAtPiBOb25lOgorICAgIGJyb3dzZXIgPSBwbGF5d3JpZ2h0LmNocm9taXVtLmxhdW5jaChoZWFkbGVzcz1GYWxzZSkKKyAgICAjIFVwZGF0ZSB0aGUgdmlld3BvcnQgbWFudWFsbHkKKyAgICBjb250ZXh0ID0gYnJvd3Nlci5uZXdfY29udGV4dCh2aWV3cG9ydD17IndpZHRoIjogOTYwLCAiaGVpZ2h0IjogMTA4MH0pCisgICAgcGFnZSA9IGNvbnRleHQubmV3X3BhZ2UoKQorICAgIHBhZ2UuZ290bygiaHR0cDovL2xvY2FsaG9zdDo4MDAwL2RvY3MiKQorICAgIHBhZ2UuZ2V0X2J5X3JvbGUoImJ1dHRvbiIsIG5hbWU9IlBPU1QgL2RhdGEgUG9zdCBEYXRhIikuY2xpY2soKQorICAgICMgTWFudWFsbHkgYWRkIHRoZSBzY3JlZW5zaG90CisgICAgcGFnZS5zY3JlZW5zaG90KHBhdGg9ImRvY3MvZW4vZG9jcy9pbWcvdHV0b3JpYWwvanNvbi1iYXNlNjQtYnl0ZXMvaW1hZ2UwMS5wbmciKQorCisgICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0KKyAgICBjb250ZXh0LmNsb3NlKCkKKyAgICBicm93c2VyLmNsb3NlKCkKKworCitwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3BlbigKKyAgICBbImZhc3RhcGkiLCAicnVuIiwgImRvY3Nfc3JjL2pzb25fYmFzZTY0X2J5dGVzL3R1dG9yaWFsMDAxX3B5MzEwLnB5Il0KKykKK3RyeToKKyAgICBmb3IgXyBpbiByYW5nZSgzKToKKyAgICAgICAgdHJ5OgorICAgICAgICAgICAgcmVzcG9uc2UgPSBodHRweC5nZXQoImh0dHA6Ly9sb2NhbGhvc3Q6ODAwMC9kb2NzIikKKyAgICAgICAgZXhjZXB0IGh0dHB4LkNvbm5lY3RFcnJvcjoKKyAgICAgICAgICAgIHRpbWUuc2xlZXAoMSkKKyAgICAgICAgICAgIGJyZWFrCisgICAgd2l0aCBzeW5jX3BsYXl3cmlnaHQoKSBhcyBwbGF5d3JpZ2h0OgorICAgICAgICBydW4ocGxheXdyaWdodCkKK2ZpbmFsbHk6CisgICAgcHJvY2Vzcy50ZXJtaW5hdGUoKQo=", "fastapi_13786": "...", "fastapi_14616": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTUxLDcgKzUxLDcgQEAKIGZyb20gZmFzdGFwaS5zZWN1cml0eS5vYXV0aDIgaW1wb3J0IFNlY3VyaXR5U2NvcGVzCiBmcm9tIGZhc3RhcGkudHlwZXMgaW1wb3J0IERlcGVuZGVuY3lDYWNoZUtleQogZnJvbSBmYXN0YXBpLnV0aWxzIGltcG9ydCBjcmVhdGVfbW9kZWxfZmllbGQsIGdldF9wYXRoX3BhcmFtX25hbWVzCi1mcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwKK2Zyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgSnNvbgogZnJvbSBweWRhbnRpYy5maWVsZHMgaW1wb3J0IEZpZWxkSW5mbwogZnJvbSBzdGFybGV0dGUuYmFja2dyb3VuZCBpbXBvcnQgQmFja2dyb3VuZFRhc2tzIGFzIFN0YXJsZXR0ZUJhY2tncm91bmRUYXNrcwogZnJvbSBzdGFybGV0dGUuY29uY3VycmVuY3kgaW1wb3J0IHJ1bl9pbl90aHJlYWRwb29sCkBAIC03MjYsMTEgKzcyNiwxOSBAQCBkZWYgX3ZhbGlkYXRlX3ZhbHVlX3dpdGhfbW9kZWxfZmllbGQoCiAgICAgICAgIHJldHVybiB2XywgW10KIAogCitkZWYgX2lzX2pzb25fZmllbGQoZmllbGQ6IE1vZGVsRmllbGQpIC0+IGJvb2w6CisgICAgcmV0dXJuIGFueSh0eXBlKGl0ZW0pIGlzIEpzb24gZm9yIGl0ZW0gaW4gZmllbGQuZmllbGRfaW5mby5tZXRhZGF0YSkKKworCiBkZWYgX2dldF9tdWx0aWRpY3RfdmFsdWUoCiAgICAgZmllbGQ6IE1vZGVsRmllbGQsIHZhbHVlczogTWFwcGluZ1tzdHIsIEFueV0sIGFsaWFzOiBVbmlvbltzdHIsIE5vbmVdID0gTm9uZQogKSAtPiBBbnk6CiAgICAgYWxpYXMgPSBhbGlhcyBvciBnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkKLSAgICBpZiBpc19zZXF1ZW5jZV9maWVsZChmaWVsZCkgYW5kIGlzaW5zdGFuY2UodmFsdWVzLCAoSW1tdXRhYmxlTXVsdGlEaWN0LCBIZWFkZXJzKSk6CisgICAgaWYgKAorICAgICAgICAobm90IF9pc19qc29uX2ZpZWxkKGZpZWxkKSkKKyAgICAgICAgYW5kIGlzX3NlcXVlbmNlX2ZpZWxkKGZpZWxkKQorICAgICAgICBhbmQgaXNpbnN0YW5jZSh2YWx1ZXMsIChJbW11dGFibGVNdWx0aURpY3QsIEhlYWRlcnMpKQorICAgICk6CiAgICAgICAgIHZhbHVlID0gdmFsdWVzLmdldGxpc3QoYWxpYXMpCiAgICAgZWxzZToKICAgICAgICAgdmFsdWUgPSB2YWx1ZXMuZ2V0KGFsaWFzLCBOb25lKQo=", "fastapi_11194": "LS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKKysrIGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKQEAgLTg3MiwyMCArODcyLDE5IEBAIGFzeW5jIGRlZiBfZXh0cmFjdF9mb3JtX2JvZHkoCiAgICAgcmVjZWl2ZWRfYm9keTogRm9ybURhdGEsCiApIC0+IERpY3Rbc3RyLCBBbnldOgogICAgIHZhbHVlcyA9IHt9Ci0gICAgZmlyc3RfZmllbGQgPSBib2R5X2ZpZWxkc1swXQotICAgIGZpcnN0X2ZpZWxkX2luZm8gPSBmaXJzdF9maWVsZC5maWVsZF9pbmZvCiAKICAgICBmb3IgZmllbGQgaW4gYm9keV9maWVsZHM6CiAgICAgICAgIHZhbHVlID0gX2dldF9tdWx0aWRpY3RfdmFsdWUoZmllbGQsIHJlY2VpdmVkX2JvZHkpCisgICAgICAgIGZpZWxkX2luZm8gPSBmaWVsZC5maWVsZF9pbmZvCiAgICAgICAgIGlmICgKLSAgICAgICAgICAgIGlzaW5zdGFuY2UoZmlyc3RfZmllbGRfaW5mbywgcGFyYW1zLkZpbGUpCisgICAgICAgICAgICBpc2luc3RhbmNlKGZpZWxkX2luZm8sIHBhcmFtcy5GaWxlKQogICAgICAgICAgICAgYW5kIGlzX2J5dGVzX2ZpZWxkKGZpZWxkKQogICAgICAgICAgICAgYW5kIGlzaW5zdGFuY2UodmFsdWUsIFVwbG9hZEZpbGUpCiAgICAgICAgICk6CiAgICAgICAgICAgICB2YWx1ZSA9IGF3YWl0IHZhbHVlLnJlYWQoKQogICAgICAgICBlbGlmICgKICAgICAgICAgICAgIGlzX2J5dGVzX3NlcXVlbmNlX2ZpZWxkKGZpZWxkKQotICAgICAgICAgICAgYW5kIGlzaW5zdGFuY2UoZmlyc3RfZmllbGRfaW5mbywgcGFyYW1zLkZpbGUpCisgICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShmaWVsZF9pbmZvLCBwYXJhbXMuRmlsZSkKICAgICAgICAgICAgIGFuZCB2YWx1ZV9pc19zZXF1ZW5jZSh2YWx1ZSkKICAgICAgICAgKToKICAgICAgICAgICAgICMgRm9yIHR5cGVzCg=="}""")
PD = Path('/kaggle/working/patches'); PD.mkdir(parents=True, exist_ok=True)
for tid, b64 in PATCHES.items():
    (PD / f'{tid}.patch').write_bytes(base64.b64decode(b64))
print('patches written:', len(PATCHES))

RUNNER = Path('/kaggle/working/replay_verify2.py')
RUNNER.write_bytes(base64.b64decode("""IiIiRmFpdGhmdWwgcmVwbGF5OiBtaXJyb3JzIHN3ZWdlbW1hJ3Mgb3duIHZlcmlmaWNhdGlvbiBydWxlLgoKc3dlZ2VtbWEgZGVyaXZlcyB0aGUgcmVxdWlyZWQgdGVzdCBub2RlcyBmcm9tIHRoZSB0ZXN0X3BhdGNoCihgX2V4dHJhY3RfdGVzdF9mdW5jdGlvbnNfZnJvbV9wYXRjaGApIGFuZCBydW5zIG9ubHkgdGhvc2UsIGZhaWxpbmcgb24KZmFpbGVkL2Vycm9yL3NraXBwZWQuIFRoaXMgc2NyaXB0IGRvZXMgZXhhY3RseSB0aGUgc2FtZSwgd2l0aCBubyBtb2RlbCBpbnZvbHZlZCwKc28gdGhlIHNhbWUgcGF0Y2hlcyBjYW4gYmUgcmVwbGF5ZWQgaW4gYW55IGVudmlyb25tZW50LgoiIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKVEFTS1MgPSBQYXRoKHN5cy5hcmd2WzFdKQpTTkFQUyA9IFBhdGgoc3lzLmFyZ3ZbMl0pClBBVENIRElSID0gUGF0aChzeXMuYXJndlszXSkKSURTID0gc3lzLmFyZ3ZbNDpdCk9VVCA9IFBhdGgob3MuZW52aXJvbi5nZXQoJ1JFUExBWV9PVVQnLCAnL291dC9yZXBsYXlfcmVzdWx0Lmpzb24nKSkKCnRhc2tzID0ge2pzb24ubG9hZHMobClbJ2luc3RhbmNlX2lkJ106IGpzb24ubG9hZHMobCkKICAgICAgICAgZm9yIGwgaW4gVEFTS1MucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCl9ClRFU1RGSUxFID0gcmUuY29tcGlsZShyJ15cK1wrXCsgYi8oLio/KSQnLCByZS5NKQoKCmRlZiBleHRyYWN0X3Rlc3RzKHRlc3RfcGF0Y2g6IHN0cik6CiAgICAiIiJCeXRlLWlkZW50aWNhbCBsb2dpYyB0byBzd2VnZW1tYS5oYXJuZXNzLnZlcmlmaWNhdGlvbi5fZXh0cmFjdF90ZXN0X2Z1bmN0aW9uc19mcm9tX3BhdGNoLiIiIgogICAgaWYgbm90IHRlc3RfcGF0Y2g6CiAgICAgICAgcmV0dXJuIFtdCiAgICBhZGRlZCA9IHJlLmZpbmRhbGwocideXCtccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkKICAgIGlmIGFkZGVkOgogICAgICAgIHJldHVybiBsaXN0KGRpY3QuZnJvbWtleXMoYWRkZWQpKQogICAgcmVtb3ZlZCA9IHNldChyZS5maW5kYWxsKHInXi1ccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkpCiAgICBodW5rX2hlYWRlcnMgPSByZS5maW5kYWxsKHInXkBAW15AXStAQFxzKig/OmFzeW5jXHMrKT9kZWZccysodGVzdF9bQS1aYS16MC05X10rKVxiJywgdGVzdF9wYXRjaCwgcmUuTVVMVElMSU5FKQogICAgY29udGV4dF9kZWZzID0gcmUuZmluZGFsbChyJ14gXHMqKD86YXN5bmNccyspP2RlZlxzKyh0ZXN0X1tBLVphLXowLTlfXSspXHMqXCgnLCB0ZXN0X3BhdGNoLCByZS5NVUxUSUxJTkUpCiAgICBjb250ZXh0X3NldCA9IHNldChjb250ZXh0X2RlZnMpCiAgICBjb21iaW5lZCA9IGxpc3QoZGljdC5mcm9ta2V5cyhodW5rX2hlYWRlcnMgKyBjb250ZXh0X2RlZnMpKQogICAgcmV0dXJuIFtmbiBmb3IgZm4gaW4gY29tYmluZWQgaWYgZm4gbm90IGluIHJlbW92ZWQgYW5kIG5vdCAoCiAgICAgICAgZm4gbm90IGluIGNvbnRleHRfc2V0IGFuZCAoYW55KHIuc3RhcnRzd2l0aChmbikgZm9yIHIgaW4gcmVtb3ZlZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBhbnkoYyAhPSBmbiBhbmQgYy5zdGFydHN3aXRoKGZuKSBmb3IgYyBpbiBjb250ZXh0X3NldCkpKV0KCgpkZWYgc2goY21kLCBjd2QpOgogICAgcmV0dXJuIHN1YnByb2Nlc3MucnVuKGNtZCwgY3dkPXN0cihjd2QpLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJywgdGltZW91dD0xODAwKQoKCnJlc3VsdHMgPSBbXQpmb3IgdGlkIGluIElEUzoKICAgIHJlYyA9IGRpY3QodGFzaz10aWQpCiAgICBzbmFwLCBwYXRjaCA9IFNOQVBTIC8gZid7dGlkfS50Z3onLCBQQVRDSERJUiAvIGYne3RpZH0ucGF0Y2gnCiAgICB0ID0gdGFza3MuZ2V0KHRpZCwge30pCiAgICB0cF90ZXh0ID0gdC5nZXQoJ3Rlc3RfcGF0Y2gnKSBvciAnJwogICAgaWYgbm90IHNuYXAuZXhpc3RzKCk6CiAgICAgICAgcmVjWydlcnJvciddID0gJ25vIHNuYXBzaG90JwogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICBjb250aW51ZQogICAgd29yayA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9ZidyX3t0aWR9XycpKQogICAgdHJ5OgogICAgICAgIGlmIHNoKFsndGFyJywgJ3h6ZicsIHN0cihzbmFwKSwgJy1DJywgc3RyKHdvcmspXSwgd29yaykucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnZXh0cmFjdCBmYWlsZWQnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBwYXRjaC5leGlzdHMoKToKICAgICAgICAgICAgcCA9IHNoKFsnZ2l0JywgJ2FwcGx5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIocGF0Y2gucmVzb2x2ZSgpKV0sIHdvcmspCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgICAgICBpZiBwLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cihwYXRjaC5yZXNvbHZlKCkpXSwgd29yaykKICAgICAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfdGFpbCddID0gKChwLnN0ZGVyciBvciAnJykgKyAocC5zdGRvdXQgb3IgJycpKVstMjAwOl0KICAgICAgICB0cCA9IHdvcmsucGFyZW50IC8gZid7dGlkfV90ZXN0LnBhdGNoJwogICAgICAgIHRwLndyaXRlX2J5dGVzKCh0cF90ZXh0IGlmIHRwX3RleHQuZW5kc3dpdGgoJ1xuJykgZWxzZSB0cF90ZXh0ICsgJ1xuJykKICAgICAgICAgICAgICAgICAgICAgICAucmVwbGFjZSgnXHJcbicsICdcbicpLmVuY29kZSgndXRmLTgnKSkKICAgICAgICBwID0gc2goWydnaXQnLCAnYXBwbHknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgIHJlY1sndGVzdF9wYXRjaF9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgaWYgcC5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgICAgICByZWNbJ3Rlc3RfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1sndGVzdF9wYXRjaF90YWlsJ10gPSAoKHAuc3RkZXJyIG9yICcnKSArIChwLnN0ZG91dCBvciAnJykpWy0yMDA6XQoKICAgICAgICBmaWxlcyA9IHNvcnRlZCh7Zi5zdHJpcCgpIGZvciBmIGluIFRFU1RGSUxFLmZpbmRhbGwodHBfdGV4dCl9KQogICAgICAgIGZucyA9IGV4dHJhY3RfdGVzdHModHBfdGV4dCkKICAgICAgICByZWNbJ3Rlc3RfZmlsZXMnXSA9IGZpbGVzCiAgICAgICAgcmVjWyd0ZXN0X2ZucyddID0gZm5zCiAgICAgICAgaWYgb3MuZW52aXJvbi5nZXQoJ1JFUExBWV9NT0RFJykgPT0gJ2ZpbGVzJzoKICAgICAgICAgICAgIyBNYXRjaCBzd2VnZW1tYTogcnVuIHRoZSBXSE9MRSB0ZXN0IGZpbGVzLCBub3QgZXh0cmFjdGVkIG5vZGVzLgogICAgICAgICAgICBub2RlcyA9IGZpbGVzCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbm9kZXMgPSBbZid7Zn06Ontmbn0nIGZvciBmIGluIGZpbGVzIGZvciBmbiBpbiBmbnNdIG9yIGZpbGVzCiAgICAgICAgaWYgbm90IG5vZGVzOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnbm8gbm9kZXMnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwID0gc2goW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncHl0ZXN0JywgJy1xJywgJy0tbm8taGVhZGVyJywgJy1wJywgJ25vOmNhY2hlcHJvdmlkZXInXSArIG5vZGVzLCB3b3JrKQogICAgICAgIHRhaWwgPSAocC5zdGRvdXQgb3IgJycpICsgKHAuc3RkZXJyIG9yICcnKQogICAgICAgIG0gPSByZS5maW5kYWxsKHInKFxkKylccysocGFzc2VkfGZhaWxlZHxlcnJvcnxlcnJvcnN8c2tpcHBlZCknLCB0YWlsKQogICAgICAgIHJlY1snY291bnRzJ10gPSB7azogaW50KHYpIGZvciB2LCBrIGluIG19CiAgICAgICAgcmVjWydub2RlcyddID0gbm9kZXMKICAgICAgICByZWNbJ3B5dGVzdF90YWlsJ10gPSB0YWlsWy0zMDA6XQogICAgICAgIHBhc3NlZCA9IHJlY1snY291bnRzJ10uZ2V0KCdwYXNzZWQnLCAwKQogICAgICAgIGJhZCA9IHN1bShyZWNbJ2NvdW50cyddLmdldChrLCAwKSBmb3IgayBpbiAoJ2ZhaWxlZCcsICdlcnJvcicsICdlcnJvcnMnLCAnc2tpcHBlZCcpKQogICAgICAgIHJlY1sncmVzb2x2ZWRfaGVyZSddID0gYm9vbChwYXNzZWQgPiAwIGFuZCBiYWQgPT0gMCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICByZWNbJ2Vycm9yJ10gPSBmJ3t0eXBlKGUpLl9fbmFtZV9ffToge2V9JwogICAgZmluYWxseToKICAgICAgICBzaHV0aWwucm10cmVlKHdvcmssIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKCk9VVC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQpPVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdHMsIGluZGVudD0xKSwgZW5jb2Rpbmc9J3V0Zi04JykKcHJpbnQoJyUtMjJzICUtN3MgJS03cyAlLTlzICVzJyAlICgndGFzaycsICdhX3JjJywgJ3RfcmMnLCAncmVzb2x2ZWQnLCAnY291bnRzIC8gZXJyJykpCmZvciByIGluIHJlc3VsdHM6CiAgICBwcmludCgnJS0yMnMgJS03cyAlLTdzICUtOXMgJXMnICUgKHIuZ2V0KCd0YXNrJyksIHIuZ2V0KCdhZ2VudF9wYXRjaF9yYycpLCByLmdldCgndGVzdF9wYXRjaF9yYycpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByLmdldCgncmVzb2x2ZWRfaGVyZScpLCByLmdldCgnY291bnRzJykgb3IgKHIuZ2V0KCdlcnJvcicpIG9yICcnKVs6NDVdKSkKb2sgPSBzdW0oMSBmb3IgciBpbiByZXN1bHRzIGlmIHIuZ2V0KCdyZXNvbHZlZF9oZXJlJykpCnByaW50KCdcbnJlc29sdmVkIGhlcmU6ICVkIC8gJWQnICUgKG9rLCBsZW4ocmVzdWx0cykpKQpwcmludCgnd3JvdGUnLCBPVVQpCg=="""))

env = dict(os.environ, REPLAY_OUT='/kaggle/working/replay_result.json')
p = subprocess.run([sys.executable, str(RUNNER), str(TASKS), str(SNAPS), str(PD)] + sorted(PATCHES),
                   capture_output=True, text=True, env=env)
print(p.stdout)
print('STDERR', p.stderr[-1500:])
res = json.loads(Path('/kaggle/working/replay_result.json').read_text())
print('=== KAGGLE-SIDE REPLAY ===')
for r in res:
    print('%-22s resolved=%-6s counts=%s err=%s' % (r['task'], r.get('resolved_here'),
          r.get('counts'), (r.get('error') or '')[:40]))
print('resolved on Kaggle: %d / %d' % (sum(1 for r in res if r.get('resolved_here')), len(res)))
